# Advanced Classification Experiments Using Dataset25

This notebook evaluates advanced LTE and 5G NR signal-quality classification experiments using Dataset25 and its 35 predictors.

The experiments examine:

- untreated three-class classification;
- class-imbalance treatments;
- binary Poor-versus-Non-Poor classification;
- exploratory decision-threshold adjustment;
- Spatial Block Cross-Validation;
- measurement-density sensitivity;
- feature-group ablation; and
- hierarchical classification.

Unless otherwise stated, experiments use five-fold cross-validation, a fixed random state of 42, training-fold-only preprocessing, and Macro-F1 as the primary overall metric.

### ※ Related dissertation sections

- Section 3.1 (Research Approach—Modelling and Evaluation)
- Sections 4.3–4.10 (Advanced Classification Experiments)
- Appendix C, Table C.1 (Reproducibility Summary)

In [ ]:
import subprocess
import sys
from pathlib import Path

REPOSITORY_NAME = "england-lte-5g-signal-prediction"
REPOSITORY_URL = (
    "https://github.com/bong33kr63-ux/"
    "england-lte-5g-signal-prediction.git"
)

if "google.colab" in sys.modules:
    REPOSITORY_ROOT = Path("/content") / REPOSITORY_NAME
    if not (REPOSITORY_ROOT / "config.py").exists():
        subprocess.run(
            ["git", "clone", "--depth", "1", REPOSITORY_URL, str(REPOSITORY_ROOT)],
            check=True
        )
else:
    candidates = [Path.cwd().resolve(), Path.cwd().resolve().parent]
    REPOSITORY_ROOT = next(
        (path for path in candidates if (path / "config.py").exists()),
        None
    )
    if REPOSITORY_ROOT is None:
        raise FileNotFoundError("Repository root containing config.py was not found.")

if str(REPOSITORY_ROOT) not in sys.path:
    sys.path.insert(0, str(REPOSITORY_ROOT))

from config import PROCESSED_DATA_DIR

## Phase 0: Dataset Loading and Validation

Dataset25 is validated for dimensions, unique grid identifiers, target classes, predictor availability, missing or infinite values, and potential target leakage.

The final modelling subsets contain 4,970 LTE-labelled grids, 4,965 5G NR-labelled grids, and 35 predictors.

Ofcom measurement-count distributions are also examined to support the subsequent label-reliability sensitivity analysis.

### ※ Related dissertation sections

- Section 3.1 (Research Approach—Data Preparation and Reproducibility)
- Section 4.8 (Ofcom Measurement Density and Label-Reliability Sensitivity)
- Table 4.10 (Grid Retention and Class Distribution by Minimum Measurement Count)
- Figure 4.15 (Distribution of Ofcom Measurement Counts)

In [ ]:
# ============================================================
# Phase 0 - Cell 1. Imports and Global Settings
# ============================================================

import os
import pandas as pd
import numpy as np

RANDOM_STATE = 42
N_SPLITS = 5

print("Pandas version:", pd.__version__)
print("NumPy version :", np.__version__)
print("RANDOM_STATE  :", RANDOM_STATE)
print("N_SPLITS      :", N_SPLITS)

In [ ]:
# ============================================================
# Phase 0 - Cell 2. Load Final Dataset25
# ============================================================

DATA_PATH = str(
    PROCESSED_DATA_DIR / "dataset25_final.csv"
)

assert os.path.exists(DATA_PATH), (
    f"File not found: {DATA_PATH}"
)

df = pd.read_csv(
    DATA_PATH,
    low_memory=False
)

print("Dataset25 path:")
print(DATA_PATH)

print("\nOriginal dataset shape:", df.shape)

print("\nUnique grid IDs:", df["grid_id"].nunique())

print(
    "Duplicate grid IDs:",
    df["grid_id"].duplicated().sum()
)

print("\nFirst five rows:")
display(df.head())

In [ ]:
# ============================================================
# Phase 0 - Cell 3. Validate LTE / NR Labels
# ============================================================

VALID_CLASSES = [
    "Excellent",
    "Good",
    "Poor"
]

EXPECTED_LABEL_COUNTS = {
    "lte_signal_class": {
        "Excellent": 4077,
        "Good": 772,
        "Poor": 121
    },
    "nr_signal_class": {
        "Excellent": 2441,
        "Good": 1718,
        "Poor": 806
    }
}


def clean_target(data, target_col):

    if target_col not in data.columns:
        raise KeyError(
            f"Target column not found: {target_col}"
        )

    target_clean = (
        data[target_col]
        .astype("string")
        .str.strip()
    )

    valid_mask = target_clean.isin(
        VALID_CLASSES
    )

    cleaned = data.loc[
        valid_mask
    ].copy()

    cleaned[target_col] = target_clean.loc[
        valid_mask
    ]

    # Check for unexpected non-missing labels
    unexpected_labels = sorted(
        target_clean[
            target_clean.notna()
            & ~target_clean.isin(VALID_CLASSES)
        ]
        .unique()
        .tolist()
    )

    if unexpected_labels:
        raise ValueError(
            f"{target_col}: unexpected labels found: "
            f"{unexpected_labels}"
        )

    return cleaned


lte_df = clean_target(
    df,
    "lte_signal_class"
)

nr_df = clean_target(
    df,
    "nr_signal_class"
)


# ------------------------------------------------------------
# Validate class distributions
# ------------------------------------------------------------

lte_counts = (
    lte_df["lte_signal_class"]
    .value_counts()
    .reindex(VALID_CLASSES)
    .astype(int)
)

nr_counts = (
    nr_df["nr_signal_class"]
    .value_counts()
    .reindex(VALID_CLASSES)
    .astype(int)
)

assert lte_counts.to_dict() == (
    EXPECTED_LABEL_COUNTS["lte_signal_class"]
), f"Unexpected LTE class distribution: {lte_counts.to_dict()}"

assert nr_counts.to_dict() == (
    EXPECTED_LABEL_COUNTS["nr_signal_class"]
), f"Unexpected NR class distribution: {nr_counts.to_dict()}"

assert lte_df["grid_id"].is_unique
assert nr_df["grid_id"].is_unique


# ------------------------------------------------------------
# Display results
# ------------------------------------------------------------

print("=" * 70)
print("LTE")
print("=" * 70)

print("Labelled grids:", f"{len(lte_df):,}")
print("Unique grid IDs:", f"{lte_df['grid_id'].nunique():,}")
print("\nClass distribution:")
print(lte_counts)


print("\n" + "=" * 70)
print("5G NR")
print("=" * 70)

print("Labelled grids:", f"{len(nr_df):,}")
print("Unique grid IDs:", f"{nr_df['grid_id'].nunique():,}")
print("\nClass distribution:")
print(nr_counts)

print("\nLabel validation completed successfully.")

In [ ]:
# ============================================================
# Phase 0 - Cell 4. Define and Validate Predictor Columns
# ============================================================

EXCLUDE_COLS = [
    "geometry",
    "grid_id",

    # Target labels
    "lte_signal_class",
    "nr_signal_class",

    # LTE target-generating variables
    "lte_min_rsrp",
    "lte_mean_rsrp",
    "lte_median_rsrp",
    "lte_point_count",

    # NR target-generating variables
    "nr_min_rsrp",
    "nr_mean_rsrp",
    "nr_median_rsrp",
    "nr_point_count"
]


# ------------------------------------------------------------
# 1. Identify candidate predictors
# ------------------------------------------------------------

candidate_cols = [
    col for col in df.columns
    if col not in EXCLUDE_COLS
]


# ------------------------------------------------------------
# 2. Check predictor data types
# ------------------------------------------------------------

non_numeric_predictors = [
    col for col in candidate_cols
    if not pd.api.types.is_numeric_dtype(
        df[col]
    )
]

if non_numeric_predictors:
    raise TypeError(
        "Non-numeric predictor columns detected: "
        f"{non_numeric_predictors}"
    )

feature_cols = candidate_cols


# ------------------------------------------------------------
# 3. Leakage checks
# ------------------------------------------------------------

forbidden_patterns = [
    "signal_class",
    "rsrp",
    "point_count"
]

suspicious_features = [
    col for col in feature_cols
    if any(
        pattern in col.lower()
        for pattern in forbidden_patterns
    )
]

if suspicious_features:
    raise ValueError(
        "Potential target leakage detected: "
        f"{suspicious_features}"
    )


# ------------------------------------------------------------
# 4. Structural checks
# ------------------------------------------------------------

if len(feature_cols) != 35:
    raise ValueError(
        "Dataset25 should contain 35 predictors, "
        f"but {len(feature_cols)} were detected."
    )

if len(feature_cols) != len(set(feature_cols)):
    raise ValueError(
        "Duplicate predictor names detected."
    )


# ------------------------------------------------------------
# 5. Display predictors
# ------------------------------------------------------------

print(
    "Number of predictor features:",
    len(feature_cols)
)

print("\nPredictor columns:")

for index, column in enumerate(
    feature_cols,
    start=1
):
    print(f"{index:02d}. {column}")

print("\nPredictor-column validation completed successfully.")

In [ ]:
# ============================================================
# Phase 0 - Cell 5. Missing and Infinite Predictor Check
# ============================================================

X_check = df[
    feature_cols
].copy()

missing_counts = (
    X_check
    .isna()
    .sum()
)

infinite_counts = pd.Series(
    np.isinf(
        X_check.to_numpy(
            dtype=float
        )
    ).sum(axis=0),
    index=feature_cols
)

quality_check = pd.DataFrame({
    "Missing": missing_counts,
    "Infinite": infinite_counts
})

problem_features = quality_check[
    (quality_check["Missing"] > 0)
    | (quality_check["Infinite"] > 0)
]

print("Predictor-quality check:")

if problem_features.empty:

    print("✓ No missing or infinite predictor values detected.")

else:

    display(problem_features)

    raise ValueError(
        "Missing or infinite predictor values detected."
    )

print("\nPredictor matrix shape:", X_check.shape)
print(
    "Total missing values:",
    int(missing_counts.sum())
)
print(
    "Total infinite values:",
    int(infinite_counts.sum())
)

In [ ]:
# ============================================================
# Phase 0 - Cell 6. Final Sanity Check
# ============================================================

EXPECTED_TOTAL_ROWS = 4985
EXPECTED_TOTAL_COLUMNS = 46
EXPECTED_LTE_ROWS = 4970
EXPECTED_NR_ROWS = 4965
EXPECTED_PREDICTORS = 35


# ------------------------------------------------------------
# 1. Dataset structure
# ------------------------------------------------------------

assert df.shape == (
    EXPECTED_TOTAL_ROWS,
    EXPECTED_TOTAL_COLUMNS
), f"Unexpected Dataset25 shape: {df.shape}"

assert df["grid_id"].is_unique, (
    "Duplicate grid IDs detected in Dataset25."
)


# ------------------------------------------------------------
# 2. Labelled-grid counts
# ------------------------------------------------------------

assert len(lte_df) == EXPECTED_LTE_ROWS, (
    f"Unexpected LTE labelled rows: {len(lte_df)}"
)

assert len(nr_df) == EXPECTED_NR_ROWS, (
    f"Unexpected 5G NR labelled rows: {len(nr_df)}"
)

assert lte_df["grid_id"].is_unique, (
    "Duplicate LTE grid IDs detected."
)

assert nr_df["grid_id"].is_unique, (
    "Duplicate 5G NR grid IDs detected."
)


# ------------------------------------------------------------
# 3. Target classes
# ------------------------------------------------------------

assert set(
    lte_df["lte_signal_class"].unique()
) == set(VALID_CLASSES), (
    "Unexpected LTE target classes detected."
)

assert set(
    nr_df["nr_signal_class"].unique()
) == set(VALID_CLASSES), (
    "Unexpected 5G NR target classes detected."
)


# ------------------------------------------------------------
# 4. Predictor integrity
# ------------------------------------------------------------

assert len(feature_cols) == EXPECTED_PREDICTORS, (
    f"Unexpected number of predictors: {len(feature_cols)}"
)

assert not problem_features.shape[0], (
    "Missing or infinite predictor values detected."
)

assert not any(
    column in EXCLUDE_COLS
    for column in feature_cols
), "Excluded or leakage columns detected in predictors."


# ------------------------------------------------------------
# 5. Final validation summary
# ------------------------------------------------------------

print("=" * 70)
print("PHASE 0 VALIDATION PASSED")
print("=" * 70)

print(
    "Dataset25 shape   :",
    df.shape
)

print(
    "Unique grid IDs   :",
    f"{df['grid_id'].nunique():,}"
)

print(
    "LTE labelled grids:",
    f"{len(lte_df):,}"
)

print(
    "5G NR labelled grids:",
    f"{len(nr_df):,}"
)

print(
    "Predictors        :",
    len(feature_cols)
)

print(
    "Missing predictors:",
    int(missing_counts.sum())
)

print(
    "Infinite values   :",
    int(infinite_counts.sum())
)

print(
    "Target leakage    : None"
)

print(
    "Random state      :",
    RANDOM_STATE
)

print(
    "CV folds          :",
    N_SPLITS
)

print(
    "\n✓ Ready for Phase 1: "
    "Three-Class Baseline Classification"
)

In [ ]:
# ============================================================
# Phase 0 - Cell 7.
# Prepare and Validate Ofcom Measurement Counts
# ============================================================

MEASUREMENT_CONFIGS = {
    "LTE": {
        "data": lte_df,
        "target_col": "lte_signal_class",
        "count_col": "lte_point_count"
    },
    "5G NR": {
        "data": nr_df,
        "target_col": "nr_signal_class",
        "count_col": "nr_point_count"
    }
}

measurement_frames = []


for network, config in MEASUREMENT_CONFIGS.items():

    data = config["data"].copy()
    target_col = config["target_col"]
    count_col = config["count_col"]

    required_cols = [
        "grid_id",
        target_col,
        count_col
    ]

    missing_cols = [
        col for col in required_cols
        if col not in data.columns
    ]

    if missing_cols:
        raise KeyError(
            f"{network}: missing columns: {missing_cols}"
        )

    measurement_count = pd.to_numeric(
        data[count_col],
        errors="coerce"
    )

    if measurement_count.isna().any():
        raise ValueError(
            f"{network}: missing or non-numeric "
            f"measurement counts detected."
        )

    if not np.isfinite(measurement_count).all():
        raise ValueError(
            f"{network}: infinite measurement counts detected."
        )

    if (measurement_count <= 0).any():
        raise ValueError(
            f"{network}: measurement counts must be positive."
        )

    if not np.allclose(
        measurement_count,
        np.round(measurement_count)
    ):
        raise ValueError(
            f"{network}: non-integer measurement counts detected."
        )

    frame = pd.DataFrame({
        "Network": network,
        "Grid ID": data["grid_id"].to_numpy(),
        "Signal Class": data[target_col].to_numpy(),
        "Measurement Count":
            measurement_count.astype(int).to_numpy()
    })

    measurement_frames.append(frame)


measurement_data = pd.concat(
    measurement_frames,
    ignore_index=True
)

measurement_data["Signal Class"] = pd.Categorical(
    measurement_data["Signal Class"],
    categories=VALID_CLASSES,
    ordered=True
)


# ------------------------------------------------------------
# Final validation
# ------------------------------------------------------------

assert len(
    measurement_data[
        measurement_data["Network"] == "LTE"
    ]
) == len(lte_df)

assert len(
    measurement_data[
        measurement_data["Network"] == "5G NR"
    ]
) == len(nr_df)

assert not measurement_data[
    "Measurement Count"
].isna().any()

assert (
    measurement_data["Measurement Count"] > 0
).all()


print("=" * 70)
print("OFCOM MEASUREMENT-COUNT VALIDATION PASSED")
print("=" * 70)

print(
    "LTE labelled grids:",
    f"{len(lte_df):,}"
)

print(
    "5G NR labelled grids:",
    f"{len(nr_df):,}"
)

print(
    "Combined records:",
    f"{len(measurement_data):,}"
)

print(
    "\n✓ Measurement-count data are ready for analysis."
)

In [ ]:
# ============================================================
# Phase 0 - Cell 8.
# Ofcom Measurement-Count Distribution and Grid Retention
# ============================================================

MEASUREMENT_THRESHOLDS = [
    1,
    20,
    50,
    100
]


# ------------------------------------------------------------
# 1. Overall distribution by network
# ------------------------------------------------------------

overall_rows = []

for network in [
    "LTE",
    "5G NR"
]:

    counts = measurement_data.loc[
        measurement_data["Network"] == network,
        "Measurement Count"
    ]

    overall_rows.append({
        "Network": network,
        "Labelled Grids": len(counts),
        "Total Measurements": int(counts.sum()),
        "Mean": counts.mean(),
        "SD": counts.std(ddof=1),
        "Minimum": counts.min(),
        "P05": counts.quantile(0.05),
        "P10": counts.quantile(0.10),
        "P25": counts.quantile(0.25),
        "Median": counts.median(),
        "P75": counts.quantile(0.75),
        "P90": counts.quantile(0.90),
        "P95": counts.quantile(0.95),
        "Maximum": counts.max()
    })


measurement_count_summary = pd.DataFrame(
    overall_rows
)

print("=" * 80)
print("OVERALL MEASUREMENT-COUNT DISTRIBUTION")
print("=" * 80)

display(
    measurement_count_summary.round(2)
)


# ------------------------------------------------------------
# 2. Distribution by network and signal class
# ------------------------------------------------------------

class_rows = []

for network in [
    "LTE",
    "5G NR"
]:

    for signal_class in VALID_CLASSES:

        counts = measurement_data.loc[
            (
                measurement_data["Network"]
                == network
            )
            & (
                measurement_data["Signal Class"]
                == signal_class
            ),
            "Measurement Count"
        ]

        if counts.empty:
            raise ValueError(
                f"{network}: no observations for "
                f"{signal_class}."
            )

        class_rows.append({
            "Network": network,
            "Signal Class": signal_class,
            "Grids": len(counts),
            "Total Measurements": int(counts.sum()),
            "Mean": counts.mean(),
            "SD": counts.std(ddof=1),
            "Minimum": counts.min(),
            "P25": counts.quantile(0.25),
            "Median": counts.median(),
            "P75": counts.quantile(0.75),
            "Maximum": counts.max()
        })


measurement_count_by_class = pd.DataFrame(
    class_rows
)

measurement_count_by_class[
    "Signal Class"
] = pd.Categorical(
    measurement_count_by_class[
        "Signal Class"
    ],
    categories=VALID_CLASSES,
    ordered=True
)

measurement_count_by_class = (
    measurement_count_by_class
    .sort_values(
        [
            "Network",
            "Signal Class"
        ]
    )
    .reset_index(drop=True)
)

print("\n" + "=" * 80)
print("MEASUREMENT COUNTS BY SIGNAL CLASS")
print("=" * 80)

display(
    measurement_count_by_class.round(2)
)


# ------------------------------------------------------------
# 3. Retained samples at alternative thresholds
# ------------------------------------------------------------

threshold_rows = []

for network in [
    "LTE",
    "5G NR"
]:

    network_data = measurement_data[
        measurement_data["Network"] == network
    ].copy()

    original_n = len(network_data)

    for threshold in MEASUREMENT_THRESHOLDS:

        retained = network_data[
            network_data["Measurement Count"]
            >= threshold
        ]

        class_counts = (
            retained["Signal Class"]
            .value_counts()
            .reindex(
                VALID_CLASSES,
                fill_value=0
            )
        )

        retained_n = len(retained)
        excluded_n = original_n - retained_n

        threshold_rows.append({
            "Network": network,
            "Minimum Measurements": threshold,
            "Original Grids": original_n,
            "Retained Grids": retained_n,
            "Excluded Grids": excluded_n,
            "Retained (%)":
                retained_n / original_n * 100,
            "Excluded (%)":
                excluded_n / original_n * 100,
            "Excellent":
                int(class_counts["Excellent"]),
            "Good":
                int(class_counts["Good"]),
            "Poor":
                int(class_counts["Poor"]),
            "Poor Prevalence (%)": (
                class_counts["Poor"]
                / retained_n
                * 100
                if retained_n > 0
                else np.nan
            )
        })


measurement_threshold_counts = pd.DataFrame(
    threshold_rows
)

print("\n" + "=" * 80)
print("GRID RETENTION BY MINIMUM MEASUREMENT COUNT")
print("=" * 80)

display(
    measurement_threshold_counts.round(2)
)


# ------------------------------------------------------------
# 4. Sparse-grid summary
# ------------------------------------------------------------

sparse_rows = []

for network in [
    "LTE",
    "5G NR"
]:

    counts = measurement_data.loc[
        measurement_data["Network"] == network,
        "Measurement Count"
    ]

    sparse_rows.append({
        "Network": network,
        "Total Grids": len(counts),
        "Exactly 1": int((counts == 1).sum()),
        "Below 5": int((counts < 5).sum()),
        "Below 10": int((counts < 10).sum()),
        "Below 20": int((counts < 20).sum())
    })


sparse_grid_summary = pd.DataFrame(
    sparse_rows
)

print("\n" + "=" * 80)
print("SPARSELY MEASURED GRIDS")
print("=" * 80)

display(
    sparse_grid_summary
)

In [ ]:
# ============================================================
# Phase 0 - Cell 9.
# Visualise Ofcom Measurement-Count Distributions
# ============================================================

import matplotlib.pyplot as plt


fig, axes = plt.subplots(
    2,
    2,
    figsize=(13, 10)
)


NETWORK_COLOURS = {
    "LTE": "#2878B5",
    "5G NR": "#2A9D8F"
}


# ------------------------------------------------------------
# 1. Histograms
# ------------------------------------------------------------

for ax, network in zip(
    axes[0],
    ["LTE", "5G NR"]
):

    counts = measurement_data.loc[
        measurement_data["Network"] == network,
        "Measurement Count"
    ]

    log_counts = np.log10(counts)

    ax.hist(
        log_counts,
        bins=40,
        color=NETWORK_COLOURS[network],
        edgecolor="white",
        alpha=0.85
    )

    median_log = np.log10(
        counts.median()
    )

    ax.axvline(
        median_log,
        color="red",
        linestyle="--",
        linewidth=2,
        label=(
            f"Median = "
            f"{counts.median():,.0f}"
        )
    )

    ax.set_title(
        f"{network}: Measurements per 1-km Grid",
        fontweight="bold"
    )

    ax.set_xlabel(
        "log10(Measurement Count)"
    )

    ax.set_ylabel(
        "Number of Grids"
    )

    ax.legend()

    ax.grid(
        axis="y",
        alpha=0.25
    )


# ------------------------------------------------------------
# 2. Class-specific boxplots
# ------------------------------------------------------------

for ax, network in zip(
    axes[1],
    ["LTE", "5G NR"]
):

    network_data = measurement_data[
        measurement_data["Network"] == network
    ]

    boxplot_data = [
        np.log10(
            network_data.loc[
                network_data["Signal Class"]
                == signal_class,
                "Measurement Count"
            ]
        )
        for signal_class in VALID_CLASSES
    ]

    boxplot = ax.boxplot(
        boxplot_data,
        tick_labels=VALID_CLASSES,
        patch_artist=True,
        showfliers=False
    )

    for patch in boxplot["boxes"]:
        patch.set_facecolor(
            NETWORK_COLOURS[network]
        )
        patch.set_alpha(0.65)

    ax.set_title(
        f"{network}: Counts by Signal Class",
        fontweight="bold"
    )

    ax.set_xlabel(
        "Signal Class"
    )

    ax.set_ylabel(
        "log10(Measurement Count)"
    )

    ax.grid(
        axis="y",
        alpha=0.25
    )


fig.suptitle(
    (
        "Distribution of Ofcom Measurements "
        "across Labelled Grids"
    ),
    fontsize=16,
    fontweight="bold"
)

plt.tight_layout()
plt.show()

## Phase 1: Untreated Three-Class Baseline Classification

Random Forest, XGBoost, and LightGBM are evaluated without class weighting, resampling, or threshold adjustment.

Performance is assessed using accuracy, balanced accuracy, Macro-F1, MCC, and Poor-class precision, recall, and F1. Identical stratified folds are used to support fair model comparison.

### ※ Related dissertation sections

- Section 3.1 (Research Approach—Modelling and Evaluation)
- Section 4.3 (Detailed Analysis of the Final Feature Set)
- Table 4.5 (Overall and Poor-Class Performance of LightGBM)

In [ ]:
# ============================================================
# Phase 1 - Cell 1. Imports
# Untreated Three-Class Baseline
# ============================================================

from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import LabelEncoder
from sklearn.base import clone

from sklearn.ensemble import RandomForestClassifier
from lightgbm import LGBMClassifier
from xgboost import XGBClassifier

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    matthews_corrcoef,
    classification_report,
    confusion_matrix
)

print("Phase 1 imports completed.")

In [ ]:
# ============================================================
# Phase 1 - Cell 2. Untreated Baseline Models
# ============================================================

def create_baseline_models():

    return {
        "RF": RandomForestClassifier(
            n_estimators=100,
            max_depth=20,
            class_weight=None,
            random_state=RANDOM_STATE,
            n_jobs=-1
        ),

        "XGB": XGBClassifier(
            n_estimators=100,
            max_depth=6,
            learning_rate=0.1,
            random_state=RANDOM_STATE,
            eval_metric="mlogloss",
            tree_method="hist",
            verbosity=0,
            n_jobs=-1
        ),

        "LGBM": LGBMClassifier(
            n_estimators=100,
            max_depth=10,
            learning_rate=0.1,
            class_weight=None,
            random_state=RANDOM_STATE,
            verbose=-1,
            n_jobs=-1
        )
    }


baseline_models = create_baseline_models()

print("Untreated baseline models:")
print(list(baseline_models.keys()))

In [ ]:
# ============================================================
# Phase 1 - Cell 3.
# Automatically Report Model Hyperparameters
# ============================================================

import sys
import sklearn
import lightgbm
import xgboost


# ------------------------------------------------------------
# 1. Use the models created by the actual model factory
# ------------------------------------------------------------

hyperparameter_models = create_baseline_models()

expected_model_names = [
    "RF",
    "XGB",
    "LGBM"
]

if list(
    hyperparameter_models.keys()
) != expected_model_names:

    raise ValueError(
        "Unexpected model names: "
        f"{list(hyperparameter_models.keys())}"
    )


# ------------------------------------------------------------
# 2. Extract every parameter automatically
# ------------------------------------------------------------

full_parameter_rows = []

for model_name, model in (
    hyperparameter_models.items()
):

    parameters = model.get_params(
        deep=False
    )

    for parameter, value in sorted(
        parameters.items()
    ):

        full_parameter_rows.append({
            "Model": model_name,
            "Estimator":
                model.__class__.__name__,
            "Parameter": parameter,
            "Value": str(value)
        })


all_model_hyperparameters = pd.DataFrame(
    full_parameter_rows
)


# ------------------------------------------------------------
# 3. Select the important parameters for reporting
# ------------------------------------------------------------

REPORT_PARAMETERS = {
    "RF": [
        "n_estimators",
        "max_depth",
        "criterion",
        "min_samples_split",
        "min_samples_leaf",
        "max_features",
        "bootstrap",
        "class_weight",
        "random_state",
        "n_jobs"
    ],

    "XGB": [
        "n_estimators",
        "max_depth",
        "learning_rate",
        "min_child_weight",
        "subsample",
        "colsample_bytree",
        "gamma",
        "reg_alpha",
        "reg_lambda",
        "tree_method",
        "eval_metric",
        "random_state",
        "n_jobs"
    ],

    "LGBM": [
        "n_estimators",
        "max_depth",
        "learning_rate",
        "num_leaves",
        "min_child_samples",
        "subsample",
        "colsample_bytree",
        "reg_alpha",
        "reg_lambda",
        "boosting_type",
        "class_weight",
        "random_state",
        "n_jobs"
    ]
}


reported_parameter_rows = []

for model_name, parameter_names in (
    REPORT_PARAMETERS.items()
):

    model = hyperparameter_models[
        model_name
    ]

    parameters = model.get_params(
        deep=False
    )

    for parameter in parameter_names:

        value = parameters.get(
            parameter,
            "Not applicable"
        )

        reported_parameter_rows.append({
            "Model": model_name,
            "Parameter": parameter,
            "Value": str(value)
        })


reported_model_hyperparameters = pd.DataFrame(
    reported_parameter_rows
)


# ------------------------------------------------------------
# 4. Create a comparison table
# ------------------------------------------------------------

hyperparameter_comparison = (
    reported_model_hyperparameters
    .pivot(
        index="Parameter",
        columns="Model",
        values="Value"
    )
    .reset_index()
)

comparison_columns = [
    column
    for column in [
        "Parameter",
        "RF",
        "XGB",
        "LGBM"
    ]
    if column in hyperparameter_comparison.columns
]

hyperparameter_comparison = (
    hyperparameter_comparison[
        comparison_columns
    ]
)


# ------------------------------------------------------------
# 5. Record software versions
# ------------------------------------------------------------

software_versions = pd.DataFrame({
    "Software": [
        "Python",
        "pandas",
        "NumPy",
        "scikit-learn",
        "XGBoost",
        "LightGBM"
    ],

    "Version": [
        sys.version.split()[0],
        pd.__version__,
        np.__version__,
        sklearn.__version__,
        xgboost.__version__,
        lightgbm.__version__
    ]
})


# ------------------------------------------------------------
# 6. Record the model-selection strategy
# ------------------------------------------------------------

model_selection_strategy = pd.DataFrame({
    "Item": [
        "Hyperparameter strategy",
        "Random state",
        "Cross-validation",
        "Primary model-selection metric",
        "Class-weight calculation"
    ],

    "Setting": [
        (
            "Fixed model configurations; "
            "no grid search or random search"
        ),
        str(RANDOM_STATE),
        f"{N_SPLITS}-fold cross-validation",
        "Mean cross-validated Macro-F1",
        (
            "Calculated separately from each "
            "training fold where applicable"
        )
    ]
})


# ------------------------------------------------------------
# 7. Display results
# ------------------------------------------------------------

print("=" * 80)
print("KEY MODEL HYPERPARAMETERS")
print("=" * 80)

display(
    reported_model_hyperparameters
)


print("\n" + "=" * 80)
print("HYPERPARAMETER COMPARISON")
print("=" * 80)

display(
    hyperparameter_comparison
)


print("\n" + "=" * 80)
print("SOFTWARE VERSIONS")
print("=" * 80)

display(
    software_versions
)


print("\n" + "=" * 80)
print("MODEL-SELECTION STRATEGY")
print("=" * 80)

display(
    model_selection_strategy
)


print(
    "\nTotal parameters extracted:",
    len(all_model_hyperparameters)
)

print(
    "\n✓ Model hyperparameters extracted successfully."
)

In [ ]:
# ============================================================
# Phase 1 - Cell 4. Prepare Predictors and Targets
# ============================================================

def prepare_xy(
    labelled_df,
    target_col
):

    # --------------------------------------------------------
    # 1. Sort grids for deterministic row order
    # --------------------------------------------------------

    modelling_df = (
        labelled_df
        .sort_values("grid_id")
        .reset_index(drop=True)
        .copy()
    )


    # --------------------------------------------------------
    # 2. Prepare predictors
    # --------------------------------------------------------

    X = modelling_df[
        feature_cols
    ].copy()

    if X.shape[1] != 35:
        raise ValueError(
            f"Expected 35 predictors, but found {X.shape[1]}."
        )

    if X.isna().any().any():
        missing_cols = X.columns[
            X.isna().any()
        ].tolist()

        raise ValueError(
            f"Missing predictor values detected: {missing_cols}"
        )

    if not np.isfinite(
        X.to_numpy(dtype=float)
    ).all():
        raise ValueError(
            "Infinite predictor values detected."
        )


    # --------------------------------------------------------
    # 3. Prepare target
    # --------------------------------------------------------

    y_text = (
        modelling_df[target_col]
        .astype("string")
        .str.strip()
    )

    detected_classes = set(
        y_text.unique().tolist()
    )

    if detected_classes != set(VALID_CLASSES):
        raise ValueError(
            f"Unexpected classes in {target_col}: "
            f"{detected_classes}"
        )


    # --------------------------------------------------------
    # 4. Encode target labels
    # --------------------------------------------------------

    label_encoder = LabelEncoder()

    y = label_encoder.fit_transform(
        y_text
    )

    label_mapping = dict(
        zip(
            label_encoder.classes_,
            label_encoder.transform(
                label_encoder.classes_
            )
        )
    )


    # --------------------------------------------------------
    # 5. Validation summary
    # --------------------------------------------------------

    print("=" * 70)
    print(target_col)
    print("=" * 70)

    print("Rows:", f"{len(X):,}")
    print("Features:", X.shape[1])

    print("\nClass distribution:")
    print(
        y_text
        .value_counts()
        .reindex(VALID_CLASSES)
    )

    print("\nLabel encoding:")
    print(label_mapping)

    return X, y, label_encoder

In [ ]:
# ============================================================
# Phase 1 - Cell 5. Five-Fold OOF Baseline Evaluation
# ============================================================

def run_threeclass_baseline_oof(
    X,
    y,
    le,
    network_name
):

    # --------------------------------------------------------
    # 1. Cross-validation splits
    # --------------------------------------------------------

    cv = StratifiedKFold(
        n_splits=N_SPLITS,
        shuffle=True,
        random_state=RANDOM_STATE
    )

    # Generate once so every model uses identical folds
    cv_splits = list(
        cv.split(X, y)
    )

    models = create_baseline_models()

    all_results = []
    all_predictions = {}
    all_fold_results = {}

    poor_label = int(
        le.transform(["Poor"])[0]
    )


    # --------------------------------------------------------
    # 2. Evaluate each baseline model
    # --------------------------------------------------------

    for model_name, model_template in models.items():

        print("\n" + "=" * 70)
        print(f"{network_name} | {model_name}")
        print("=" * 70)

        # -1 allows unassigned predictions to be detected
        oof_pred = np.full(
            len(y),
            fill_value=-1,
            dtype=int
        )

        validation_count = np.zeros(
            len(y),
            dtype=int
        )

        fold_results = []


        # ----------------------------------------------------
        # 3. Fold-level training and validation
        # ----------------------------------------------------

        for fold, (train_idx, val_idx) in enumerate(
            cv_splits,
            start=1
        ):

            X_train = X.iloc[
                train_idx
            ].copy()

            X_val = X.iloc[
                val_idx
            ].copy()

            y_train = y[
                train_idx
            ]

            y_val = y[
                val_idx
            ]

            # Fresh independent model for every fold
            model = clone(
                model_template
            )

            model.fit(
                X_train,
                y_train
            )

            y_pred = model.predict(
                X_val
            ).astype(int)

            oof_pred[
                val_idx
            ] = y_pred

            validation_count[
                val_idx
            ] += 1


            # ------------------------------------------------
            # 4. Fold metrics
            # ------------------------------------------------

            fold_result = {
                "Fold": fold,

                "Accuracy": accuracy_score(
                    y_val,
                    y_pred
                ),

                "Balanced Accuracy":
                    balanced_accuracy_score(
                        y_val,
                        y_pred
                    ),

                "Macro-F1": f1_score(
                    y_val,
                    y_pred,
                    average="macro",
                    zero_division=0
                ),

                "MCC": matthews_corrcoef(
                    y_val,
                    y_pred
                ),

                "Poor Precision": precision_score(
                    y_val,
                    y_pred,
                    labels=[poor_label],
                    average=None,
                    zero_division=0
                )[0],

                "Poor Recall": recall_score(
                    y_val,
                    y_pred,
                    labels=[poor_label],
                    average=None,
                    zero_division=0
                )[0],

                "Poor F1": f1_score(
                    y_val,
                    y_pred,
                    labels=[poor_label],
                    average=None,
                    zero_division=0
                )[0],

                "Poor Support": int(
                    np.sum(
                        y_val == poor_label
                    )
                )
            }

            fold_results.append(
                fold_result
            )

            print(
                f"Fold {fold}: "
                f"Macro-F1 = "
                f"{fold_result['Macro-F1']:.4f}, "
                f"Poor F1 = "
                f"{fold_result['Poor F1']:.4f}"
            )


        # ----------------------------------------------------
        # 5. Validate OOF assignment
        # ----------------------------------------------------

        if np.any(oof_pred == -1):
            raise RuntimeError(
                f"{network_name} | {model_name}: "
                "some observations have no OOF prediction."
            )

        if not np.all(
            validation_count == 1
        ):
            raise RuntimeError(
                f"{network_name} | {model_name}: "
                "each observation must appear exactly once "
                "in a validation fold."
            )


        # ----------------------------------------------------
        # 6. Overall OOF metrics
        # ----------------------------------------------------

        fold_results_df = pd.DataFrame(
            fold_results
        )

        result = {
            "Network": network_name,
            "Experiment": "Untreated Baseline",
            "Model": model_name,

            "N Rows": len(y),
            "N Features": X.shape[1],

            "Accuracy": accuracy_score(
                y,
                oof_pred
            ),

            "Balanced Accuracy":
                balanced_accuracy_score(
                    y,
                    oof_pred
                ),

            "Macro-F1": f1_score(
                y,
                oof_pred,
                average="macro",
                zero_division=0
            ),

            "MCC": matthews_corrcoef(
                y,
                oof_pred
            ),

            "Poor Precision": precision_score(
                y,
                oof_pred,
                labels=[poor_label],
                average=None,
                zero_division=0
            )[0],

            "Poor Recall": recall_score(
                y,
                oof_pred,
                labels=[poor_label],
                average=None,
                zero_division=0
            )[0],

            "Poor F1": f1_score(
                y,
                oof_pred,
                labels=[poor_label],
                average=None,
                zero_division=0
            )[0],

            "Poor Support": int(
                np.sum(
                    y == poor_label
                )
            ),

            "CV Macro-F1 Mean":
                fold_results_df[
                    "Macro-F1"
                ].mean(),

            "CV Macro-F1 SD":
                fold_results_df[
                    "Macro-F1"
                ].std(ddof=0)
        }

        all_results.append(
            result
        )

        all_predictions[
            model_name
        ] = oof_pred.copy()

        all_fold_results[
            model_name
        ] = fold_results_df


        # ----------------------------------------------------
        # 7. Display model summary
        # ----------------------------------------------------

        print("\nOOF Summary")

        print(
            f"Accuracy        : "
            f"{result['Accuracy']:.4f}"
        )

        print(
            f"Balanced Accuracy: "
            f"{result['Balanced Accuracy']:.4f}"
        )

        print(
            f"Macro-F1        : "
            f"{result['Macro-F1']:.4f}"
        )

        print(
            f"CV Macro-F1     : "
            f"{result['CV Macro-F1 Mean']:.4f} "
            f"± {result['CV Macro-F1 SD']:.4f}"
        )

        print(
            f"Poor Precision  : "
            f"{result['Poor Precision']:.4f}"
        )

        print(
            f"Poor Recall     : "
            f"{result['Poor Recall']:.4f}"
        )

        print(
            f"Poor F1         : "
            f"{result['Poor F1']:.4f}"
        )

        print(
            f"MCC             : "
            f"{result['MCC']:.4f}"
        )


    return (
        pd.DataFrame(
            all_results
        ),
        all_predictions,
        all_fold_results
    )

In [ ]:
# ============================================================
# Phase 1 - Cell 6. LTE Untreated Baseline Evaluation
# ============================================================

X_lte, y_lte, le_lte = prepare_xy(
    labelled_df=lte_df,
    target_col="lte_signal_class"
)

(
    lte_baseline_results,
    lte_baseline_predictions,
    lte_baseline_folds
) = run_threeclass_baseline_oof(
    X=X_lte,
    y=y_lte,
    le=le_lte,
    network_name="LTE"
)

lte_display_cols = [
    "Network",
    "Experiment",
    "Model",
    "N Rows",
    "N Features",
    "Accuracy",
    "Balanced Accuracy",
    "Macro-F1",
    "CV Macro-F1 Mean",
    "CV Macro-F1 SD",
    "MCC",
    "Poor Precision",
    "Poor Recall",
    "Poor F1",
    "Poor Support"
]

print("\nLTE Untreated Baseline Results")

display(
    lte_baseline_results[
        lte_display_cols
    ].round(4)
)

In [ ]:
# ============================================================
# Phase 1 - Cell 7. 5G NR Untreated Baseline Evaluation
# ============================================================

X_nr, y_nr, le_nr = prepare_xy(
    labelled_df=nr_df,
    target_col="nr_signal_class"
)

(
    nr_baseline_results,
    nr_baseline_predictions,
    nr_baseline_folds
) = run_threeclass_baseline_oof(
    X=X_nr,
    y=y_nr,
    le=le_nr,
    network_name="5G NR"
)

nr_display_cols = [
    "Network",
    "Experiment",
    "Model",
    "N Rows",
    "N Features",
    "Accuracy",
    "Balanced Accuracy",
    "Macro-F1",
    "CV Macro-F1 Mean",
    "CV Macro-F1 SD",
    "MCC",
    "Poor Precision",
    "Poor Recall",
    "Poor F1",
    "Poor Support"
]

print("\n5G NR Untreated Baseline Results")

display(
    nr_baseline_results[
        nr_display_cols
    ].round(4)
)

In [ ]:
# ============================================================
# Phase 1 - Cell 8. Combined Untreated Baseline Results
# ============================================================

baseline_results = pd.concat(
    [
        lte_baseline_results,
        nr_baseline_results
    ],
    ignore_index=True
)

# ------------------------------------------------------------
# Fix display order
# ------------------------------------------------------------

network_order = [
    "LTE",
    "5G NR"
]

model_order = [
    "RF",
    "XGB",
    "LGBM"
]

baseline_results["Network"] = pd.Categorical(
    baseline_results["Network"],
    categories=network_order,
    ordered=True
)

baseline_results["Model"] = pd.Categorical(
    baseline_results["Model"],
    categories=model_order,
    ordered=True
)

baseline_results = (
    baseline_results
    .sort_values(
        ["Network", "Model"]
    )
    .reset_index(drop=True)
)


# ------------------------------------------------------------
# Select result columns
# ------------------------------------------------------------

baseline_display_cols = [
    "Network",
    "Experiment",
    "Model",
    "N Rows",
    "N Features",
    "Accuracy",
    "Balanced Accuracy",
    "Macro-F1",
    "CV Macro-F1 Mean",
    "CV Macro-F1 SD",
    "MCC",
    "Poor Precision",
    "Poor Recall",
    "Poor F1",
    "Poor Support"
]

print("Combined Untreated Three-Class Baseline Results")

display(
    baseline_results[
        baseline_display_cols
    ].round(4)
)

In [ ]:
# ============================================================
# Phase 1 - Cell 9. Best Baseline Model by CV Macro-F1
# ============================================================

best_indices = (
    baseline_results
    .groupby(
        "Network",
        observed=True
    )["CV Macro-F1 Mean"]
    .idxmax()
)

best_baseline = (
    baseline_results
    .loc[best_indices]
    .sort_values("Network")
    .reset_index(drop=True)
)

best_baseline_display_cols = [
    "Network",
    "Experiment",
    "Model",
    "N Rows",
    "N Features",
    "Accuracy",
    "Balanced Accuracy",
    "Macro-F1",
    "CV Macro-F1 Mean",
    "CV Macro-F1 SD",
    "MCC",
    "Poor Precision",
    "Poor Recall",
    "Poor F1",
    "Poor Support"
]

print("Best Untreated Baseline Model by Mean CV Macro-F1")

display(
    best_baseline[
        best_baseline_display_cols
    ].round(4)
)

In [ ]:
# ============================================================
# Phase 1 - Cell 10. Baseline Reproducibility Sanity Check
# ============================================================

EXPECTED_CV_MACRO_F1 = {
    ("LTE", "RF"): 0.3938,
    ("LTE", "XGB"): 0.4112,
    ("LTE", "LGBM"): 0.4271,

    ("5G NR", "RF"): 0.5169,
    ("5G NR", "XGB"): 0.5276,
    ("5G NR", "LGBM"): 0.5228
}

TOLERANCE = 0.0005

sanity_rows = []

for (
    network,
    model_name
), expected_score in EXPECTED_CV_MACRO_F1.items():

    matched_rows = baseline_results[
        (baseline_results["Network"] == network)
        & (baseline_results["Model"] == model_name)
    ]

    if len(matched_rows) != 1:
        raise ValueError(
            f"Expected one result for {network} | {model_name}, "
            f"but found {len(matched_rows)}."
        )

    result_row = matched_rows.iloc[0]

    observed_score = float(
        result_row["CV Macro-F1 Mean"]
    )

    difference = (
        observed_score - expected_score
    )

    matched = np.isclose(
        observed_score,
        expected_score,
        atol=TOLERANCE,
        rtol=0
    )

    sanity_rows.append({
        "Network": network,
        "Model": model_name,
        "Expected CV Macro-F1": expected_score,
        "Observed CV Macro-F1": observed_score,
        "Difference": difference,
        "Matched": matched
    })


sanity_check = pd.DataFrame(
    sanity_rows
)

print("=" * 70)
print("BASELINE REPRODUCIBILITY SANITY CHECK")
print("=" * 70)

display(
    sanity_check.round(4)
)

if not sanity_check["Matched"].all():

    failed_results = sanity_check[
        ~sanity_check["Matched"]
    ]

    raise ValueError(
        "Baseline results do not match the expected results:\n"
        f"{failed_results}"
    )


# ------------------------------------------------------------
# Confirm the best model for each network
# ------------------------------------------------------------

best_model_check = (
    baseline_results
    .loc[
        baseline_results
        .groupby(
            "Network",
            observed=True
        )["CV Macro-F1 Mean"]
        .idxmax()
    ]
    [
        [
            "Network",
            "Model",
            "CV Macro-F1 Mean",
            "CV Macro-F1 SD",
            "Macro-F1",
            "Poor F1"
        ]
    ]
    .sort_values("Network")
    .reset_index(drop=True)
)

print("\nBest baseline models:")
display(
    best_model_check.round(4)
)

assert (
    best_model_check.loc[
        best_model_check["Network"] == "LTE",
        "Model"
    ].iloc[0] == "LGBM"
), "Unexpected best LTE baseline model."

assert (
    best_model_check.loc[
        best_model_check["Network"] == "5G NR",
        "Model"
    ].iloc[0] == "XGB"
), "Unexpected best 5G NR baseline model."

print("\n✓ Phase 1 baseline results reproduced successfully.")

## Phase 2: Three-Class Imbalance Handling

Five imbalance-handling configurations are evaluated:

- untreated baseline;
- class weighting;
- SMOTE;
- SMOTE-Tomek; and
- SMOTE combined with class weighting.

Class weights and resampling procedures are calculated or fitted using the training fold only. Selected baseline and imbalance-treated models are subsequently compared using pooled out-of-fold confusion matrices.

### ※ Related dissertation sections

- Section 2.8 (Class Imbalance and Poor-Signal Detection)
- Section 4.4 (Class-Imbalance Treatment)
- Table 4.6 (Baseline and Selected Class-Imbalance Treatment Results)
- Figure 4.9 (Three-Class Imbalance-Handling Workflow)
- Figure 4.10 (Baseline and Class-Weighted Confusion Matrices)

In [ ]:
# ============================================================
# Phase 2 - Cell 1. Imports
# Three-Class Imbalance Handling
# ============================================================

from imblearn.over_sampling import SMOTE
from imblearn.combine import SMOTETomek

from sklearn.preprocessing import StandardScaler
from sklearn.utils.class_weight import (
    compute_class_weight,
    compute_sample_weight
)

print("Phase 2 imports completed.")

In [ ]:
# ============================================================
# Phase 2 - Cell 2. Three-Class Imbalance Methods
# ============================================================

IMBALANCE_METHODS = [
    "Baseline",
    "Class Weight",
    "SMOTE",
    "SMOTE-Tomek",
    "SMOTE + Class Weight"
]

print(
    "Number of imbalance methods:",
    len(IMBALANCE_METHODS)
)

print("\nImbalance methods:")

for index, method in enumerate(
    IMBALANCE_METHODS,
    start=1
):
    print(f"{index}. {method}")

In [ ]:
# ============================================================
# Phase 2 - Cell 3. Fold-Specific Class and Sample Weights
# ============================================================
# Class weights are calculated using only the original
# training fold to prevent information leakage.

def calculate_class_weight_dict(
    y_train
):

    classes = np.unique(
        y_train
    )

    class_weights = compute_class_weight(
        class_weight="balanced",
        classes=classes,
        y=y_train
    )

    weight_dict = {
        int(label): float(weight)
        for label, weight in zip(
            classes,
            class_weights
        )
    }

    return weight_dict


def create_sample_weights(
    y,
    weight_dict
):

    unknown_labels = (
        set(np.unique(y))
        - set(weight_dict.keys())
    )

    if unknown_labels:
        raise ValueError(
            "Labels without class weights detected: "
            f"{unknown_labels}"
        )

    sample_weights = np.asarray(
        [
            weight_dict[int(label)]
            for label in y
        ],
        dtype=float
    )

    return sample_weights

In [ ]:
# ============================================================
# Phase 2 - Cell 4.
# Three-Class Imbalance Evaluation with Five-Fold OOF
# ============================================================

def run_threeclass_imbalance_oof(
    X,
    y,
    le,
    network_name
):

    # --------------------------------------------------------
    # 1. Create identical CV splits for every experiment
    # --------------------------------------------------------

    cv = StratifiedKFold(
        n_splits=N_SPLITS,
        shuffle=True,
        random_state=RANDOM_STATE
    )

    cv_splits = list(
        cv.split(X, y)
    )

    model_templates = create_baseline_models()

    all_results = []
    all_predictions = {}
    all_fold_results = {}

    poor_label = int(
        le.transform(["Poor"])[0]
    )


    # --------------------------------------------------------
    # 2. Evaluate each method and model
    # --------------------------------------------------------

    for method in IMBALANCE_METHODS:

        for model_name, model_template in (
            model_templates.items()
        ):

            print("\n" + "=" * 75)

            print(
                f"{network_name} | "
                f"{method} | "
                f"{model_name}"
            )

            print("=" * 75)

            oof_pred = np.full(
                len(y),
                fill_value=-1,
                dtype=int
            )

            validation_count = np.zeros(
                len(y),
                dtype=int
            )

            fold_results = []


            # ------------------------------------------------
            # 3. Five-fold cross-validation
            # ------------------------------------------------

            for fold, (train_idx, val_idx) in enumerate(
                cv_splits,
                start=1
            ):

                X_train = X.iloc[
                    train_idx
                ].copy()

                X_val = X.iloc[
                    val_idx
                ].copy()

                y_train = y[
                    train_idx
                ]

                y_val = y[
                    val_idx
                ]


                # --------------------------------------------
                # Default: untreated training data
                # --------------------------------------------

                X_fit = X_train.copy()
                X_predict = X_val.copy()
                y_fit = y_train.copy()

                sample_weight = None


                # --------------------------------------------
                # Baseline
                # --------------------------------------------

                if method == "Baseline":

                    pass


                # --------------------------------------------
                # Class Weight
                # --------------------------------------------

                elif method == "Class Weight":

                    weight_dict = (
                        calculate_class_weight_dict(
                            y_train
                        )
                    )

                    sample_weight = (
                        create_sample_weights(
                            y_train,
                            weight_dict
                        )
                    )


                # --------------------------------------------
                # SMOTE-based methods
                # --------------------------------------------

                elif method in [
                    "SMOTE",
                    "SMOTE-Tomek",
                    "SMOTE + Class Weight"
                ]:

                    # Fit scaler using the training fold only
                    scaler = StandardScaler()

                    X_train_scaled = pd.DataFrame(
                        scaler.fit_transform(
                            X_train
                        ),
                        columns=X.columns,
                        index=X_train.index
                    )

                    X_predict = pd.DataFrame(
                        scaler.transform(
                            X_val
                        ),
                        columns=X.columns,
                        index=X_val.index
                    )


                    # Select resampling method
                    if method in [
                        "SMOTE",
                        "SMOTE + Class Weight"
                    ]:

                        sampler = SMOTE(
                            random_state=RANDOM_STATE,
                            k_neighbors=5
                        )

                    else:

                        sampler = SMOTETomek(
                            random_state=RANDOM_STATE,
                            smote=SMOTE(
                                random_state=RANDOM_STATE,
                                k_neighbors=5
                            )
                        )


                    # Resample training fold only
                    X_fit, y_fit = (
                        sampler.fit_resample(
                            X_train_scaled,
                            y_train
                        )
                    )


                    # ----------------------------------------
                    # SMOTE plus original class weights
                    # ----------------------------------------

                    if method == "SMOTE + Class Weight":

                        # Calculate using original y_train
                        weight_dict = (
                            calculate_class_weight_dict(
                                y_train
                            )
                        )

                        # Map original weights to resampled data
                        sample_weight = (
                            create_sample_weights(
                                y_fit,
                                weight_dict
                            )
                        )


                else:

                    raise ValueError(
                        f"Unknown imbalance method: {method}"
                    )


                # --------------------------------------------
                # 4. Fit a fresh model
                # --------------------------------------------

                model = clone(
                    model_template
                )

                if sample_weight is None:

                    model.fit(
                        X_fit,
                        y_fit
                    )

                else:

                    model.fit(
                        X_fit,
                        y_fit,
                        sample_weight=sample_weight
                    )


                # --------------------------------------------
                # 5. Validation prediction
                # --------------------------------------------

                y_pred = model.predict(
                    X_predict
                ).astype(int)

                oof_pred[
                    val_idx
                ] = y_pred

                validation_count[
                    val_idx
                ] += 1


                # --------------------------------------------
                # 6. Fold-level metrics
                # --------------------------------------------

                fold_result = {
                    "Fold": fold,

                    "Accuracy": accuracy_score(
                        y_val,
                        y_pred
                    ),

                    "Balanced Accuracy":
                        balanced_accuracy_score(
                            y_val,
                            y_pred
                        ),

                    "Macro-F1": f1_score(
                        y_val,
                        y_pred,
                        average="macro",
                        zero_division=0
                    ),

                    "MCC": matthews_corrcoef(
                        y_val,
                        y_pred
                    ),

                    "Poor Precision": precision_score(
                        y_val,
                        y_pred,
                        labels=[poor_label],
                        average=None,
                        zero_division=0
                    )[0],

                    "Poor Recall": recall_score(
                        y_val,
                        y_pred,
                        labels=[poor_label],
                        average=None,
                        zero_division=0
                    )[0],

                    "Poor F1": f1_score(
                        y_val,
                        y_pred,
                        labels=[poor_label],
                        average=None,
                        zero_division=0
                    )[0],

                    "Poor Support": int(
                        np.sum(
                            y_val == poor_label
                        )
                    ),

                    "Training Rows Before":
                        len(y_train),

                    "Training Rows After":
                        len(y_fit)
                }

                fold_results.append(
                    fold_result
                )

                print(
                    f"Fold {fold}: "
                    f"Macro-F1 = "
                    f"{fold_result['Macro-F1']:.4f}, "
                    f"Poor Recall = "
                    f"{fold_result['Poor Recall']:.4f}, "
                    f"Poor F1 = "
                    f"{fold_result['Poor F1']:.4f}"
                )


            # ------------------------------------------------
            # 7. Validate OOF assignment
            # ------------------------------------------------

            if np.any(oof_pred == -1):

                raise RuntimeError(
                    f"{network_name} | {method} | "
                    f"{model_name}: missing OOF predictions."
                )

            if not np.all(
                validation_count == 1
            ):

                raise RuntimeError(
                    f"{network_name} | {method} | "
                    f"{model_name}: invalid OOF assignment."
                )


            # ------------------------------------------------
            # 8. Overall OOF results
            # ------------------------------------------------

            fold_results_df = pd.DataFrame(
                fold_results
            )

            result = {
                "Network": network_name,
                "Method": method,
                "Model": model_name,

                "N Rows": len(y),
                "N Features": X.shape[1],

                "Accuracy": accuracy_score(
                    y,
                    oof_pred
                ),

                "Balanced Accuracy":
                    balanced_accuracy_score(
                        y,
                        oof_pred
                    ),

                "Macro-F1": f1_score(
                    y,
                    oof_pred,
                    average="macro",
                    zero_division=0
                ),

                "MCC": matthews_corrcoef(
                    y,
                    oof_pred
                ),

                "Poor Precision": precision_score(
                    y,
                    oof_pred,
                    labels=[poor_label],
                    average=None,
                    zero_division=0
                )[0],

                "Poor Recall": recall_score(
                    y,
                    oof_pred,
                    labels=[poor_label],
                    average=None,
                    zero_division=0
                )[0],

                "Poor F1": f1_score(
                    y,
                    oof_pred,
                    labels=[poor_label],
                    average=None,
                    zero_division=0
                )[0],

                "Poor Support": int(
                    np.sum(
                        y == poor_label
                    )
                ),

                "CV Macro-F1 Mean":
                    fold_results_df[
                        "Macro-F1"
                    ].mean(),

                "CV Macro-F1 SD":
                    fold_results_df[
                        "Macro-F1"
                    ].std(ddof=0)
            }

            all_results.append(
                result
            )

            result_key = (
                method,
                model_name
            )

            all_predictions[
                result_key
            ] = oof_pred.copy()

            all_fold_results[
                result_key
            ] = fold_results_df


            # ------------------------------------------------
            # 9. Display experiment summary
            # ------------------------------------------------

            print("\nOOF Summary")

            print(
                f"Macro-F1       : "
                f"{result['Macro-F1']:.4f}"
            )

            print(
                f"CV Macro-F1    : "
                f"{result['CV Macro-F1 Mean']:.4f} "
                f"± {result['CV Macro-F1 SD']:.4f}"
            )

            print(
                f"Balanced Acc.  : "
                f"{result['Balanced Accuracy']:.4f}"
            )

            print(
                f"Poor Precision : "
                f"{result['Poor Precision']:.4f}"
            )

            print(
                f"Poor Recall    : "
                f"{result['Poor Recall']:.4f}"
            )

            print(
                f"Poor F1        : "
                f"{result['Poor F1']:.4f}"
            )


    return (
        pd.DataFrame(
            all_results
        ),
        all_predictions,
        all_fold_results
    )

In [ ]:
# ============================================================
# Phase 2 - Cell 5. LTE Three-Class Imbalance Experiments
# ============================================================

(
    lte_imbalance_results,
    lte_imbalance_predictions,
    lte_imbalance_folds
) = run_threeclass_imbalance_oof(
    X=X_lte,
    y=y_lte,
    le=le_lte,
    network_name="LTE"
)

lte_imbalance_display_cols = [
    "Network",
    "Method",
    "Model",
    "N Rows",
    "N Features",
    "Accuracy",
    "Balanced Accuracy",
    "Macro-F1",
    "CV Macro-F1 Mean",
    "CV Macro-F1 SD",
    "MCC",
    "Poor Precision",
    "Poor Recall",
    "Poor F1",
    "Poor Support"
]

print("\nLTE Three-Class Imbalance-Handling Results")

display(
    lte_imbalance_results[
        lte_imbalance_display_cols
    ].round(4)
)

assert len(lte_imbalance_results) == 15, (
    "Expected 15 LTE experiment results."
)

In [ ]:
# ============================================================
# Phase 2 - Cell 6. 5G NR Three-Class Imbalance Experiments
# ============================================================

(
    nr_imbalance_results,
    nr_imbalance_predictions,
    nr_imbalance_folds
) = run_threeclass_imbalance_oof(
    X=X_nr,
    y=y_nr,
    le=le_nr,
    network_name="5G NR"
)

nr_imbalance_display_cols = [
    "Network",
    "Method",
    "Model",
    "N Rows",
    "N Features",
    "Accuracy",
    "Balanced Accuracy",
    "Macro-F1",
    "CV Macro-F1 Mean",
    "CV Macro-F1 SD",
    "MCC",
    "Poor Precision",
    "Poor Recall",
    "Poor F1",
    "Poor Support"
]

print("\n5G NR Three-Class Imbalance-Handling Results")

display(
    nr_imbalance_results[
        nr_imbalance_display_cols
    ].round(4)
)

assert len(nr_imbalance_results) == 15, (
    "Expected 15 5G NR experiment results."
)

In [ ]:
# ============================================================
# Phase 2 - Cell 7. Combined Three-Class Imbalance Results
# ============================================================

imbalance_results = pd.concat(
    [
        lte_imbalance_results,
        nr_imbalance_results
    ],
    ignore_index=True
)

network_order = [
    "LTE",
    "5G NR"
]

method_order = [
    "Baseline",
    "Class Weight",
    "SMOTE",
    "SMOTE-Tomek",
    "SMOTE + Class Weight"
]

model_order = [
    "RF",
    "XGB",
    "LGBM"
]

imbalance_results["Network"] = pd.Categorical(
    imbalance_results["Network"],
    categories=network_order,
    ordered=True
)

imbalance_results["Method"] = pd.Categorical(
    imbalance_results["Method"],
    categories=method_order,
    ordered=True
)

imbalance_results["Model"] = pd.Categorical(
    imbalance_results["Model"],
    categories=model_order,
    ordered=True
)

imbalance_results = (
    imbalance_results
    .sort_values(
        [
            "Network",
            "Method",
            "Model"
        ]
    )
    .reset_index(drop=True)
)

imbalance_display_cols = [
    "Network",
    "Method",
    "Model",
    "Accuracy",
    "Balanced Accuracy",
    "Macro-F1",
    "CV Macro-F1 Mean",
    "CV Macro-F1 SD",
    "MCC",
    "Poor Precision",
    "Poor Recall",
    "Poor F1",
    "Poor Support"
]

assert len(imbalance_results) == 30, (
    f"Expected 30 results, but found "
    f"{len(imbalance_results)}."
)

print("Combined Three-Class Imbalance-Handling Results")
print("Total experiment results:", len(imbalance_results))

display(
    imbalance_results[
        imbalance_display_cols
    ].round(4)
)

In [ ]:
# ============================================================
# Phase 2 - Cell 8.
# Best Configuration by Mean CV Macro-F1
# ============================================================

best_macro_indices = (
    imbalance_results
    .groupby(
        "Network",
        observed=True
    )["CV Macro-F1 Mean"]
    .idxmax()
)

best_macro = (
    imbalance_results
    .loc[best_macro_indices]
    .sort_values("Network")
    .reset_index(drop=True)
)

best_macro_cols = [
    "Network",
    "Method",
    "Model",
    "Accuracy",
    "Balanced Accuracy",
    "Macro-F1",
    "CV Macro-F1 Mean",
    "CV Macro-F1 SD",
    "MCC",
    "Poor Precision",
    "Poor Recall",
    "Poor F1",
    "Poor Support"
]

print(
    "Best Three-Class Configuration "
    "by Mean CV Macro-F1"
)

display(
    best_macro[
        best_macro_cols
    ].round(4)
)

In [ ]:
# ============================================================
# Phase 2 - Cell 9.
# Best Configuration by OOF Poor-Class F1
# ============================================================

best_poor_indices = (
    imbalance_results
    .groupby(
        "Network",
        observed=True
    )["Poor F1"]
    .idxmax()
)

best_poor = (
    imbalance_results
    .loc[best_poor_indices]
    .sort_values("Network")
    .reset_index(drop=True)
)

best_poor_cols = [
    "Network",
    "Method",
    "Model",
    "Accuracy",
    "Balanced Accuracy",
    "Macro-F1",
    "CV Macro-F1 Mean",
    "CV Macro-F1 SD",
    "MCC",
    "Poor Precision",
    "Poor Recall",
    "Poor F1",
    "Poor Support"
]

print(
    "Best Three-Class Configuration "
    "by OOF Poor-Class F1"
)

display(
    best_poor[
        best_poor_cols
    ].round(4)
)

In [ ]:
# ============================================================
# Phase 2 - Cell 10.
# Baseline versus Best Imbalance Treatment
# ============================================================

comparison_rows = []

for network in [
    "LTE",
    "5G NR"
]:

    network_results = imbalance_results[
        imbalance_results["Network"] == network
    ].copy()


    # --------------------------------------------------------
    # 1. Best untreated baseline by mean CV Macro-F1
    # --------------------------------------------------------

    baseline_candidates = network_results[
        network_results["Method"] == "Baseline"
    ]

    baseline_index = (
        baseline_candidates[
            "CV Macro-F1 Mean"
        ].idxmax()
    )

    baseline = network_results.loc[
        baseline_index
    ]


    # --------------------------------------------------------
    # 2. Best non-baseline treatment by OOF Poor F1
    # --------------------------------------------------------

    treatment_candidates = network_results[
        network_results["Method"] != "Baseline"
    ]

    best_index = (
        treatment_candidates[
            "Poor F1"
        ].idxmax()
    )

    best = network_results.loc[
        best_index
    ]


    # --------------------------------------------------------
    # 3. Comparison
    # --------------------------------------------------------

    comparison_rows.append({
        "Network": network,

        "Baseline Model":
            baseline["Model"],

        "Baseline CV Macro-F1":
            baseline["CV Macro-F1 Mean"],

        "Baseline Accuracy":
            baseline["Accuracy"],

        "Baseline Poor Precision":
            baseline["Poor Precision"],

        "Baseline Poor Recall":
            baseline["Poor Recall"],

        "Baseline Poor F1":
            baseline["Poor F1"],

        "Best Treatment":
            best["Method"],

        "Treatment Model":
            best["Model"],

        "Treatment CV Macro-F1":
            best["CV Macro-F1 Mean"],

        "Treatment Accuracy":
            best["Accuracy"],

        "Treatment Poor Precision":
            best["Poor Precision"],

        "Treatment Poor Recall":
            best["Poor Recall"],

        "Treatment Poor F1":
            best["Poor F1"],

        "CV Macro-F1 Change":
            best["CV Macro-F1 Mean"]
            - baseline["CV Macro-F1 Mean"],

        "Accuracy Change":
            best["Accuracy"]
            - baseline["Accuracy"],

        "Poor Precision Change":
            best["Poor Precision"]
            - baseline["Poor Precision"],

        "Poor Recall Change":
            best["Poor Recall"]
            - baseline["Poor Recall"],

        "Poor F1 Change":
            best["Poor F1"]
            - baseline["Poor F1"]
    })


baseline_vs_imbalance = pd.DataFrame(
    comparison_rows
)

print(
    "Untreated Baseline versus Best "
    "Imbalance Treatment by Poor-Class F1"
)

display(
    baseline_vs_imbalance.round(4)
)

In [ ]:
# ============================================================
# Phase 2 - Cell 11.
# Baseline versus Selected Imbalance-Treatment Confusion Matrices
# ============================================================

import matplotlib.pyplot as plt

from sklearn.metrics import (
    confusion_matrix,
    ConfusionMatrixDisplay
)


# ------------------------------------------------------------
# 1. Define selected configurations
# ------------------------------------------------------------

LTE_BASELINE_KEY = (
    "Baseline",
    "LGBM"
)

LTE_TREATMENT_KEY = (
    "Class Weight",
    "LGBM"
)

NR_BASELINE_KEY = (
    "Baseline",
    "XGB"
)

NR_TREATMENT_KEY = (
    "Class Weight",
    "LGBM"
)


# ------------------------------------------------------------
# 2. Retrieve OOF predictions
# ------------------------------------------------------------

lte_baseline_pred = (
    lte_imbalance_predictions[
        LTE_BASELINE_KEY
    ]
)

lte_treatment_pred = (
    lte_imbalance_predictions[
        LTE_TREATMENT_KEY
    ]
)

nr_baseline_pred = (
    nr_imbalance_predictions[
        NR_BASELINE_KEY
    ]
)

nr_treatment_pred = (
    nr_imbalance_predictions[
        NR_TREATMENT_KEY
    ]
)


# ------------------------------------------------------------
# 3. Fix class order
# ------------------------------------------------------------

CLASS_ORDER = [
    "Excellent",
    "Good",
    "Poor"
]

lte_label_order = le_lte.transform(
    CLASS_ORDER
)

nr_label_order = le_nr.transform(
    CLASS_ORDER
)


# ------------------------------------------------------------
# 4. Create confusion matrices
# ------------------------------------------------------------

lte_baseline_cm = confusion_matrix(
    y_lte,
    lte_baseline_pred,
    labels=lte_label_order
)

lte_treatment_cm = confusion_matrix(
    y_lte,
    lte_treatment_pred,
    labels=lte_label_order
)

nr_baseline_cm = confusion_matrix(
    y_nr,
    nr_baseline_pred,
    labels=nr_label_order
)

nr_treatment_cm = confusion_matrix(
    y_nr,
    nr_treatment_pred,
    labels=nr_label_order
)


# ------------------------------------------------------------
# 5. Obtain performance values for figure titles
# ------------------------------------------------------------

def get_experiment_result(
    results,
    method,
    model
):

    matched = results[
        (results["Method"] == method)
        & (results["Model"] == model)
    ]

    if len(matched) != 1:
        raise ValueError(
            f"Expected one result for "
            f"{method} + {model}, "
            f"but found {len(matched)}."
        )

    return matched.iloc[0]


lte_baseline_result = get_experiment_result(
    lte_imbalance_results,
    *LTE_BASELINE_KEY
)

lte_treatment_result = get_experiment_result(
    lte_imbalance_results,
    *LTE_TREATMENT_KEY
)

nr_baseline_result = get_experiment_result(
    nr_imbalance_results,
    *NR_BASELINE_KEY
)

nr_treatment_result = get_experiment_result(
    nr_imbalance_results,
    *NR_TREATMENT_KEY
)


# ------------------------------------------------------------
# 6. Use the same colour scale within each network
# ------------------------------------------------------------

lte_vmax = max(
    lte_baseline_cm.max(),
    lte_treatment_cm.max()
)

nr_vmax = max(
    nr_baseline_cm.max(),
    nr_treatment_cm.max()
)


# ------------------------------------------------------------
# 7. Create 2 × 2 comparison figure
# ------------------------------------------------------------

fig, axes = plt.subplots(
    2,
    2,
    figsize=(13, 11)
)


# LTE baseline
disp1 = ConfusionMatrixDisplay(
    confusion_matrix=lte_baseline_cm,
    display_labels=CLASS_ORDER
)

disp1.plot(
    ax=axes[0, 0],
    cmap="Blues",
    values_format="d",
    colorbar=False,
    im_kw={
        "vmin": 0,
        "vmax": lte_vmax
    }
)

axes[0, 0].set_title(
    (
        "(a) LTE Baseline: LGBM\n"
        f"Macro-F1 = "
        f"{lte_baseline_result['Macro-F1']:.3f}, "
        f"Poor F1 = "
        f"{lte_baseline_result['Poor F1']:.3f}"
    ),
    fontsize=14,
    fontweight="bold",
    pad=10
)


# LTE class weighting
disp2 = ConfusionMatrixDisplay(
    confusion_matrix=lte_treatment_cm,
    display_labels=CLASS_ORDER
)

disp2.plot(
    ax=axes[0, 1],
    cmap="Blues",
    values_format="d",
    colorbar=False,
    im_kw={
        "vmin": 0,
        "vmax": lte_vmax
    }
)

axes[0, 1].set_title(
    (
        "(b) LTE Class Weight: LGBM\n"
        f"Macro-F1 = "
        f"{lte_treatment_result['Macro-F1']:.3f}, "
        f"Poor F1 = "
        f"{lte_treatment_result['Poor F1']:.3f}"
    ),
    fontsize=14,
    fontweight="bold",
    pad=10
)


# 5G NR baseline
disp3 = ConfusionMatrixDisplay(
    confusion_matrix=nr_baseline_cm,
    display_labels=CLASS_ORDER
)

disp3.plot(
    ax=axes[1, 0],
    cmap="Greens",
    values_format="d",
    colorbar=False,
    im_kw={
        "vmin": 0,
        "vmax": nr_vmax
    }
)

axes[1, 0].set_title(
    (
        "(c) 5G NR Baseline: XGB\n"
        f"Macro-F1 = "
        f"{nr_baseline_result['Macro-F1']:.3f}, "
        f"Poor F1 = "
        f"{nr_baseline_result['Poor F1']:.3f}"
    ),
    fontsize=14,
    fontweight="bold",
    pad=10
)


# 5G NR class weighting
disp4 = ConfusionMatrixDisplay(
    confusion_matrix=nr_treatment_cm,
    display_labels=CLASS_ORDER
)

disp4.plot(
    ax=axes[1, 1],
    cmap="Greens",
    values_format="d",
    colorbar=False,
    im_kw={
        "vmin": 0,
        "vmax": nr_vmax
    }
)

axes[1, 1].set_title(
    (
        "(d) 5G NR Class Weight: LGBM\n"
        f"Macro-F1 = "
        f"{nr_treatment_result['Macro-F1']:.3f}, "
        f"Poor F1 = "
        f"{nr_treatment_result['Poor F1']:.3f}"
    ),
    fontsize=14,
    fontweight="bold",
    pad=10
)


# ------------------------------------------------------------
# 8. Format axes and matrix values
# ------------------------------------------------------------

for ax in axes.flat:

    ax.set_xlabel(
        "Predicted class",
        fontsize=12
    )

    ax.set_ylabel(
        "True class",
        fontsize=12
    )

    ax.tick_params(
        axis="both",
        labelsize=11
    )

    ax.set_xticklabels(
        CLASS_ORDER,
        rotation=0
    )

    ax.set_yticklabels(
        CLASS_ORDER,
        rotation=0
    )


for display_object in [
    disp1,
    disp2,
    disp3,
    disp4
]:

    for text in display_object.text_.ravel():
        text.set_fontsize(14)


# ------------------------------------------------------------
# 9. Main title
# ------------------------------------------------------------

fig.suptitle(
    (
        "Three-Class Classification: "
        "Untreated Baseline versus Class Weighting"
    ),
    fontsize=17,
    fontweight="bold",
    y=1.01
)

plt.tight_layout()
plt.show()

## Phase 3A: Binary Poor-versus-Non-Poor Classification

The three-class target is reformulated as a binary task:

- Poor: 1
- Non-Poor: 0

The binary experiments examine whether combining Excellent and Good observations allows the models to focus more directly on Poor-signal detection.

Performance includes Poor-class precision, recall and F1, together with ROC-AUC and PR-AUC.

### ※ Related dissertation sections

- Section 4.5 (Binary Poor-versus-Non-Poor Classification)
- Table 4.7 (Three-Class and Binary Poor-Signal Detection Performance)
- Figure 4.11 (Binary Poor-versus-Non-Poor Classification Design)
- Section 5.3 (Binary Classification and Decision-Threshold Adjustment)

In [ ]:
# ============================================================
# Phase 3A - Cell 1. Binary Classification Metrics
# Poor versus Non-Poor
# ============================================================

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    matthews_corrcoef,
    roc_auc_score,
    average_precision_score
)

print("Phase 3A imports completed.")

In [ ]:
# ============================================================
# Phase 3A - Cell 2. Create Binary Targets
# Poor = 1, Non-Poor = 0
# ============================================================

def create_binary_target(
    labelled_df,
    target_col,
    expected_poor_count=None
):

    # Match the row order used by prepare_xy()
    modelling_df = (
        labelled_df
        .sort_values("grid_id")
        .reset_index(drop=True)
        .copy()
    )

    target_text = (
        modelling_df[target_col]
        .astype("string")
        .str.strip()
    )

    detected_classes = set(
        target_text.unique().tolist()
    )

    if detected_classes != set(VALID_CLASSES):
        raise ValueError(
            f"Unexpected classes in {target_col}: "
            f"{detected_classes}"
        )

    y_binary = (
        target_text
        .eq("Poor")
        .astype(int)
        .to_numpy()
    )

    poor_count = int(
        np.sum(y_binary == 1)
    )

    nonpoor_count = int(
        np.sum(y_binary == 0)
    )

    if expected_poor_count is not None:

        assert poor_count == expected_poor_count, (
            f"{target_col}: expected "
            f"{expected_poor_count} Poor observations, "
            f"but found {poor_count}."
        )

    if len(np.unique(y_binary)) != 2:
        raise ValueError(
            f"{target_col}: binary target must contain "
            "both classes 0 and 1."
        )

    print("=" * 70)
    print(target_col)
    print("=" * 70)

    print(
        "Total observations:",
        f"{len(y_binary):,}"
    )

    print(
        "Non-Poor (0):",
        f"{nonpoor_count:,}"
    )

    print(
        "Poor (1)    :",
        f"{poor_count:,}"
    )

    print(
        "Poor proportion:",
        f"{np.mean(y_binary):.4f} "
        f"({np.mean(y_binary) * 100:.2f}%)"
    )

    return y_binary


y_lte_binary = create_binary_target(
    labelled_df=lte_df,
    target_col="lte_signal_class",
    expected_poor_count=121
)

y_nr_binary = create_binary_target(
    labelled_df=nr_df,
    target_col="nr_signal_class",
    expected_poor_count=806
)


# Confirm alignment with predictor matrices
assert len(X_lte) == len(y_lte_binary)
assert len(X_nr) == len(y_nr_binary)

print("\n✓ Binary targets align with predictor matrices.")

In [ ]:
# ============================================================
# Phase 3A - Cell 3.
# Binary Poor versus Non-Poor Five-Fold OOF Evaluation
# ============================================================

def run_binary_imbalance_oof(
    X,
    y,
    network_name
):

    # --------------------------------------------------------
    # 1. Validate binary target
    # --------------------------------------------------------

    if set(np.unique(y)) != {0, 1}:
        raise ValueError(
            "Binary target must contain exactly 0 and 1."
        )


    # --------------------------------------------------------
    # 2. Create identical CV splits for every experiment
    # --------------------------------------------------------

    cv = StratifiedKFold(
        n_splits=N_SPLITS,
        shuffle=True,
        random_state=RANDOM_STATE
    )

    cv_splits = list(
        cv.split(X, y)
    )

    model_templates = create_baseline_models()

    all_results = []
    all_predictions = {}
    all_probabilities = {}
    all_fold_results = {}


    # --------------------------------------------------------
    # 3. Evaluate every method and model
    # --------------------------------------------------------

    for method in IMBALANCE_METHODS:

        for model_name, model_template in (
            model_templates.items()
        ):

            print("\n" + "=" * 75)

            print(
                f"{network_name} | "
                f"{method} | "
                f"{model_name}"
            )

            print("=" * 75)

            oof_pred = np.full(
                len(y),
                fill_value=-1,
                dtype=int
            )

            oof_prob = np.full(
                len(y),
                fill_value=np.nan,
                dtype=float
            )

            validation_count = np.zeros(
                len(y),
                dtype=int
            )

            fold_results = []


            # ------------------------------------------------
            # 4. Five-fold cross-validation
            # ------------------------------------------------

            for fold, (train_idx, val_idx) in enumerate(
                cv_splits,
                start=1
            ):

                X_train = X.iloc[
                    train_idx
                ].copy()

                X_val = X.iloc[
                    val_idx
                ].copy()

                y_train = y[
                    train_idx
                ]

                y_val = y[
                    val_idx
                ]


                # --------------------------------------------
                # Default untreated data
                # --------------------------------------------

                X_fit = X_train.copy()
                X_predict = X_val.copy()
                y_fit = y_train.copy()

                sample_weight = None


                # --------------------------------------------
                # Baseline
                # --------------------------------------------

                if method == "Baseline":

                    pass


                # --------------------------------------------
                # Class Weight
                # --------------------------------------------

                elif method == "Class Weight":

                    weight_dict = (
                        calculate_class_weight_dict(
                            y_train
                        )
                    )

                    sample_weight = (
                        create_sample_weights(
                            y_train,
                            weight_dict
                        )
                    )


                # --------------------------------------------
                # SMOTE-based methods
                # --------------------------------------------

                elif method in [
                    "SMOTE",
                    "SMOTE-Tomek",
                    "SMOTE + Class Weight"
                ]:

                    # Scaling fitted on training fold only
                    scaler = StandardScaler()

                    X_train_scaled = pd.DataFrame(
                        scaler.fit_transform(
                            X_train
                        ),
                        columns=X.columns,
                        index=X_train.index
                    )

                    X_predict = pd.DataFrame(
                        scaler.transform(
                            X_val
                        ),
                        columns=X.columns,
                        index=X_val.index
                    )


                    if method in [
                        "SMOTE",
                        "SMOTE + Class Weight"
                    ]:

                        sampler = SMOTE(
                            random_state=RANDOM_STATE,
                            k_neighbors=5
                        )

                    else:

                        sampler = SMOTETomek(
                            random_state=RANDOM_STATE,
                            smote=SMOTE(
                                random_state=RANDOM_STATE,
                                k_neighbors=5
                            )
                        )


                    # Resample training fold only
                    X_fit, y_fit = (
                        sampler.fit_resample(
                            X_train_scaled,
                            y_train
                        )
                    )


                    # Original-fold weights mapped to
                    # resampled observations
                    if method == "SMOTE + Class Weight":

                        weight_dict = (
                            calculate_class_weight_dict(
                                y_train
                            )
                        )

                        sample_weight = (
                            create_sample_weights(
                                y_fit,
                                weight_dict
                            )
                        )


                else:

                    raise ValueError(
                        f"Unknown imbalance method: {method}"
                    )


                # --------------------------------------------
                # 5. Train a fresh model
                # --------------------------------------------

                model = clone(
                    model_template
                )

                if sample_weight is None:

                    model.fit(
                        X_fit,
                        y_fit
                    )

                else:

                    model.fit(
                        X_fit,
                        y_fit,
                        sample_weight=sample_weight
                    )


                # --------------------------------------------
                # 6. Predict Poor-class probability
                # --------------------------------------------

                positive_class_index = int(
                    np.where(
                        model.classes_ == 1
                    )[0][0]
                )

                y_prob = model.predict_proba(
                    X_predict
                )[:, positive_class_index]

                # Default decision threshold
                y_pred = (
                    y_prob >= 0.50
                ).astype(int)

                oof_pred[
                    val_idx
                ] = y_pred

                oof_prob[
                    val_idx
                ] = y_prob

                validation_count[
                    val_idx
                ] += 1


                # --------------------------------------------
                # 7. Fold-level metrics
                # --------------------------------------------

                fold_result = {
                    "Fold": fold,

                    "Accuracy": accuracy_score(
                        y_val,
                        y_pred
                    ),

                    "Balanced Accuracy":
                        balanced_accuracy_score(
                            y_val,
                            y_pred
                        ),

                    "MCC": matthews_corrcoef(
                        y_val,
                        y_pred
                    ),

                    "Poor Precision": precision_score(
                        y_val,
                        y_pred,
                        pos_label=1,
                        zero_division=0
                    ),

                    "Poor Recall": recall_score(
                        y_val,
                        y_pred,
                        pos_label=1,
                        zero_division=0
                    ),

                    "Poor F1": f1_score(
                        y_val,
                        y_pred,
                        pos_label=1,
                        zero_division=0
                    ),

                    "ROC-AUC": roc_auc_score(
                        y_val,
                        y_prob
                    ),

                    "PR-AUC": average_precision_score(
                        y_val,
                        y_prob
                    ),

                    "Poor Support": int(
                        np.sum(
                            y_val == 1
                        )
                    ),

                    "Training Rows Before":
                        len(y_train),

                    "Training Rows After":
                        len(y_fit)
                }

                fold_results.append(
                    fold_result
                )

                print(
                    f"Fold {fold}: "
                    f"Poor F1 = "
                    f"{fold_result['Poor F1']:.4f}, "
                    f"PR-AUC = "
                    f"{fold_result['PR-AUC']:.4f}"
                )


            # ------------------------------------------------
            # 8. Validate OOF assignments
            # ------------------------------------------------

            if np.any(oof_pred == -1):
                raise RuntimeError(
                    f"{network_name} | {method} | "
                    f"{model_name}: missing OOF predictions."
                )

            if np.isnan(oof_prob).any():
                raise RuntimeError(
                    f"{network_name} | {method} | "
                    f"{model_name}: missing OOF probabilities."
                )

            if not np.all(
                validation_count == 1
            ):
                raise RuntimeError(
                    f"{network_name} | {method} | "
                    f"{model_name}: invalid OOF assignment."
                )

            if np.any(
                (oof_prob < 0)
                | (oof_prob > 1)
            ):
                raise RuntimeError(
                    "Predicted probabilities outside [0, 1]."
                )


            # ------------------------------------------------
            # 9. Overall OOF metrics
            # ------------------------------------------------

            fold_results_df = pd.DataFrame(
                fold_results
            )

            result = {
                "Network": network_name,
                "Method": method,
                "Model": model_name,
                "Threshold": 0.50,

                "N Rows": len(y),
                "N Features": X.shape[1],
                "Poor Support": int(
                    np.sum(y == 1)
                ),
                "Poor Prevalence": float(
                    np.mean(y)
                ),

                "Accuracy": accuracy_score(
                    y,
                    oof_pred
                ),

                "Balanced Accuracy":
                    balanced_accuracy_score(
                        y,
                        oof_pred
                    ),

                "MCC": matthews_corrcoef(
                    y,
                    oof_pred
                ),

                "Poor Precision": precision_score(
                    y,
                    oof_pred,
                    pos_label=1,
                    zero_division=0
                ),

                "Poor Recall": recall_score(
                    y,
                    oof_pred,
                    pos_label=1,
                    zero_division=0
                ),

                "Poor F1": f1_score(
                    y,
                    oof_pred,
                    pos_label=1,
                    zero_division=0
                ),

                "ROC-AUC": roc_auc_score(
                    y,
                    oof_prob
                ),

                "PR-AUC": average_precision_score(
                    y,
                    oof_prob
                ),

                "CV Poor F1 Mean":
                    fold_results_df[
                        "Poor F1"
                    ].mean(),

                "CV Poor F1 SD":
                    fold_results_df[
                        "Poor F1"
                    ].std(ddof=0),

                "CV ROC-AUC Mean":
                    fold_results_df[
                        "ROC-AUC"
                    ].mean(),

                "CV ROC-AUC SD":
                    fold_results_df[
                        "ROC-AUC"
                    ].std(ddof=0),

                "CV PR-AUC Mean":
                    fold_results_df[
                        "PR-AUC"
                    ].mean(),

                "CV PR-AUC SD":
                    fold_results_df[
                        "PR-AUC"
                    ].std(ddof=0)
            }

            all_results.append(
                result
            )

            result_key = (
                method,
                model_name
            )

            all_predictions[
                result_key
            ] = oof_pred.copy()

            all_probabilities[
                result_key
            ] = oof_prob.copy()

            all_fold_results[
                result_key
            ] = fold_results_df


            # ------------------------------------------------
            # 10. Display experiment summary
            # ------------------------------------------------

            print("\nOOF Summary")

            print(
                f"Poor Precision : "
                f"{result['Poor Precision']:.4f}"
            )

            print(
                f"Poor Recall    : "
                f"{result['Poor Recall']:.4f}"
            )

            print(
                f"Poor F1        : "
                f"{result['Poor F1']:.4f}"
            )

            print(
                f"Balanced Acc.  : "
                f"{result['Balanced Accuracy']:.4f}"
            )

            print(
                f"MCC            : "
                f"{result['MCC']:.4f}"
            )

            print(
                f"ROC-AUC        : "
                f"{result['ROC-AUC']:.4f}"
            )

            print(
                f"PR-AUC         : "
                f"{result['PR-AUC']:.4f}"
            )


    return (
        pd.DataFrame(
            all_results
        ),
        all_predictions,
        all_probabilities,
        all_fold_results
    )

In [ ]:
# ============================================================
# Phase 3A - Cell 4. LTE Binary Experiments
# ============================================================

(
    lte_binary_results,
    lte_binary_predictions,
    lte_binary_probabilities,
    lte_binary_folds
) = run_binary_imbalance_oof(
    X=X_lte,
    y=y_lte_binary,
    network_name="LTE"
)

binary_display_cols = [
    "Network",
    "Method",
    "Model",
    "Threshold",
    "Accuracy",
    "Balanced Accuracy",
    "MCC",
    "Poor Precision",
    "Poor Recall",
    "Poor F1",
    "CV Poor F1 Mean",
    "CV Poor F1 SD",
    "ROC-AUC",
    "PR-AUC",
    "CV PR-AUC Mean",
    "CV PR-AUC SD",
    "Poor Support",
    "Poor Prevalence"
]

assert len(lte_binary_results) == 15, (
    f"Expected 15 LTE binary results, "
    f"but found {len(lte_binary_results)}."
)

print("\nLTE Binary Poor-versus-Non-Poor Results")

display(
    lte_binary_results[
        binary_display_cols
    ].round(4)
)# ============================================================
# Phase 3A - Cell 4. LTE Binary Experiments
# ============================================================

(
    lte_binary_results,
    lte_binary_predictions,
    lte_binary_probabilities,
    lte_binary_folds
) = run_binary_imbalance_oof(
    X=X_lte,
    y=y_lte_binary,
    network_name="LTE"
)

binary_display_cols = [
    "Network",
    "Method",
    "Model",
    "Threshold",
    "Accuracy",
    "Balanced Accuracy",
    "MCC",
    "Poor Precision",
    "Poor Recall",
    "Poor F1",
    "CV Poor F1 Mean",
    "CV Poor F1 SD",
    "ROC-AUC",
    "PR-AUC",
    "CV PR-AUC Mean",
    "CV PR-AUC SD",
    "Poor Support",
    "Poor Prevalence"
]

assert len(lte_binary_results) == 15, (
    f"Expected 15 LTE binary results, "
    f"but found {len(lte_binary_results)}."
)

print("\nLTE Binary Poor-versus-Non-Poor Results")

display(
    lte_binary_results[
        binary_display_cols
    ].round(4)
)

In [ ]:
# ============================================================
# Phase 3A - Cell 5. 5G NR Binary Experiments
# ============================================================

(
    nr_binary_results,
    nr_binary_predictions,
    nr_binary_probabilities,
    nr_binary_folds
) = run_binary_imbalance_oof(
    X=X_nr,
    y=y_nr_binary,
    network_name="5G NR"
)

assert len(nr_binary_results) == 15, (
    f"Expected 15 5G NR binary results, "
    f"but found {len(nr_binary_results)}."
)

print("\n5G NR Binary Poor-versus-Non-Poor Results")

display(
    nr_binary_results[
        binary_display_cols
    ].round(4)
)

In [ ]:
# ============================================================
# Phase 3A - Cell 6. Combined Binary Results
# ============================================================

binary_results = pd.concat(
    [
        lte_binary_results,
        nr_binary_results
    ],
    ignore_index=True
)

# ------------------------------------------------------------
# Fix result order
# ------------------------------------------------------------

network_order = [
    "LTE",
    "5G NR"
]

method_order = [
    "Baseline",
    "Class Weight",
    "SMOTE",
    "SMOTE-Tomek",
    "SMOTE + Class Weight"
]

model_order = [
    "RF",
    "XGB",
    "LGBM"
]

binary_results["Network"] = pd.Categorical(
    binary_results["Network"],
    categories=network_order,
    ordered=True
)

binary_results["Method"] = pd.Categorical(
    binary_results["Method"],
    categories=method_order,
    ordered=True
)

binary_results["Model"] = pd.Categorical(
    binary_results["Model"],
    categories=model_order,
    ordered=True
)

binary_results = (
    binary_results
    .sort_values(
        [
            "Network",
            "Method",
            "Model"
        ]
    )
    .reset_index(drop=True)
)

assert len(binary_results) == 30, (
    f"Expected 30 binary results, "
    f"but found {len(binary_results)}."
)

binary_result_cols = [
    "Network",
    "Method",
    "Model",
    "Threshold",
    "Accuracy",
    "Balanced Accuracy",
    "MCC",
    "Poor Precision",
    "Poor Recall",
    "Poor F1",
    "CV Poor F1 Mean",
    "CV Poor F1 SD",
    "ROC-AUC",
    "PR-AUC",
    "CV PR-AUC Mean",
    "CV PR-AUC SD",
    "Poor Support",
    "Poor Prevalence"
]

print("Combined Binary Poor-versus-Non-Poor Results")
print("Total experiment results:", len(binary_results))

display(
    binary_results[
        binary_result_cols
    ].round(4)
)

In [ ]:
# ============================================================
# Phase 3A - Cell 7.
# Best Binary Configuration by Mean CV Poor-Class F1
# ============================================================

best_binary_indices = (
    binary_results
    .groupby(
        "Network",
        observed=True
    )["CV Poor F1 Mean"]
    .idxmax()
)

best_binary_poor = (
    binary_results
    .loc[best_binary_indices]
    .sort_values("Network")
    .reset_index(drop=True)
)

best_binary_cols = [
    "Network",
    "Method",
    "Model",
    "Threshold",
    "Accuracy",
    "Balanced Accuracy",
    "MCC",
    "Poor Precision",
    "Poor Recall",
    "Poor F1",
    "CV Poor F1 Mean",
    "CV Poor F1 SD",
    "ROC-AUC",
    "PR-AUC",
    "CV PR-AUC Mean",
    "CV PR-AUC SD",
    "Poor Support",
    "Poor Prevalence"
]

print(
    "Best Binary Configuration "
    "by Mean CV Poor-Class F1"
)

display(
    best_binary_poor[
        best_binary_cols
    ].round(4)
)

In [ ]:
# ============================================================
# Phase 3A - Cell 8.
# Three-Class versus Binary Poor Detection
# Binary threshold = 0.50
# ============================================================

comparison_rows = []

for network in [
    "LTE",
    "5G NR"
]:

    # --------------------------------------------------------
    # 1. Best three-class configuration by OOF Poor F1
    # --------------------------------------------------------

    three_class_candidates = imbalance_results[
        imbalance_results["Network"] == network
    ]

    three_class_index = (
        three_class_candidates[
            "Poor F1"
        ].idxmax()
    )

    three_class = imbalance_results.loc[
        three_class_index
    ]


    # --------------------------------------------------------
    # 2. Best binary configuration by OOF Poor F1
    # Threshold = 0.50
    # --------------------------------------------------------

    binary_candidates = binary_results[
        binary_results["Network"] == network
    ]

    binary_index = (
        binary_candidates[
            "Poor F1"
        ].idxmax()
    )

    binary = binary_results.loc[
        binary_index
    ]


    # --------------------------------------------------------
    # 3. Compare Poor-class performance
    # --------------------------------------------------------

    comparison_rows.append({
        "Network": network,

        "Three-Class Method":
            three_class["Method"],

        "Three-Class Model":
            three_class["Model"],

        "Three-Class Poor Precision":
            three_class["Poor Precision"],

        "Three-Class Poor Recall":
            three_class["Poor Recall"],

        "Three-Class Poor F1":
            three_class["Poor F1"],

        "Binary Method":
            binary["Method"],

        "Binary Model":
            binary["Model"],

        "Binary Threshold":
            binary["Threshold"],

        "Binary Poor Precision":
            binary["Poor Precision"],

        "Binary Poor Recall":
            binary["Poor Recall"],

        "Binary Poor F1":
            binary["Poor F1"],

        "Precision Difference":
            binary["Poor Precision"]
            - three_class["Poor Precision"],

        "Recall Difference":
            binary["Poor Recall"]
            - three_class["Poor Recall"],

        "Poor F1 Difference":
            binary["Poor F1"]
            - three_class["Poor F1"]
    })


threeclass_vs_binary = pd.DataFrame(
    comparison_rows
)

print(
    "Best Three-Class versus Best Binary "
    "Poor Detection at Threshold 0.50"
)

display(
    threeclass_vs_binary.round(4)
)

## Phase 3B: Exploratory Decision-Threshold Analysis

Poor-class probability thresholds are varied to examine the trade-off between precision and recall.

The selected thresholds are identified using pooled out-of-fold predictions and Poor-class F1. Therefore, this analysis is exploratory and should not be interpreted as an unbiased estimate of future threshold performance.

### ※ Related dissertation sections

- Section 3.1 (Research Approach—Modelling)
- Section 4.6 (Exploratory Decision-Threshold Analysis)
- Table 4.8 (Poor-Class Performance at Default and Selected Thresholds)
- Figure 4.12 (Poor-Class Performance across Alternative Thresholds)
- Section 5.3 (Binary Classification and Decision-Threshold Adjustment)

In [ ]:
# ============================================================
# Phase 3B - Cell 1.
# Select Binary OOF Probabilities by Mean CV PR-AUC
# ============================================================

def select_probability_configuration(
    results,
    probability_dict,
    network_name
):

    # Select using a threshold-independent metric
    best_index = results[
        "CV PR-AUC Mean"
    ].idxmax()

    best_result = results.loc[
        best_index
    ]

    best_key = (
        str(best_result["Method"]),
        str(best_result["Model"])
    )

    if best_key not in probability_dict:
        raise KeyError(
            f"{network_name}: probability key "
            f"not found: {best_key}"
        )

    best_probabilities = np.asarray(
        probability_dict[
            best_key
        ],
        dtype=float
    )

    if np.isnan(
        best_probabilities
    ).any():
        raise ValueError(
            f"{network_name}: missing OOF probabilities."
        )

    if np.any(
        (best_probabilities < 0)
        | (best_probabilities > 1)
    ):
        raise ValueError(
            f"{network_name}: probabilities "
            "outside [0, 1]."
        )

    return (
        best_result,
        best_key,
        best_probabilities
    )


# ------------------------------------------------------------
# LTE
# ------------------------------------------------------------

(
    lte_probability_result,
    lte_binary_best_key,
    lte_best_prob
) = select_probability_configuration(
    results=lte_binary_results,
    probability_dict=lte_binary_probabilities,
    network_name="LTE"
)


# ------------------------------------------------------------
# 5G NR
# ------------------------------------------------------------

(
    nr_probability_result,
    nr_binary_best_key,
    nr_best_prob
) = select_probability_configuration(
    results=nr_binary_results,
    probability_dict=nr_binary_probabilities,
    network_name="5G NR"
)


# ------------------------------------------------------------
# Validation summary
# ------------------------------------------------------------

assert len(lte_best_prob) == len(
    y_lte_binary
)

assert len(nr_best_prob) == len(
    y_nr_binary
)

selected_probability_models = pd.DataFrame([
    {
        "Network": "LTE",
        "Method": lte_binary_best_key[0],
        "Model": lte_binary_best_key[1],
        "CV PR-AUC Mean":
            lte_probability_result[
                "CV PR-AUC Mean"
            ],
        "CV PR-AUC SD":
            lte_probability_result[
                "CV PR-AUC SD"
            ],
        "OOF PR-AUC":
            lte_probability_result[
                "PR-AUC"
            ]
    },
    {
        "Network": "5G NR",
        "Method": nr_binary_best_key[0],
        "Model": nr_binary_best_key[1],
        "CV PR-AUC Mean":
            nr_probability_result[
                "CV PR-AUC Mean"
            ],
        "CV PR-AUC SD":
            nr_probability_result[
                "CV PR-AUC SD"
            ],
        "OOF PR-AUC":
            nr_probability_result[
                "PR-AUC"
            ]
    }
])

print(
    "Binary probability models selected "
    "by mean CV PR-AUC:"
)

display(
    selected_probability_models.round(4)
)

print(
    "LTE probabilities:",
    len(lte_best_prob)
)

print(
    "5G NR probabilities:",
    len(nr_best_prob)
)

In [ ]:
# ============================================================
# Phase 3B - Cell 2.
# Exploratory Pooled-OOF Threshold Sweep Function
# ============================================================

def evaluate_oof_thresholds(
    y_true,
    y_prob,
    network_name,
    method_name,
    model_name,
    start=0.05,
    stop=0.95,
    step=0.01
):

    # --------------------------------------------------------
    # 1. Validate inputs
    # --------------------------------------------------------

    y_true = np.asarray(
        y_true,
        dtype=int
    )

    y_prob = np.asarray(
        y_prob,
        dtype=float
    )

    if len(y_true) != len(y_prob):
        raise ValueError(
            "y_true and y_prob must have the same length."
        )

    if set(np.unique(y_true)) != {0, 1}:
        raise ValueError(
            "y_true must contain binary labels 0 and 1."
        )

    if np.isnan(y_prob).any():
        raise ValueError(
            "Missing predicted probabilities detected."
        )

    if np.any(
        (y_prob < 0)
        | (y_prob > 1)
    ):
        raise ValueError(
            "Predicted probabilities must be within [0, 1]."
        )

    if not (
        0 <= start < stop <= 1
    ):
        raise ValueError(
            "Threshold range must satisfy "
            "0 <= start < stop <= 1."
        )

    if step <= 0:
        raise ValueError(
            "step must be greater than zero."
        )


    # --------------------------------------------------------
    # 2. Create threshold sequence
    # --------------------------------------------------------

    number_of_steps = int(
        round(
            (stop - start) / step
        )
    )

    thresholds = np.linspace(
        start,
        stop,
        number_of_steps + 1
    )


    # --------------------------------------------------------
    # 3. Evaluate every threshold
    # --------------------------------------------------------

    results = []

    for threshold in thresholds:

        y_pred = (
            y_prob >= threshold
        ).astype(int)

        tn, fp, fn, tp = confusion_matrix(
            y_true,
            y_pred,
            labels=[0, 1]
        ).ravel()

        specificity = (
            tn / (tn + fp)
            if (tn + fp) > 0
            else 0.0
        )

        predicted_poor_count = int(
            np.sum(y_pred == 1)
        )

        results.append({
            "Analysis": "Exploratory pooled OOF",
            "Network": network_name,
            "Method": method_name,
            "Model": model_name,
            "Threshold": float(threshold),

            "Accuracy": accuracy_score(
                y_true,
                y_pred
            ),

            "Balanced Accuracy":
                balanced_accuracy_score(
                    y_true,
                    y_pred
                ),

            "MCC": matthews_corrcoef(
                y_true,
                y_pred
            ),

            "Poor Precision": precision_score(
                y_true,
                y_pred,
                pos_label=1,
                zero_division=0
            ),

            "Poor Recall": recall_score(
                y_true,
                y_pred,
                pos_label=1,
                zero_division=0
            ),

            "Poor F1": f1_score(
                y_true,
                y_pred,
                pos_label=1,
                zero_division=0
            ),

            "Specificity": specificity,

            "True Poor Count": int(
                np.sum(y_true == 1)
            ),

            "Predicted Poor Count":
                predicted_poor_count,

            "Predicted Poor Rate":
                predicted_poor_count
                / len(y_true),

            "TN": int(tn),
            "FP": int(fp),
            "FN": int(fn),
            "TP": int(tp)
        })

    threshold_results = pd.DataFrame(
        results
    )

    if len(threshold_results) != (
        number_of_steps + 1
    ):
        raise RuntimeError(
            "Unexpected number of threshold results."
        )

    return threshold_results

In [ ]:
# ============================================================
# Phase 3B - Cell 3.
# Exploratory LTE Pooled-OOF Threshold Sweep
# ============================================================

lte_threshold_results = evaluate_oof_thresholds(
    y_true=y_lte_binary,
    y_prob=lte_best_prob,
    network_name="LTE",
    method_name=lte_binary_best_key[0],
    model_name=lte_binary_best_key[1],
    start=0.05,
    stop=0.95,
    step=0.01
)

assert len(lte_threshold_results) == 91, (
    f"Expected 91 threshold results, "
    f"but found {len(lte_threshold_results)}."
)

print(
    "LTE threshold sweep configuration:",
    lte_binary_best_key
)

print(
    "Number of thresholds:",
    len(lte_threshold_results)
)

# Show the first and last five thresholds
display(
    pd.concat(
        [
            lte_threshold_results.head(5),
            lte_threshold_results.tail(5)
        ]
    )
    .round(4)
)

In [ ]:
# ============================================================
# Phase 3B - Cell 4.
# Exploratory 5G NR Pooled-OOF Threshold Sweep
# ============================================================

nr_threshold_results = evaluate_oof_thresholds(
    y_true=y_nr_binary,
    y_prob=nr_best_prob,
    network_name="5G NR",
    method_name=nr_binary_best_key[0],
    model_name=nr_binary_best_key[1],
    start=0.05,
    stop=0.95,
    step=0.01
)

assert len(nr_threshold_results) == 91, (
    f"Expected 91 threshold results, "
    f"but found {len(nr_threshold_results)}."
)

print(
    "5G NR threshold sweep configuration:",
    nr_binary_best_key
)

print(
    "Number of thresholds:",
    len(nr_threshold_results)
)

display(
    pd.concat(
        [
            nr_threshold_results.head(5),
            nr_threshold_results.tail(5)
        ]
    )
    .round(4)
)

In [ ]:
# ============================================================
# Phase 3B - Cell 5.
# Exploratory Best Pooled-OOF Threshold by Poor F1
# ============================================================

def select_exploratory_threshold(
    threshold_results
):

    selection_table = (
        threshold_results
        .copy()
    )

    selection_table[
        "Distance from 0.50"
    ] = (
        selection_table["Threshold"]
        - 0.50
    ).abs()

    best_row = (
        selection_table
        .sort_values(
            [
                "Poor F1",
                "MCC",
                "Balanced Accuracy",
                "Distance from 0.50"
            ],
            ascending=[
                False,
                False,
                False,
                True
            ]
        )
        .iloc[0]
        .drop(
            labels=["Distance from 0.50"]
        )
    )

    return best_row


lte_best_threshold = (
    select_exploratory_threshold(
        lte_threshold_results
    )
)

nr_best_threshold = (
    select_exploratory_threshold(
        nr_threshold_results
    )
)

best_threshold_summary = pd.DataFrame([
    lte_best_threshold,
    nr_best_threshold
]).reset_index(drop=True)

best_threshold_cols = [
    "Analysis",
    "Network",
    "Method",
    "Model",
    "Threshold",
    "Accuracy",
    "Balanced Accuracy",
    "MCC",
    "Poor Precision",
    "Poor Recall",
    "Poor F1",
    "Specificity",
    "True Poor Count",
    "Predicted Poor Count",
    "Predicted Poor Rate",
    "TN",
    "FP",
    "FN",
    "TP"
]

print(
    "Exploratory Best Pooled-OOF Threshold "
    "by Poor-Class F1"
)

display(
    best_threshold_summary[
        best_threshold_cols
    ].round(4)
)

In [ ]:
# ============================================================
# Phase 3B - Cell 6.
# Default versus Exploratory Selected Pooled-OOF Threshold
# ============================================================

comparison_rows = []

for (
    network,
    threshold_df,
    selected_row
) in [
    (
        "LTE",
        lte_threshold_results,
        lte_best_threshold
    ),
    (
        "5G NR",
        nr_threshold_results,
        nr_best_threshold
    )
]:

    default_matches = threshold_df[
        np.isclose(
            threshold_df["Threshold"],
            0.50
        )
    ]

    if len(default_matches) != 1:
        raise ValueError(
            f"{network}: expected one result at "
            f"threshold 0.50, but found "
            f"{len(default_matches)}."
        )

    default_row = (
        default_matches.iloc[0]
    )

    comparison_rows.append({
        "Network": network,

        "Default Threshold":
            default_row["Threshold"],

        "Default Accuracy":
            default_row["Accuracy"],

        "Default Balanced Accuracy":
            default_row["Balanced Accuracy"],

        "Default MCC":
            default_row["MCC"],

        "Default Poor Precision":
            default_row["Poor Precision"],

        "Default Poor Recall":
            default_row["Poor Recall"],

        "Default Poor F1":
            default_row["Poor F1"],

        "Default Specificity":
            default_row["Specificity"],

        "Selected Threshold":
            selected_row["Threshold"],

        "Selected Accuracy":
            selected_row["Accuracy"],

        "Selected Balanced Accuracy":
            selected_row["Balanced Accuracy"],

        "Selected MCC":
            selected_row["MCC"],

        "Selected Poor Precision":
            selected_row["Poor Precision"],

        "Selected Poor Recall":
            selected_row["Poor Recall"],

        "Selected Poor F1":
            selected_row["Poor F1"],

        "Selected Specificity":
            selected_row["Specificity"],

        "Accuracy Change":
            selected_row["Accuracy"]
            - default_row["Accuracy"],

        "Balanced Accuracy Change":
            selected_row["Balanced Accuracy"]
            - default_row["Balanced Accuracy"],

        "MCC Change":
            selected_row["MCC"]
            - default_row["MCC"],

        "Poor Precision Change":
            selected_row["Poor Precision"]
            - default_row["Poor Precision"],

        "Poor Recall Change":
            selected_row["Poor Recall"]
            - default_row["Poor Recall"],

        "Poor F1 Change":
            selected_row["Poor F1"]
            - default_row["Poor F1"],

        "Specificity Change":
            selected_row["Specificity"]
            - default_row["Specificity"]
    })


threshold_comparison = pd.DataFrame(
    comparison_rows
)

print(
    "Default Threshold versus Exploratory "
    "Selected Pooled-OOF Threshold"
)

display(
    threshold_comparison.round(4)
)

In [ ]:
# ============================================================
# Phase 3B - Cell 7.
# Exploratory Pooled-OOF Threshold Performance Curves
# ============================================================

fig, axes = plt.subplots(
    1,
    2,
    figsize=(14, 5.5),
    sharey=True
)

plot_configs = [
    (
        axes[0],
        lte_threshold_results,
        lte_best_threshold,
        "(a) LTE"
    ),
    (
        axes[1],
        nr_threshold_results,
        nr_best_threshold,
        "(b) 5G NR"
    )
]


for ax, results, best_row, panel_title in plot_configs:

    selected_threshold = float(
        best_row["Threshold"]
    )

    selected_precision = float(
        best_row["Poor Precision"]
    )

    selected_recall = float(
        best_row["Poor Recall"]
    )

    selected_f1 = float(
        best_row["Poor F1"]
    )


    # Poor-class metrics
    ax.plot(
        results["Threshold"],
        results["Poor Precision"],
        label="Poor Precision",
        linewidth=2,
        color="tab:blue"
    )

    ax.plot(
        results["Threshold"],
        results["Poor Recall"],
        label="Poor Recall",
        linewidth=2,
        color="tab:orange"
    )

    ax.plot(
        results["Threshold"],
        results["Poor F1"],
        label="Poor F1",
        linewidth=2.7,
        color="tab:green"
    )


    # Default threshold
    ax.axvline(
        0.50,
        color="grey",
        linestyle=":",
        linewidth=2,
        label="Default threshold (0.50)"
    )


    # Exploratory selected threshold
    ax.axvline(
        selected_threshold,
        color="black",
        linestyle="--",
        linewidth=2,
        label=(
            f"Selected threshold "
            f"({selected_threshold:.2f})"
        )
    )


    # Mark selected metric values
    ax.scatter(
        selected_threshold,
        selected_precision,
        color="tab:blue",
        s=55,
        zorder=5
    )

    ax.scatter(
        selected_threshold,
        selected_recall,
        color="tab:orange",
        s=55,
        zorder=5
    )

    ax.scatter(
        selected_threshold,
        selected_f1,
        color="tab:green",
        s=65,
        zorder=5
    )


    ax.set_title(
        (
            f"{panel_title}\n"
            f"Selected threshold = "
            f"{selected_threshold:.2f}, "
            f"Poor F1 = {selected_f1:.3f}"
        ),
        fontsize=13,
        fontweight="bold"
    )

    ax.set_xlabel(
        "Decision threshold",
        fontsize=12
    )

    ax.set_ylabel(
        "Score",
        fontsize=12
    )

    ax.set_xlim(
        0.05,
        0.95
    )

    ax.set_ylim(
        0,
        1
    )

    ax.tick_params(
        axis="both",
        labelsize=10
    )

    ax.grid(
        linestyle="--",
        alpha=0.3
    )

    ax.legend(
        fontsize=9,
        loc="best"
    )


fig.suptitle(
    (
        "Exploratory Poor-Class Performance "
        "across Pooled-OOF Decision Thresholds"
    ),
    fontsize=16,
    fontweight="bold",
    y=1.03
)

plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# Phase 3B - Cell 8.
# Default versus Exploratory Selected Threshold
# Binary Confusion Matrices
# ============================================================

# ------------------------------------------------------------
# 1. Thresholds
# ------------------------------------------------------------

DEFAULT_THRESHOLD = 0.50

lte_selected_threshold = float(
    lte_best_threshold["Threshold"]
)

nr_selected_threshold = float(
    nr_best_threshold["Threshold"]
)


# ------------------------------------------------------------
# 2. Predictions
# ------------------------------------------------------------

lte_default_pred = (
    lte_best_prob >= DEFAULT_THRESHOLD
).astype(int)

lte_selected_pred = (
    lte_best_prob >= lte_selected_threshold
).astype(int)

nr_default_pred = (
    nr_best_prob >= DEFAULT_THRESHOLD
).astype(int)

nr_selected_pred = (
    nr_best_prob >= nr_selected_threshold
).astype(int)


# ------------------------------------------------------------
# 3. Confusion matrices
# ------------------------------------------------------------

binary_labels = [
    0,
    1
]

display_labels = [
    "Non-Poor",
    "Poor"
]

lte_default_cm = confusion_matrix(
    y_lte_binary,
    lte_default_pred,
    labels=binary_labels
)

lte_selected_cm = confusion_matrix(
    y_lte_binary,
    lte_selected_pred,
    labels=binary_labels
)

nr_default_cm = confusion_matrix(
    y_nr_binary,
    nr_default_pred,
    labels=binary_labels
)

nr_selected_cm = confusion_matrix(
    y_nr_binary,
    nr_selected_pred,
    labels=binary_labels
)


# ------------------------------------------------------------
# 4. Metric helper for titles
# ------------------------------------------------------------

def calculate_binary_title_metrics(
    y_true,
    y_pred
):

    return {
        "Poor F1": f1_score(
            y_true,
            y_pred,
            pos_label=1,
            zero_division=0
        ),

        "Poor Recall": recall_score(
            y_true,
            y_pred,
            pos_label=1,
            zero_division=0
        )
    }


lte_default_metrics = (
    calculate_binary_title_metrics(
        y_lte_binary,
        lte_default_pred
    )
)

lte_selected_metrics = (
    calculate_binary_title_metrics(
        y_lte_binary,
        lte_selected_pred
    )
)

nr_default_metrics = (
    calculate_binary_title_metrics(
        y_nr_binary,
        nr_default_pred
    )
)

nr_selected_metrics = (
    calculate_binary_title_metrics(
        y_nr_binary,
        nr_selected_pred
    )
)


# ------------------------------------------------------------
# 5. Use the same colour scale within each network
# ------------------------------------------------------------

lte_vmax = max(
    lte_default_cm.max(),
    lte_selected_cm.max()
)

nr_vmax = max(
    nr_default_cm.max(),
    nr_selected_cm.max()
)


# ------------------------------------------------------------
# 6. Plot helper
# ------------------------------------------------------------

def plot_binary_cm(
    ax,
    cm,
    title,
    cmap,
    vmax
):

    display = ConfusionMatrixDisplay(
        confusion_matrix=cm,
        display_labels=display_labels
    )

    display.plot(
        ax=ax,
        cmap=cmap,
        values_format="d",
        colorbar=False,
        im_kw={
            "vmin": 0,
            "vmax": vmax
        }
    )

    ax.set_title(
        title,
        fontsize=13,
        fontweight="bold",
        pad=10
    )

    ax.set_xlabel(
        "Predicted class",
        fontsize=11
    )

    ax.set_ylabel(
        "True class",
        fontsize=11
    )

    ax.tick_params(
        axis="both",
        labelsize=10
    )

    for text in display.text_.ravel():
        text.set_fontsize(14)

    return display


# ------------------------------------------------------------
# 7. Create 2 × 2 comparison
# ------------------------------------------------------------

fig, axes = plt.subplots(
    2,
    2,
    figsize=(11, 10)
)

plot_binary_cm(
    ax=axes[0, 0],
    cm=lte_default_cm,
    title=(
        "(a) LTE Default Threshold = 0.50\n"
        f"Poor F1 = "
        f"{lte_default_metrics['Poor F1']:.3f}, "
        f"Recall = "
        f"{lte_default_metrics['Poor Recall']:.3f}"
    ),
    cmap="Blues",
    vmax=lte_vmax
)

plot_binary_cm(
    ax=axes[0, 1],
    cm=lte_selected_cm,
    title=(
        f"(b) LTE Selected Threshold = "
        f"{lte_selected_threshold:.2f}\n"
        f"Poor F1 = "
        f"{lte_selected_metrics['Poor F1']:.3f}, "
        f"Recall = "
        f"{lte_selected_metrics['Poor Recall']:.3f}"
    ),
    cmap="Blues",
    vmax=lte_vmax
)

plot_binary_cm(
    ax=axes[1, 0],
    cm=nr_default_cm,
    title=(
        "(c) 5G NR Default Threshold = 0.50\n"
        f"Poor F1 = "
        f"{nr_default_metrics['Poor F1']:.3f}, "
        f"Recall = "
        f"{nr_default_metrics['Poor Recall']:.3f}"
    ),
    cmap="Greens",
    vmax=nr_vmax
)

plot_binary_cm(
    ax=axes[1, 1],
    cm=nr_selected_cm,
    title=(
        f"(d) 5G NR Selected Threshold = "
        f"{nr_selected_threshold:.2f}\n"
        f"Poor F1 = "
        f"{nr_selected_metrics['Poor F1']:.3f}, "
        f"Recall = "
        f"{nr_selected_metrics['Poor Recall']:.3f}"
    ),
    cmap="Greens",
    vmax=nr_vmax
)


fig.suptitle(
    (
        "Binary Poor Detection: Default versus "
        "Exploratory Selected Pooled-OOF Threshold"
    ),
    fontsize=16,
    fontweight="bold",
    y=1.01
)

plt.tight_layout()
plt.show()

## Phase 4A: Spatial Block Cross-Validation

Spatial Block Cross-Validation evaluates model generalisation to geographically separated areas.

Grid cells are assigned to regular 25 km, 50 km, and 100 km spatial blocks. All observations within the same block remain in the same fold, preventing block-level overlap between training and validation data.

The class-weighted LightGBM model is evaluated using five spatial folds for each block size.

### ※ Related dissertation sections

- Section 2.11 (Validation and Geographical Generalisation)
- Section 3.1 (Research Approach—Evaluation)
- Section 4.7 (Spatial Validation and Block-Size Robustness)
- Table 4.9 (Spatial Block CV Performance)
- Figures 4.13–4.14
- Section 5.4 (Spatial Generalisation)

In [ ]:
# ============================================================
# Phase 4 - Cell 1.
# Multiple Spatial-Block Cross-Validation Settings
# ============================================================

from pyproj import Transformer

from sklearn.model_selection import (
    StratifiedGroupKFold
)

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    matthews_corrcoef,
    precision_score,
    recall_score,
    confusion_matrix,
    ConfusionMatrixDisplay
)


# ------------------------------------------------------------
# 1. Spatial validation settings
# ------------------------------------------------------------

SPATIAL_BLOCK_SIZES_M = [
    25_000,
    50_000,
    100_000
]

SPATIAL_N_SPLITS = 5

# Retained for compatibility with existing function defaults
SPATIAL_BLOCK_SIZE_M = 50_000

# The same candidate-seed range is used for every block size
# to ensure a consistent and reproducible comparison.
SPATIAL_SEED_RANGE = range(500)


# ------------------------------------------------------------
# 2. Validate settings
# ------------------------------------------------------------

if any(
    block_size <= 0
    for block_size in SPATIAL_BLOCK_SIZES_M
):
    raise ValueError(
        "All spatial block sizes must be positive."
    )

if len(
    SPATIAL_BLOCK_SIZES_M
) != len(
    set(SPATIAL_BLOCK_SIZES_M)
):
    raise ValueError(
        "Duplicate spatial block sizes detected."
    )

if SPATIAL_N_SPLITS < 2:
    raise ValueError(
        "At least two spatial folds are required."
    )


# ------------------------------------------------------------
# 3. Display settings
# ------------------------------------------------------------

print("=" * 70)
print("MULTIPLE SPATIAL-BLOCK CV SETTINGS")
print("=" * 70)

print(
    "Block sizes:",
    [
        f"{size / 1000:.0f} km"
        for size in SPATIAL_BLOCK_SIZES_M
    ]
)

print(
    "Spatial folds:",
    SPATIAL_N_SPLITS
)

print(
    "Candidate seeds:",
    len(SPATIAL_SEED_RANGE)
)

print(
    "Validation method:",
    "StratifiedGroupKFold"
)

print(
    "Model:",
    "Class Weight + LightGBM"
)

In [ ]:
# ============================================================
# Phase 4 - Cell 2.
# Construct Regular Spatial Blocks
# ============================================================

def create_spatial_blocks(
    X,
    block_size_m=SPATIAL_BLOCK_SIZE_M,
    n_splits=SPATIAL_N_SPLITS
):

    # --------------------------------------------------------
    # 1. Validate inputs
    # --------------------------------------------------------

    required_cols = [
        "longitude",
        "latitude"
    ]

    missing_cols = [
        column
        for column in required_cols
        if column not in X.columns
    ]

    if missing_cols:
        raise ValueError(
            f"Missing coordinate columns: {missing_cols}"
        )

    if block_size_m <= 0:
        raise ValueError(
            "block_size_m must be greater than zero."
        )

    longitude = X[
        "longitude"
    ].to_numpy(
        dtype=float
    )

    latitude = X[
        "latitude"
    ].to_numpy(
        dtype=float
    )

    if not np.isfinite(
        longitude
    ).all():
        raise ValueError(
            "Invalid longitude values detected."
        )

    if not np.isfinite(
        latitude
    ).all():
        raise ValueError(
            "Invalid latitude values detected."
        )

    if np.any(
        (longitude < -180)
        | (longitude > 180)
    ):
        raise ValueError(
            "Longitude values outside valid range."
        )

    if np.any(
        (latitude < -90)
        | (latitude > 90)
    ):
        raise ValueError(
            "Latitude values outside valid range."
        )


    # --------------------------------------------------------
    # 2. Convert WGS84 to British National Grid
    # --------------------------------------------------------

    transformer = Transformer.from_crs(
        "EPSG:4326",
        "EPSG:27700",
        always_xy=True
    )

    easting, northing = transformer.transform(
        longitude,
        latitude,
        errcheck=True
    )

    easting = np.asarray(
        easting,
        dtype=float
    )

    northing = np.asarray(
        northing,
        dtype=float
    )

    if not (
        np.isfinite(easting).all()
        and np.isfinite(northing).all()
    ):
        raise ValueError(
            "Invalid transformed coordinates detected."
        )


    # --------------------------------------------------------
    # 3. Construct regular spatial-block indices
    # --------------------------------------------------------

    block_x = np.floor(
        easting / block_size_m
    ).astype(int)

    block_y = np.floor(
        northing / block_size_m
    ).astype(int)

    block_id = np.asarray([
        f"{x}_{y}"
        for x, y in zip(
            block_x,
            block_y
        )
    ])


    # --------------------------------------------------------
    # 4. Create validation table
    # --------------------------------------------------------

    block_df = pd.DataFrame({
        "longitude": longitude,
        "latitude": latitude,
        "easting": easting,
        "northing": northing,
        "block_x": block_x,
        "block_y": block_y,
        "block_id": block_id
    })

    unique_blocks = (
        block_df["block_id"]
        .nunique()
    )

    if unique_blocks < n_splits:
        raise ValueError(
            f"Only {unique_blocks} spatial blocks were "
            f"created for {n_splits} folds."
        )


    # --------------------------------------------------------
    # 5. Summary
    # --------------------------------------------------------

    print("=" * 70)

    print(
        "Spatial block size:",
        f"{block_size_m / 1000:.0f} km"
    )

    print(
        "Observations:",
        f"{len(block_df):,}"
    )

    print(
        "Unique spatial blocks:",
        unique_blocks
    )

    print("\nObservations per block:")

    print(
        block_df[
            "block_id"
        ]
        .value_counts()
        .describe()
        .round(2)
    )

    return (
        block_id,
        block_df
    )

In [ ]:
# ============================================================
# Phase 4 - Cell 3.
# Search for Valid and Balanced Spatial Group Splits
# ============================================================

def search_balanced_spatial_splits(
    X,
    y,
    groups,
    n_splits=SPATIAL_N_SPLITS,
    seeds=range(500)
):

    # --------------------------------------------------------
    # 1. Validate inputs
    # --------------------------------------------------------

    y = np.asarray(
        y,
        dtype=int
    )

    groups = np.asarray(
        groups
    )

    if not (
        len(X)
        == len(y)
        == len(groups)
    ):
        raise ValueError(
            "X, y and groups must have the same length."
        )

    if len(np.unique(groups)) < n_splits:
        raise ValueError(
            "The number of spatial groups must be "
            "at least the number of folds."
        )

    classes = np.unique(
        y
    )

    expected_classes = np.arange(
        len(classes)
    )

    if not np.array_equal(
        classes,
        expected_classes
    ):
        raise ValueError(
            "Class labels must be consecutive integers "
            "starting from zero."
        )

    n_classes = len(
        classes
    )

    global_counts = np.bincount(
        y,
        minlength=n_classes
    )

    global_proportions = (
        global_counts / len(y)
    )

    expected_fold_size = (
        len(y) / n_splits
    )

    candidates = []


    # --------------------------------------------------------
    # 2. Evaluate candidate random seeds
    # --------------------------------------------------------

    for seed in seeds:

        cv = StratifiedGroupKFold(
            n_splits=n_splits,
            shuffle=True,
            random_state=seed
        )

        splits = list(
            cv.split(
                X,
                y,
                groups=groups
            )
        )

        fold_proportions = []
        fold_sizes = []

        validation_count = np.zeros(
            len(y),
            dtype=int
        )

        valid_candidate = True


        for train_idx, val_idx in splits:

            y_train = y[
                train_idx
            ]

            y_val = y[
                val_idx
            ]

            train_counts = np.bincount(
                y_train,
                minlength=n_classes
            )

            val_counts = np.bincount(
                y_val,
                minlength=n_classes
            )


            # Every training and validation fold
            # must contain every class
            if (
                np.any(train_counts == 0)
                or np.any(val_counts == 0)
            ):
                valid_candidate = False
                break


            # No spatial block may appear in both sets
            train_groups = set(
                groups[train_idx]
            )

            val_groups = set(
                groups[val_idx]
            )

            if train_groups & val_groups:
                valid_candidate = False
                break


            validation_count[
                val_idx
            ] += 1

            fold_proportions.append(
                val_counts / len(val_idx)
            )

            fold_sizes.append(
                len(val_idx)
            )


        if not valid_candidate:
            continue


        # Every observation must be validated once
        if not np.all(
            validation_count == 1
        ):
            continue


        fold_proportions = np.asarray(
            fold_proportions
        )

        fold_sizes = np.asarray(
            fold_sizes
        )


        # ----------------------------------------------------
        # 3. Class-proportion deviation
        # Relative deviation gives each class influence,
        # including the rare Poor class.
        # ----------------------------------------------------

        class_deviation = np.mean(
            np.abs(
                fold_proportions
                - global_proportions
            )
            / np.maximum(
                global_proportions,
                1e-12
            )
        )


        # ----------------------------------------------------
        # 4. Fold-size deviation
        # ----------------------------------------------------

        size_deviation = np.mean(
            np.abs(
                fold_sizes
                - expected_fold_size
            )
            / expected_fold_size
        )


        # Equal weighting of both objectives
        score = (
            class_deviation
            + size_deviation
        )


        candidates.append({
            "Seed": int(seed),
            "Score": score,
            "Class Deviation":
                class_deviation,
            "Size Deviation":
                size_deviation,
            "Minimum Fold Size":
                int(fold_sizes.min()),
            "Maximum Fold Size":
                int(fold_sizes.max())
        })


    # --------------------------------------------------------
    # 5. Validate candidate search
    # --------------------------------------------------------

    if not candidates:
        raise ValueError(
            "No valid spatial split was found. "
            "Consider changing the block size, "
            "number of folds or candidate seeds."
        )

    candidates_df = (
        pd.DataFrame(
            candidates
        )
        .sort_values(
            [
                "Score",
                "Seed"
            ],
            ascending=[
                True,
                True
            ]
        )
        .reset_index(drop=True)
    )

    best_seed = int(
        candidates_df.loc[
            0,
            "Seed"
        ]
    )


    # --------------------------------------------------------
    # 6. Reconstruct the selected splits
    # --------------------------------------------------------

    best_cv = StratifiedGroupKFold(
        n_splits=n_splits,
        shuffle=True,
        random_state=best_seed
    )

    best_splits = list(
        best_cv.split(
            X,
            y,
            groups=groups
        )
    )


    print(
        "Valid candidate seeds:",
        len(candidates_df)
    )

    print(
        "Selected seed:",
        best_seed
    )

    print(
        "Selected balance score:",
        f"{candidates_df.loc[0, 'Score']:.4f}"
    )

    return (
        best_seed,
        best_splits,
        candidates_df
    )

In [ ]:
# ============================================================
# Phase 4 - Cell 4.
# Validate Selected Spatial Group Folds
# ============================================================

def validate_spatial_splits(
    y,
    groups,
    splits,
    le,
    network_name
):

    y = np.asarray(
        y,
        dtype=int
    )

    groups = np.asarray(
        groups
    )

    if len(y) != len(groups):
        raise ValueError(
            "y and groups must have the same length."
        )

    class_order = [
        "Excellent",
        "Good",
        "Poor"
    ]

    rows = []

    validation_count = np.zeros(
        len(y),
        dtype=int
    )

    validation_block_assignments = {}


    for fold, (
        train_idx,
        val_idx
    ) in enumerate(
        splits,
        start=1
    ):

        # ----------------------------------------------------
        # 1. Spatial-block separation
        # ----------------------------------------------------

        train_blocks = set(
            groups[train_idx]
        )

        val_blocks = set(
            groups[val_idx]
        )

        overlap = (
            train_blocks
            & val_blocks
        )

        if overlap:
            raise RuntimeError(
                f"{network_name} Fold {fold}: "
                f"spatial block leakage detected: "
                f"{sorted(overlap)}"
            )


        # ----------------------------------------------------
        # 2. Check validation assignments
        # ----------------------------------------------------

        validation_count[
            val_idx
        ] += 1

        for block in val_blocks:

            if block in validation_block_assignments:
                raise RuntimeError(
                    f"{network_name}: block {block} "
                    "appears in more than one validation fold."
                )

            validation_block_assignments[
                block
            ] = fold


        # ----------------------------------------------------
        # 3. Class counts
        # ----------------------------------------------------

        train_labels = le.inverse_transform(
            y[train_idx]
        )

        val_labels = le.inverse_transform(
            y[val_idx]
        )

        train_counts = (
            pd.Series(train_labels)
            .value_counts()
            .reindex(
                class_order,
                fill_value=0
            )
        )

        val_counts = (
            pd.Series(val_labels)
            .value_counts()
            .reindex(
                class_order,
                fill_value=0
            )
        )


        if (
            (train_counts == 0).any()
            or (val_counts == 0).any()
        ):
            raise RuntimeError(
                f"{network_name} Fold {fold}: "
                "a class is absent from training or validation."
            )


        # ----------------------------------------------------
        # 4. Fold summary
        # ----------------------------------------------------

        rows.append({
            "Network": network_name,
            "Fold": fold,

            "Training Samples":
                len(train_idx),

            "Validation Samples":
                len(val_idx),

            "Training Blocks":
                len(train_blocks),

            "Validation Blocks":
                len(val_blocks),

            "Excellent":
                int(
                    val_counts["Excellent"]
                ),

            "Good":
                int(
                    val_counts["Good"]
                ),

            "Poor":
                int(
                    val_counts["Poor"]
                ),

            "Excellent (%)":
                val_counts["Excellent"]
                / len(val_idx)
                * 100,

            "Good (%)":
                val_counts["Good"]
                / len(val_idx)
                * 100,

            "Poor (%)":
                val_counts["Poor"]
                / len(val_idx)
                * 100
        })


    # --------------------------------------------------------
    # 5. Global split-integrity checks
    # --------------------------------------------------------

    if not np.all(
        validation_count == 1
    ):
        raise RuntimeError(
            f"{network_name}: every observation must appear "
            "exactly once in validation."
        )

    if len(
        validation_block_assignments
    ) != len(
        np.unique(groups)
    ):
        raise RuntimeError(
            f"{network_name}: not every spatial block was "
            "assigned to a validation fold."
        )


    summary = pd.DataFrame(
        rows
    )

    if summary[
        "Validation Samples"
    ].sum() != len(y):
        raise RuntimeError(
            f"{network_name}: validation sample totals "
            "do not match the dataset."
        )


    # --------------------------------------------------------
    # 6. Display results
    # --------------------------------------------------------

    print("=" * 70)

    print(
        network_name,
        "Spatial Fold Summary"
    )

    print("=" * 70)

    display(
        summary.round(2)
    )

    print(
        "Total validation samples:",
        f"{summary['Validation Samples'].sum():,}"
    )

    print(
        "Total unique validation blocks:",
        len(validation_block_assignments)
    )

    print(
        "✓ Every class is present in every fold."
    )

    print(
        "✓ Every observation is validated exactly once."
    )

    print(
        "✓ No spatial block leakage detected."
    )

    return summary

In [ ]:
# ============================================================
# Phase 4 - Cell 5.
# Spatially Grouped OOF Evaluation
# Class Weight + LightGBM
# ============================================================

def run_spatial_oof(
    X,
    y,
    splits,
    le,
    network_name,
    spatial_seed,
    block_size_m=SPATIAL_BLOCK_SIZE_M
):

    # --------------------------------------------------------
    # 1. Initialise OOF storage
    # --------------------------------------------------------

    oof_pred = np.full(
        len(y),
        fill_value=-1,
        dtype=int
    )

    validation_count = np.zeros(
        len(y),
        dtype=int
    )

    fold_results = []

    poor_label = int(
        le.transform(
            ["Poor"]
        )[0]
    )

    model_template = (
        create_baseline_models()[
            "LGBM"
        ]
    )


    # --------------------------------------------------------
    # 2. Spatial folds
    # --------------------------------------------------------

    for fold, (
        train_idx,
        val_idx
    ) in enumerate(
        splits,
        start=1
    ):

        X_train = X.iloc[
            train_idx
        ].copy()

        X_val = X.iloc[
            val_idx
        ].copy()

        y_train = y[
            train_idx
        ]

        y_val = y[
            val_idx
        ]


        # ----------------------------------------------------
        # 3. Validate predictors
        # ----------------------------------------------------

        if (
            X_train.isna().any().any()
            or X_val.isna().any().any()
        ):
            raise ValueError(
                f"{network_name} Fold {fold}: "
                "missing predictor values detected."
            )

        if not (
            np.isfinite(
                X_train.to_numpy(dtype=float)
            ).all()
            and np.isfinite(
                X_val.to_numpy(dtype=float)
            ).all()
        ):
            raise ValueError(
                f"{network_name} Fold {fold}: "
                "infinite predictor values detected."
            )


        # ----------------------------------------------------
        # 4. Training-fold class weights
        # ----------------------------------------------------

        weight_dict = (
            calculate_class_weight_dict(
                y_train
            )
        )

        sample_weight = (
            create_sample_weights(
                y_train,
                weight_dict
            )
        )


        # ----------------------------------------------------
        # 5. Fresh LightGBM model
        # ----------------------------------------------------

        model = clone(
            model_template
        )

        model.fit(
            X_train,
            y_train,
            sample_weight=sample_weight
        )


        # ----------------------------------------------------
        # 6. Spatial validation prediction
        # ----------------------------------------------------

        y_pred = model.predict(
            X_val
        ).astype(int)

        oof_pred[
            val_idx
        ] = y_pred

        validation_count[
            val_idx
        ] += 1


        # ----------------------------------------------------
        # 7. Fold-level results
        # ----------------------------------------------------

        fold_result = {
            "Network": network_name,
            "Fold": fold,

            "Training Samples":
                len(train_idx),

            "Validation Samples":
                len(val_idx),

            "Poor Support":
                int(
                    np.sum(
                        y_val == poor_label
                    )
                ),

            "Accuracy": accuracy_score(
                y_val,
                y_pred
            ),

            "Balanced Accuracy":
                balanced_accuracy_score(
                    y_val,
                    y_pred
                ),

            "Macro-F1": f1_score(
                y_val,
                y_pred,
                average="macro",
                zero_division=0
            ),

            "MCC": matthews_corrcoef(
                y_val,
                y_pred
            ),

            "Poor Precision": precision_score(
                y_val,
                y_pred,
                labels=[poor_label],
                average=None,
                zero_division=0
            )[0],

            "Poor Recall": recall_score(
                y_val,
                y_pred,
                labels=[poor_label],
                average=None,
                zero_division=0
            )[0],

            "Poor F1": f1_score(
                y_val,
                y_pred,
                labels=[poor_label],
                average=None,
                zero_division=0
            )[0]
        }

        fold_results.append(
            fold_result
        )

        print(
            f"{network_name} | Fold {fold}: "
            f"Macro-F1 = "
            f"{fold_result['Macro-F1']:.4f}, "
            f"Poor F1 = "
            f"{fold_result['Poor F1']:.4f}"
        )


    # --------------------------------------------------------
    # 8. Validate OOF assignments
    # --------------------------------------------------------

    if np.any(
        oof_pred == -1
    ):
        raise RuntimeError(
            f"{network_name}: missing spatial OOF predictions."
        )

    if not np.all(
        validation_count == 1
    ):
        raise RuntimeError(
            f"{network_name}: every observation must receive "
            "exactly one spatial OOF prediction."
        )


    # --------------------------------------------------------
    # 9. Overall spatial OOF metrics
    # --------------------------------------------------------

    fold_df = pd.DataFrame(
        fold_results
    )

    overall = {
        "Network": network_name,
        "Validation":
            "Spatially Grouped Block CV",

        "Method":
            "Class Weight",

        "Model":
            "LGBM",

        "N Rows":
            len(y),

        "N Features":
            X.shape[1],

        "Block Size (km)":
            block_size_m / 1000,

        "Spatial Seed":
            spatial_seed,

        "Accuracy": accuracy_score(
            y,
            oof_pred
        ),

        "Balanced Accuracy":
            balanced_accuracy_score(
                y,
                oof_pred
            ),

        "Macro-F1": f1_score(
            y,
            oof_pred,
            average="macro",
            zero_division=0
        ),

        "MCC": matthews_corrcoef(
            y,
            oof_pred
        ),

        "Poor Precision": precision_score(
            y,
            oof_pred,
            labels=[poor_label],
            average=None,
            zero_division=0
        )[0],

        "Poor Recall": recall_score(
            y,
            oof_pred,
            labels=[poor_label],
            average=None,
            zero_division=0
        )[0],

        "Poor F1": f1_score(
            y,
            oof_pred,
            labels=[poor_label],
            average=None,
            zero_division=0
        )[0],

        "Poor Support": int(
            np.sum(
                y == poor_label
            )
        ),

        "CV Macro-F1 Mean":
            fold_df[
                "Macro-F1"
            ].mean(),

        "CV Macro-F1 SD":
            fold_df[
                "Macro-F1"
            ].std(ddof=0),

        "CV Poor F1 Mean":
            fold_df[
                "Poor F1"
            ].mean(),

        "CV Poor F1 SD":
            fold_df[
                "Poor F1"
            ].std(ddof=0)
    }


    # --------------------------------------------------------
    # 10. Display summary
    # --------------------------------------------------------

    print("\n" + "=" * 70)
    print(
        network_name,
        "Spatial OOF Summary"
    )
    print("=" * 70)

    print(
        f"Macro-F1      : "
        f"{overall['Macro-F1']:.4f}"
    )

    print(
        f"CV Macro-F1   : "
        f"{overall['CV Macro-F1 Mean']:.4f} "
        f"± {overall['CV Macro-F1 SD']:.4f}"
    )

    print(
        f"Poor F1       : "
        f"{overall['Poor F1']:.4f}"
    )

    print(
        f"CV Poor F1    : "
        f"{overall['CV Poor F1 Mean']:.4f} "
        f"± {overall['CV Poor F1 SD']:.4f}"
    )

    return (
        overall,
        fold_df,
        oof_pred
    )

In [ ]:
# ============================================================
# Phase 4 - Cell 6.
# Run 25-, 50- and 100-km Spatial Block CV
# ============================================================

NETWORK_CONFIGS = {
    "LTE": {
        "X": X_lte,
        "y": y_lte,
        "le": le_lte
    },

    "5G NR": {
        "X": X_nr,
        "y": y_nr,
        "le": le_nr
    }
}


# ------------------------------------------------------------
# 1. Result containers
# ------------------------------------------------------------

spatial_overall_rows = []
spatial_fold_result_tables = []
spatial_fold_summary_tables = []
spatial_block_summary_rows = []

spatial_oof_predictions = {}
spatial_split_objects = {}
spatial_group_objects = {}
spatial_candidate_tables = {}


# ------------------------------------------------------------
# 2. Run every network and block-size combination
# ------------------------------------------------------------

for block_size_m in SPATIAL_BLOCK_SIZES_M:

    block_size_km = (
        block_size_m / 1000
    )

    for network_name, config in (
        NETWORK_CONFIGS.items()
    ):

        X = config["X"]
        y = config["y"]
        le = config["le"]

        experiment_key = (
            network_name,
            int(block_size_km)
        )

        print("\n" + "=" * 90)

        print(
            f"{network_name} | "
            f"{block_size_km:.0f}-km Spatial Block CV"
        )

        print("=" * 90)


        # ----------------------------------------------------
        # A. Construct spatial blocks
        # ----------------------------------------------------

        groups, block_df = (
            create_spatial_blocks(
                X=X,
                block_size_m=block_size_m,
                n_splits=SPATIAL_N_SPLITS
            )
        )

        if not (
            len(groups)
            == len(X)
            == len(y)
        ):
            raise RuntimeError(
                f"{network_name} | "
                f"{block_size_km:.0f} km: "
                "X, y and spatial groups are not aligned."
            )

        unique_blocks = int(
            block_df["block_id"].nunique()
        )


        # ----------------------------------------------------
        # B. Search for balanced spatial folds
        # ----------------------------------------------------

        (
            selected_seed,
            selected_splits,
            candidate_table
        ) = search_balanced_spatial_splits(
            X=X,
            y=y,
            groups=groups,
            n_splits=SPATIAL_N_SPLITS,
            seeds=SPATIAL_SEED_RANGE
        )


        # ----------------------------------------------------
        # C. Validate fold composition
        # ----------------------------------------------------

        fold_summary = (
            validate_spatial_splits(
                y=y,
                groups=groups,
                splits=selected_splits,
                le=le,
                network_name=network_name
            )
        )

        fold_summary.insert(
            1,
            "Block Size (km)",
            block_size_km
        )

        fold_summary[
            "Spatial Seed"
        ] = selected_seed


        # ----------------------------------------------------
        # D. Run Class Weight + LightGBM
        # ----------------------------------------------------

        (
            overall_result,
            fold_results,
            oof_predictions
        ) = run_spatial_oof(
            X=X,
            y=y,
            splits=selected_splits,
            le=le,
            network_name=network_name,
            spatial_seed=selected_seed,
            block_size_m=block_size_m
        )

        fold_results.insert(
            1,
            "Block Size (km)",
            block_size_km
        )

        fold_results[
            "Spatial Seed"
        ] = selected_seed


        # ----------------------------------------------------
        # E. Add block and fold statistics
        # ----------------------------------------------------

        overall_result[
            "Unique Spatial Blocks"
        ] = unique_blocks

        fold_metrics = [
            "Accuracy",
            "Balanced Accuracy",
            "Macro-F1",
            "MCC",
            "Poor Precision",
            "Poor Recall",
            "Poor F1"
        ]

        for metric in fold_metrics:

            overall_result[
                f"CV {metric} Mean"
            ] = fold_results[
                metric
            ].mean()

            overall_result[
                f"CV {metric} SD"
            ] = fold_results[
                metric
            ].std(ddof=0)


        # ----------------------------------------------------
        # F. Store results
        # ----------------------------------------------------

        spatial_overall_rows.append(
            overall_result
        )

        spatial_fold_result_tables.append(
            fold_results
        )

        spatial_fold_summary_tables.append(
            fold_summary
        )

        spatial_block_summary_rows.append({
            "Network": network_name,
            "Block Size (km)": block_size_km,
            "Observations": len(X),
            "Unique Spatial Blocks":
                unique_blocks,
            "Mean Observations per Block":
                block_df[
                    "block_id"
                ].value_counts().mean(),
            "Median Observations per Block":
                block_df[
                    "block_id"
                ].value_counts().median(),
            "Minimum Observations per Block":
                block_df[
                    "block_id"
                ].value_counts().min(),
            "Maximum Observations per Block":
                block_df[
                    "block_id"
                ].value_counts().max(),
            "Spatial Seed": selected_seed,
            "Valid Candidate Seeds":
                len(candidate_table)
        })

        spatial_oof_predictions[
            experiment_key
        ] = oof_predictions.copy()

        spatial_split_objects[
            experiment_key
        ] = selected_splits

        spatial_group_objects[
            experiment_key
        ] = groups.copy()

        candidate_table = (
            candidate_table.copy()
        )

        candidate_table.insert(
            0,
            "Network",
            network_name
        )

        candidate_table.insert(
            1,
            "Block Size (km)",
            block_size_km
        )

        spatial_candidate_tables[
            experiment_key
        ] = candidate_table

        print(
            f"\n✓ Completed: {network_name} | "
            f"{block_size_km:.0f} km"
        )


# ------------------------------------------------------------
# 3. Combine all results
# ------------------------------------------------------------

spatial_overall_results = pd.DataFrame(
    spatial_overall_rows
)

spatial_fold_results = pd.concat(
    spatial_fold_result_tables,
    ignore_index=True
)

spatial_fold_summaries = pd.concat(
    spatial_fold_summary_tables,
    ignore_index=True
)

spatial_block_summary = pd.DataFrame(
    spatial_block_summary_rows
)


# ------------------------------------------------------------
# 4. Sort results
# ------------------------------------------------------------

network_order = [
    "LTE",
    "5G NR"
]

for table in [
    spatial_overall_results,
    spatial_fold_results,
    spatial_fold_summaries,
    spatial_block_summary
]:

    table["Network"] = pd.Categorical(
        table["Network"],
        categories=network_order,
        ordered=True
    )

    table.sort_values(
        [
            "Network",
            "Block Size (km)"
        ],
        inplace=True
    )

    table.reset_index(
        drop=True,
        inplace=True
    )


# ------------------------------------------------------------
# 5. Validate completed experiments
# ------------------------------------------------------------

expected_experiments = (
    len(NETWORK_CONFIGS)
    * len(SPATIAL_BLOCK_SIZES_M)
)

if len(
    spatial_overall_results
) != expected_experiments:

    raise RuntimeError(
        f"Expected {expected_experiments} spatial "
        f"experiments, but found "
        f"{len(spatial_overall_results)}."
    )

if len(
    spatial_fold_results
) != (
    expected_experiments
    * SPATIAL_N_SPLITS
):

    raise RuntimeError(
        "Unexpected number of spatial fold results."
    )


# ------------------------------------------------------------
# 6. Display main results
# ------------------------------------------------------------

spatial_display_cols = [
    "Network",
    "Block Size (km)",
    "Unique Spatial Blocks",
    "Spatial Seed",
    "N Rows",
    "Macro-F1",
    "CV Macro-F1 Mean",
    "CV Macro-F1 SD",
    "Balanced Accuracy",
    "MCC",
    "Poor Precision",
    "Poor Recall",
    "Poor F1",
    "CV Poor F1 Mean",
    "CV Poor F1 SD"
]


print("\n" + "=" * 90)
print("MULTIPLE SPATIAL-BLOCK CV RESULTS")
print("=" * 90)

display(
    spatial_overall_results[
        spatial_display_cols
    ].round(4)
)


print("\n" + "=" * 90)
print("SPATIAL BLOCK SUMMARY")
print("=" * 90)

display(
    spatial_block_summary.round(2)
)


print(
    "\n✓ All six spatial-block experiments "
    "completed successfully."
)

## Phase 4B: Measurement-Density Sensitivity Analysis

Label reliability is examined by progressively excluding grids with low numbers of Ofcom measurements.

Separate LTE and 5G NR experiments apply minimum measurement-count thresholds of 1, 20, 50, and 100. The class-weighted LightGBM model is re-evaluated at each threshold using stratified five-fold cross-validation.

### ※ Related dissertation sections

- Section 3.1 (Research Approach—Evaluation)
- Section 4.8 (Ofcom Measurement Density and Label-Reliability Sensitivity)
- Table 4.10 (Grid Retention and Class Distribution)
- Table 4.11 (Classification Sensitivity to Minimum Measurement Count)
- Figure 4.15 (Distribution of Ofcom Measurement Counts)

In [ ]:
# ============================================================
# Phase 4 - Cell 7.
# Measurement-Count Sensitivity Evaluation Function
# Class Weight + LightGBM
# ============================================================

MEASUREMENT_THRESHOLDS = [
    1,
    20,
    50,
    100
]


def calculate_sensitivity_metrics(
    y_true,
    y_pred,
    poor_label
):

    return {
        "Accuracy": accuracy_score(
            y_true,
            y_pred
        ),

        "Balanced Accuracy":
            balanced_accuracy_score(
                y_true,
                y_pred
            ),

        "Macro-F1": f1_score(
            y_true,
            y_pred,
            average="macro",
            zero_division=0
        ),

        "MCC": matthews_corrcoef(
            y_true,
            y_pred
        ),

        "Poor Precision": precision_score(
            y_true,
            y_pred,
            labels=[poor_label],
            average=None,
            zero_division=0
        )[0],

        "Poor Recall": recall_score(
            y_true,
            y_pred,
            labels=[poor_label],
            average=None,
            zero_division=0
        )[0],

        "Poor F1": f1_score(
            y_true,
            y_pred,
            labels=[poor_label],
            average=None,
            zero_division=0
        )[0]
    }


def run_measurement_sensitivity_oof(
    labelled_df,
    target_col,
    count_col,
    minimum_count,
    network_name
):

    # --------------------------------------------------------
    # 1. Validate columns
    # --------------------------------------------------------

    required_cols = (
        [
            "grid_id",
            target_col,
            count_col
        ]
        + feature_cols
    )

    missing_cols = [
        column
        for column in required_cols
        if column not in labelled_df.columns
    ]

    if missing_cols:
        raise KeyError(
            f"{network_name}: missing columns: "
            f"{missing_cols}"
        )


    # --------------------------------------------------------
    # 2. Validate and filter measurement counts
    # --------------------------------------------------------

    data = labelled_df.copy()

    data[count_col] = pd.to_numeric(
        data[count_col],
        errors="coerce"
    )

    if data[count_col].isna().any():
        raise ValueError(
            f"{network_name}: invalid values "
            f"detected in {count_col}."
        )

    if (
        data[count_col] <= 0
    ).any():
        raise ValueError(
            f"{network_name}: measurement counts "
            "must be positive."
        )

    original_n = len(data)

    filtered_df = (
        data[
            data[count_col]
            >= minimum_count
        ]
        .sort_values("grid_id")
        .reset_index(drop=True)
        .copy()
    )

    retained_n = len(filtered_df)
    excluded_n = original_n - retained_n

    if retained_n == 0:
        raise ValueError(
            f"{network_name} | minimum count "
            f"{minimum_count}: no grids retained."
        )


    # --------------------------------------------------------
    # 3. Check retained class distribution
    # --------------------------------------------------------

    class_counts = (
        filtered_df[target_col]
        .value_counts()
        .reindex(
            VALID_CLASSES,
            fill_value=0
        )
        .astype(int)
    )

    if (
        class_counts < N_SPLITS
    ).any():

        raise ValueError(
            f"{network_name} | minimum count "
            f"{minimum_count}: insufficient class "
            f"support for {N_SPLITS}-fold CV: "
            f"{class_counts.to_dict()}"
        )


    # --------------------------------------------------------
    # 4. Prepare predictors and target
    # --------------------------------------------------------

    X = filtered_df[
        feature_cols
    ].copy()

    if X.shape[1] != 35:
        raise ValueError(
            f"Expected 35 predictors, "
            f"but found {X.shape[1]}."
        )

    if X.isna().any().any():
        raise ValueError(
            f"{network_name}: missing predictors."
        )

    if not np.isfinite(
        X.to_numpy(dtype=float)
    ).all():
        raise ValueError(
            f"{network_name}: infinite predictors."
        )

    y_text = (
        filtered_df[target_col]
        .astype("string")
        .str.strip()
    )

    label_encoder = LabelEncoder()

    y = label_encoder.fit_transform(
        y_text
    )

    if set(
        label_encoder.classes_
    ) != set(
        VALID_CLASSES
    ):
        raise ValueError(
            f"{network_name}: unexpected classes "
            f"after filtering."
        )

    poor_label = int(
        label_encoder.transform(
            ["Poor"]
        )[0]
    )


    # --------------------------------------------------------
    # 5. Create five stratified folds
    # --------------------------------------------------------

    cv = StratifiedKFold(
        n_splits=N_SPLITS,
        shuffle=True,
        random_state=RANDOM_STATE
    )

    cv_splits = list(
        cv.split(
            X,
            y
        )
    )


    # --------------------------------------------------------
    # 6. Prepare OOF storage
    # --------------------------------------------------------

    oof_pred = np.full(
        len(y),
        -1,
        dtype=int
    )

    validation_count = np.zeros(
        len(y),
        dtype=int
    )

    fold_rows = []

    model_template = (
        create_baseline_models()[
            "LGBM"
        ]
    )


    # --------------------------------------------------------
    # 7. Train and evaluate each fold
    # --------------------------------------------------------

    for fold, (
        train_idx,
        val_idx
    ) in enumerate(
        cv_splits,
        start=1
    ):

        X_train = X.iloc[
            train_idx
        ].copy()

        X_val = X.iloc[
            val_idx
        ].copy()

        y_train = y[
            train_idx
        ]

        y_val = y[
            val_idx
        ]


        # Calculate weights using training data only
        weight_dict = (
            calculate_class_weight_dict(
                y_train
            )
        )

        sample_weight = (
            create_sample_weights(
                y_train,
                weight_dict
            )
        )


        # Train a fresh LightGBM model
        model = clone(
            model_template
        )

        model.fit(
            X_train,
            y_train,
            sample_weight=sample_weight
        )

        y_pred = model.predict(
            X_val
        ).astype(int)


        # Store OOF prediction
        oof_pred[
            val_idx
        ] = y_pred

        validation_count[
            val_idx
        ] += 1


        # Calculate fold metrics
        fold_metrics = (
            calculate_sensitivity_metrics(
                y_true=y_val,
                y_pred=y_pred,
                poor_label=poor_label
            )
        )

        fold_rows.append({
            "Network": network_name,
            "Minimum Measurements":
                minimum_count,
            "Fold": fold,
            "Training Samples":
                len(train_idx),
            "Validation Samples":
                len(val_idx),
            "Poor Support": int(
                np.sum(
                    y_val == poor_label
                )
            ),
            **fold_metrics
        })


    # --------------------------------------------------------
    # 8. Validate OOF assignments
    # --------------------------------------------------------

    if np.any(
        oof_pred == -1
    ):
        raise RuntimeError(
            f"{network_name} | minimum count "
            f"{minimum_count}: missing OOF predictions."
        )

    if not np.all(
        validation_count == 1
    ):
        raise RuntimeError(
            f"{network_name} | minimum count "
            f"{minimum_count}: invalid OOF assignments."
        )


    # --------------------------------------------------------
    # 9. Overall OOF metrics
    # --------------------------------------------------------

    fold_df = pd.DataFrame(
        fold_rows
    )

    pooled_metrics = (
        calculate_sensitivity_metrics(
            y_true=y,
            y_pred=oof_pred,
            poor_label=poor_label
        )
    )

    overall = {
        "Network": network_name,
        "Minimum Measurements":
            minimum_count,
        "Method": "Class Weight",
        "Model": "LGBM",
        "Original Grids": original_n,
        "Retained Grids": retained_n,
        "Excluded Grids": excluded_n,
        "Retained (%)":
            retained_n / original_n * 100,
        "Excellent":
            int(class_counts["Excellent"]),
        "Good":
            int(class_counts["Good"]),
        "Poor":
            int(class_counts["Poor"]),
        "Poor Prevalence (%)":
            class_counts["Poor"]
            / retained_n
            * 100,
        "N Features": X.shape[1],
        **pooled_metrics
    }


    # --------------------------------------------------------
    # 10. Mean and SD across folds
    # --------------------------------------------------------

    metric_names = [
        "Accuracy",
        "Balanced Accuracy",
        "Macro-F1",
        "MCC",
        "Poor Precision",
        "Poor Recall",
        "Poor F1"
    ]

    for metric in metric_names:

        overall[
            f"CV {metric} Mean"
        ] = fold_df[
            metric
        ].mean()

        overall[
            f"CV {metric} SD"
        ] = fold_df[
            metric
        ].std(ddof=0)


    # --------------------------------------------------------
    # 11. Display progress
    # --------------------------------------------------------

    print(
        f"{network_name} | "
        f"minimum ≥ {minimum_count} | "
        f"retained {retained_n:,} "
        f"({retained_n / original_n * 100:.2f}%) | "
        f"Macro-F1 "
        f"{overall['CV Macro-F1 Mean']:.4f} "
        f"± {overall['CV Macro-F1 SD']:.4f} | "
        f"Poor F1 "
        f"{overall['CV Poor F1 Mean']:.4f} "
        f"± {overall['CV Poor F1 SD']:.4f}"
    )


    return (
        overall,
        fold_df,
        oof_pred,
        filtered_df["grid_id"].copy()
    )

In [ ]:
# ============================================================
# Phase 4 - Cell 8.
# Run Measurement-Count Sensitivity Analysis
# ============================================================

MEASUREMENT_SENSITIVITY_CONFIGS = {
    "LTE": {
        "data": lte_df,
        "target_col": "lte_signal_class",
        "count_col": "lte_point_count"
    },

    "5G NR": {
        "data": nr_df,
        "target_col": "nr_signal_class",
        "count_col": "nr_point_count"
    }
}


# ------------------------------------------------------------
# 1. Result containers
# ------------------------------------------------------------

measurement_sensitivity_rows = []
measurement_sensitivity_fold_tables = []

measurement_sensitivity_predictions = {}
measurement_sensitivity_grid_ids = {}


# ------------------------------------------------------------
# 2. Run all eight experiments
# ------------------------------------------------------------

for network_name, config in (
    MEASUREMENT_SENSITIVITY_CONFIGS.items()
):

    print("\n" + "=" * 90)

    print(
        f"{network_name} Measurement-Count "
        "Sensitivity Analysis"
    )

    print("=" * 90)

    for minimum_count in (
        MEASUREMENT_THRESHOLDS
    ):

        experiment_key = (
            network_name,
            minimum_count
        )

        (
            overall_result,
            fold_results,
            oof_predictions,
            retained_grid_ids
        ) = run_measurement_sensitivity_oof(
            labelled_df=config["data"],
            target_col=config["target_col"],
            count_col=config["count_col"],
            minimum_count=minimum_count,
            network_name=network_name
        )

        measurement_sensitivity_rows.append(
            overall_result
        )

        measurement_sensitivity_fold_tables.append(
            fold_results
        )

        measurement_sensitivity_predictions[
            experiment_key
        ] = oof_predictions.copy()

        measurement_sensitivity_grid_ids[
            experiment_key
        ] = retained_grid_ids.copy()


# ------------------------------------------------------------
# 3. Combine result tables
# ------------------------------------------------------------

measurement_sensitivity_results = (
    pd.DataFrame(
        measurement_sensitivity_rows
    )
)

measurement_sensitivity_fold_results = (
    pd.concat(
        measurement_sensitivity_fold_tables,
        ignore_index=True
    )
)


# ------------------------------------------------------------
# 4. Sort result tables
# ------------------------------------------------------------

network_order = [
    "LTE",
    "5G NR"
]

for table in [
    measurement_sensitivity_results,
    measurement_sensitivity_fold_results
]:

    table["Network"] = pd.Categorical(
        table["Network"],
        categories=network_order,
        ordered=True
    )

    table.sort_values(
        [
            "Network",
            "Minimum Measurements"
        ],
        inplace=True
    )

    table.reset_index(
        drop=True,
        inplace=True
    )


# ------------------------------------------------------------
# 5. Validate result counts
# ------------------------------------------------------------

expected_experiments = (
    len(
        MEASUREMENT_SENSITIVITY_CONFIGS
    )
    * len(
        MEASUREMENT_THRESHOLDS
    )
)

if len(
    measurement_sensitivity_results
) != expected_experiments:

    raise RuntimeError(
        "Unexpected number of sensitivity results."
    )

if len(
    measurement_sensitivity_fold_results
) != (
    expected_experiments
    * N_SPLITS
):

    raise RuntimeError(
        "Unexpected number of sensitivity fold results."
    )


# ------------------------------------------------------------
# 6. Confirm that threshold ≥1 reproduces the original model
# ------------------------------------------------------------

for network_name in [
    "LTE",
    "5G NR"
]:

    sensitivity_baseline = (
        measurement_sensitivity_results[
            (
                measurement_sensitivity_results[
                    "Network"
                ] == network_name
            )
            & (
                measurement_sensitivity_results[
                    "Minimum Measurements"
                ] == 1
            )
        ]
    )

    original_baseline = imbalance_results[
        (
            imbalance_results["Network"]
            == network_name
        )
        & (
            imbalance_results["Method"]
            == "Class Weight"
        )
        & (
            imbalance_results["Model"]
            == "LGBM"
        )
    ]

    if len(sensitivity_baseline) != 1:
        raise RuntimeError(
            f"{network_name}: sensitivity baseline "
            "was not found."
        )

    if len(original_baseline) != 1:
        raise RuntimeError(
            f"{network_name}: original Class Weight "
            "+ LGBM result was not found."
        )

    observed_macro_f1 = float(
        sensitivity_baseline.iloc[0][
            "Macro-F1"
        ]
    )

    expected_macro_f1 = float(
        original_baseline.iloc[0][
            "Macro-F1"
        ]
    )

    if not np.isclose(
        observed_macro_f1,
        expected_macro_f1,
        atol=1e-12,
        rtol=0
    ):
        raise RuntimeError(
            f"{network_name}: threshold ≥1 did not "
            "reproduce the original result. "
            f"Observed={observed_macro_f1:.6f}, "
            f"Expected={expected_macro_f1:.6f}"
        )


# ------------------------------------------------------------
# 7. Display final results
# ------------------------------------------------------------

sensitivity_display_cols = [
    "Network",
    "Minimum Measurements",
    "Retained Grids",
    "Retained (%)",
    "Excellent",
    "Good",
    "Poor",
    "Poor Prevalence (%)",
    "Macro-F1",
    "CV Macro-F1 Mean",
    "CV Macro-F1 SD",
    "Balanced Accuracy",
    "MCC",
    "Poor Precision",
    "Poor Recall",
    "Poor F1",
    "CV Poor F1 Mean",
    "CV Poor F1 SD"
]


print("\n" + "=" * 90)
print("MEASUREMENT-COUNT SENSITIVITY RESULTS")
print("=" * 90)

display(
    measurement_sensitivity_results[
        sensitivity_display_cols
    ].round(4)
)


print(
    "\n✓ Measurement-count sensitivity "
    "analysis completed successfully."
)

## Phase 5: Dependency-Aware Feature-Group Ablation

Feature groups are removed one at a time to examine their contribution to predictive performance.

The ablation design accounts for dependencies between base predictors and derived interaction features. All experiments use the same cross-validation folds and the class-weighted LightGBM model.

Performance changes are calculated relative to the complete 35-predictor model.

### ※ Related dissertation sections

- Section 2.10 (Feature Contribution and Predictor-Set Complexity)
- Section 4.9 (Feature-Group Ablation Study)
- Table 4.12 (Performance Changes following Feature-Group Removal)
- Figure 4.16 (Dependency-Aware Feature-Group Ablation Design)
- Section 5.5 (Feature Ablation and Predictor-Set Complexity)

In [ ]:
# ============================================================
# Phase 5 - Cell 1.
# Define Dependency-Aware Feature Groups
# ============================================================

# ------------------------------------------------------------
# 1. Base feature groups
# ------------------------------------------------------------

SATELLITE_BASE = [
    "VV",
    "VH",
    "NDVI",
    "NDBI"
]

LAND_COVER_BASE = [
    "tree_ratio",
    "grass_ratio",
    "crop_ratio",
    "builtup_ratio",
    "water_ratio"
]

SOCIOECONOMIC_BASE = [
    "population",
    "nightlight"
]

CELL_INFRASTRUCTURE_BASE = [
    "cell_count",
    "operator_count",
    "gsm_count",
    "umts_count",
    "lte_count",
    "nr_count",
    "nearest_lte_distance_m",
    "nearest_nr_distance_m",
    "lte_density_3km",
    "nr_density_3km",
    "lte_density_5km",
    "nr_density_5km"
]

ENGINEERED_INTERACTIONS = [
    "pop_x_lte",
    "pop_x_nr",
    "builtup_x_lte",
    "builtup_x_nr",
    "nightlight_x_lte",
    "nightlight_x_nr",
    "vv_x_ndvi",
    "vh_x_builtup",
    "ndvi_minus_ndbi"
]

GEOGRAPHIC = [
    "longitude",
    "latitude",
    "distance_to_london_km"
]


# ------------------------------------------------------------
# 2. Validate the base partition of all 35 predictors
# ------------------------------------------------------------

BASE_FEATURE_GROUPS = {
    "Satellite": SATELLITE_BASE,
    "Land Cover": LAND_COVER_BASE,
    "Socioeconomic": SOCIOECONOMIC_BASE,
    "Cell Infrastructure":
        CELL_INFRASTRUCTURE_BASE,
    "Engineered Interactions":
        ENGINEERED_INTERACTIONS,
    "Geographic": GEOGRAPHIC
}

all_grouped_features = [
    feature
    for group in BASE_FEATURE_GROUPS.values()
    for feature in group
]

missing_from_groups = sorted(
    set(feature_cols)
    - set(all_grouped_features)
)

unexpected_features = sorted(
    set(all_grouped_features)
    - set(feature_cols)
)

duplicated_features = sorted({
    feature
    for feature in all_grouped_features
    if all_grouped_features.count(feature) > 1
})

if missing_from_groups:
    raise ValueError(
        "Predictors not assigned to a base group: "
        f"{missing_from_groups}"
    )

if unexpected_features:
    raise ValueError(
        "Grouped features not present in Dataset25: "
        f"{unexpected_features}"
    )

if duplicated_features:
    raise ValueError(
        "Predictors assigned to multiple base groups: "
        f"{duplicated_features}"
    )

if len(all_grouped_features) != 35:
    raise ValueError(
        f"Expected 35 grouped predictors, "
        f"but found {len(all_grouped_features)}."
    )


# ------------------------------------------------------------
# 3. Define dependent interaction features
# ------------------------------------------------------------

SATELLITE_DEPENDENCIES = [
    "vv_x_ndvi",
    "vh_x_builtup",
    "ndvi_minus_ndbi"
]

LAND_COVER_DEPENDENCIES = [
    "builtup_x_lte",
    "builtup_x_nr",
    "vh_x_builtup"
]

SOCIOECONOMIC_DEPENDENCIES = [
    "pop_x_lte",
    "pop_x_nr",
    "nightlight_x_lte",
    "nightlight_x_nr"
]

INFRASTRUCTURE_DEPENDENCIES = [
    "pop_x_lte",
    "pop_x_nr",
    "builtup_x_lte",
    "builtup_x_nr",
    "nightlight_x_lte",
    "nightlight_x_nr"
]


# ------------------------------------------------------------
# 4. Create dependency-aware removal groups
# ------------------------------------------------------------

def preserve_feature_order(
    selected_features
):

    selected_set = set(
        selected_features
    )

    return [
        feature
        for feature in feature_cols
        if feature in selected_set
    ]


ABLATION_REMOVAL_GROUPS = {
    "Satellite": preserve_feature_order(
        SATELLITE_BASE
        + SATELLITE_DEPENDENCIES
    ),

    "Land Cover": preserve_feature_order(
        LAND_COVER_BASE
        + LAND_COVER_DEPENDENCIES
    ),

    "Socioeconomic": preserve_feature_order(
        SOCIOECONOMIC_BASE
        + SOCIOECONOMIC_DEPENDENCIES
    ),

    "Cell Infrastructure":
        preserve_feature_order(
            CELL_INFRASTRUCTURE_BASE
            + INFRASTRUCTURE_DEPENDENCIES
        ),

    "Engineered Interactions":
        preserve_feature_order(
            ENGINEERED_INTERACTIONS
        ),

    "Geographic": preserve_feature_order(
        GEOGRAPHIC
    )
}


# ------------------------------------------------------------
# 5. Display ablation design
# ------------------------------------------------------------

print(
    "All 35 predictors were assigned "
    "to exactly one base group."
)

print(
    "\nDependency-aware ablation groups:"
)

for group_name, removal_features in (
    ABLATION_REMOVAL_GROUPS.items()
):

    print(
        f"\n{group_name}: "
        f"remove {len(removal_features)} features"
    )

    print(removal_features)

In [ ]:
# ============================================================
# Phase 5 - Cell 2.
# Create Dependency-Aware Ablation Experiments
# ============================================================

# ------------------------------------------------------------
# 1. Full model plus one-group-at-a-time removal
# ------------------------------------------------------------

ABLATION_EXPERIMENTS = {
    "Full Model": []
}

for group_name, removal_features in (
    ABLATION_REMOVAL_GROUPS.items()
):

    ABLATION_EXPERIMENTS[
        f"- {group_name}"
    ] = removal_features.copy()


# ------------------------------------------------------------
# 2. Validate every ablation experiment
# ------------------------------------------------------------

ablation_design_rows = []

for experiment_name, removed_features in (
    ABLATION_EXPERIMENTS.items()
):

    unknown_features = sorted(
        set(removed_features)
        - set(feature_cols)
    )

    if unknown_features:
        raise ValueError(
            f"{experiment_name}: unknown features: "
            f"{unknown_features}"
        )

    if len(removed_features) != len(
        set(removed_features)
    ):
        raise ValueError(
            f"{experiment_name}: duplicate removal "
            "features detected."
        )

    retained_features = [
        feature
        for feature in feature_cols
        if feature not in set(
            removed_features
        )
    ]

    if experiment_name == "Full Model":

        assert len(
            retained_features
        ) == 35

    elif len(retained_features) == 0:

        raise ValueError(
            f"{experiment_name}: no predictors remain."
        )

    ablation_design_rows.append({
        "Experiment": experiment_name,
        "Removed Features":
            len(removed_features),
        "Retained Features":
            len(retained_features),
        "Removed Feature Names":
            ", ".join(
                removed_features
            )
    })


ablation_design = pd.DataFrame(
    ablation_design_rows
)


# ------------------------------------------------------------
# 3. Display experiment design
# ------------------------------------------------------------

print(
    "Number of ablation experiments:",
    len(ABLATION_EXPERIMENTS)
)

display(
    ablation_design
)

In [ ]:
# ============================================================
# Phase 5 - Cell 3.
# Five-Fold OOF Dependency-Aware Ablation Evaluation
# Model: Class Weight + LightGBM
# ============================================================

def evaluate_ablation(
    X,
    y,
    le,
    network_name,
    experiment_name,
    removed_features,
    cv_splits
):

    # --------------------------------------------------------
    # 1. Validate removal features
    # --------------------------------------------------------

    unknown_features = sorted(
        set(removed_features)
        - set(X.columns)
    )

    if unknown_features:
        raise ValueError(
            f"{experiment_name}: unknown removal features: "
            f"{unknown_features}"
        )

    if len(removed_features) != len(
        set(removed_features)
    ):
        raise ValueError(
            f"{experiment_name}: duplicated removal features."
        )


    # --------------------------------------------------------
    # 2. Select remaining predictors
    # --------------------------------------------------------

    removed_set = set(
        removed_features
    )

    selected_features = [
        feature
        for feature in X.columns
        if feature not in removed_set
    ]

    if not selected_features:
        raise ValueError(
            f"{experiment_name}: no predictors remain."
        )

    X_selected = X[
        selected_features
    ].copy()

    if X_selected.isna().any().any():
        raise ValueError(
            f"{experiment_name}: missing predictor values."
        )

    if not np.isfinite(
        X_selected.to_numpy(dtype=float)
    ).all():
        raise ValueError(
            f"{experiment_name}: infinite predictor values."
        )


    # --------------------------------------------------------
    # 3. Initialise OOF storage
    # --------------------------------------------------------

    oof_pred = np.full(
        len(y),
        fill_value=-1,
        dtype=int
    )

    validation_count = np.zeros(
        len(y),
        dtype=int
    )

    fold_results = []

    poor_label = int(
        le.transform(["Poor"])[0]
    )

    model_template = (
        create_baseline_models()[
            "LGBM"
        ]
    )


    # --------------------------------------------------------
    # 4. Fixed five-fold evaluation
    # --------------------------------------------------------

    for fold, (
        train_idx,
        val_idx
    ) in enumerate(
        cv_splits,
        start=1
    ):

        X_train = X_selected.iloc[
            train_idx
        ].copy()

        X_val = X_selected.iloc[
            val_idx
        ].copy()

        y_train = y[
            train_idx
        ]

        y_val = y[
            val_idx
        ]


        # ----------------------------------------------------
        # 5. Fold-specific class weights
        # ----------------------------------------------------

        weight_dict = (
            calculate_class_weight_dict(
                y_train
            )
        )

        sample_weight = (
            create_sample_weights(
                y_train,
                weight_dict
            )
        )


        # ----------------------------------------------------
        # 6. Fresh LightGBM model
        # ----------------------------------------------------

        model = clone(
            model_template
        )

        model.fit(
            X_train,
            y_train,
            sample_weight=sample_weight
        )

        y_pred = model.predict(
            X_val
        ).astype(int)

        oof_pred[
            val_idx
        ] = y_pred

        validation_count[
            val_idx
        ] += 1


        # ----------------------------------------------------
        # 7. Fold metrics
        # ----------------------------------------------------

        fold_result = {
            "Network": network_name,
            "Experiment": experiment_name,
            "Fold": fold,

            "Removed Features":
                len(removed_features),

            "Number of Features":
                len(selected_features),

            "Validation Samples":
                len(val_idx),

            "Poor Support":
                int(
                    np.sum(
                        y_val == poor_label
                    )
                ),

            "Accuracy": accuracy_score(
                y_val,
                y_pred
            ),

            "Balanced Accuracy":
                balanced_accuracy_score(
                    y_val,
                    y_pred
                ),

            "Macro-F1": f1_score(
                y_val,
                y_pred,
                average="macro",
                zero_division=0
            ),

            "MCC": matthews_corrcoef(
                y_val,
                y_pred
            ),

            "Poor Precision": precision_score(
                y_val,
                y_pred,
                labels=[poor_label],
                average=None,
                zero_division=0
            )[0],

            "Poor Recall": recall_score(
                y_val,
                y_pred,
                labels=[poor_label],
                average=None,
                zero_division=0
            )[0],

            "Poor F1": f1_score(
                y_val,
                y_pred,
                labels=[poor_label],
                average=None,
                zero_division=0
            )[0]
        }

        fold_results.append(
            fold_result
        )


    # --------------------------------------------------------
    # 8. Validate OOF assignments
    # --------------------------------------------------------

    if np.any(
        oof_pred == -1
    ):
        raise RuntimeError(
            f"{network_name} | {experiment_name}: "
            "missing OOF predictions."
        )

    if not np.all(
        validation_count == 1
    ):
        raise RuntimeError(
            f"{network_name} | {experiment_name}: "
            "each observation must be validated exactly once."
        )


    # --------------------------------------------------------
    # 9. Overall OOF metrics
    # --------------------------------------------------------

    fold_df = pd.DataFrame(
        fold_results
    )

    overall = {
        "Network": network_name,
        "Experiment": experiment_name,

        "Method": "Class Weight",
        "Model": "LGBM",

        "Removed Features":
            len(removed_features),

        "Number of Features":
            len(selected_features),

        "Removed Feature Names":
            ", ".join(
                removed_features
            ),

        "Accuracy": accuracy_score(
            y,
            oof_pred
        ),

        "Balanced Accuracy":
            balanced_accuracy_score(
                y,
                oof_pred
            ),

        "Macro-F1": f1_score(
            y,
            oof_pred,
            average="macro",
            zero_division=0
        ),

        "MCC": matthews_corrcoef(
            y,
            oof_pred
        ),

        "Poor Precision": precision_score(
            y,
            oof_pred,
            labels=[poor_label],
            average=None,
            zero_division=0
        )[0],

        "Poor Recall": recall_score(
            y,
            oof_pred,
            labels=[poor_label],
            average=None,
            zero_division=0
        )[0],

        "Poor F1": f1_score(
            y,
            oof_pred,
            labels=[poor_label],
            average=None,
            zero_division=0
        )[0],

        "Poor Support": int(
            np.sum(
                y == poor_label
            )
        ),

        "CV Macro-F1 Mean":
            fold_df[
                "Macro-F1"
            ].mean(),

        "CV Macro-F1 SD":
            fold_df[
                "Macro-F1"
            ].std(ddof=0),

        "CV Poor F1 Mean":
            fold_df[
                "Poor F1"
            ].mean(),

        "CV Poor F1 SD":
            fold_df[
                "Poor F1"
            ].std(ddof=0)
    }

    return (
        overall,
        fold_df,
        oof_pred
    )

In [ ]:
# ============================================================
# Phase 5 - Cell 4.
# Run All Dependency-Aware Ablation Experiments
# ============================================================

def run_all_ablation_experiments(
    X,
    y,
    le,
    network_name,
    cv_splits
):

    if len(
        cv_splits
    ) != N_SPLITS:
        raise ValueError(
            f"Expected {N_SPLITS} CV splits, "
            f"but found {len(cv_splits)}."
        )

    overall_results = []
    fold_results = {}
    predictions = {}


    for (
        experiment_name,
        removed_features
    ) in ABLATION_EXPERIMENTS.items():

        print("\n" + "=" * 75)

        print(
            f"{network_name} | "
            f"{experiment_name}"
        )

        print("=" * 75)

        (
            overall,
            fold_df,
            oof_pred
        ) = evaluate_ablation(
            X=X,
            y=y,
            le=le,
            network_name=network_name,
            experiment_name=experiment_name,
            removed_features=removed_features,
            cv_splits=cv_splits
        )

        overall_results.append(
            overall
        )

        fold_results[
            experiment_name
        ] = fold_df

        predictions[
            experiment_name
        ] = oof_pred.copy()


        print(
            "Removed features:",
            overall["Removed Features"]
        )

        print(
            "Retained features:",
            overall["Number of Features"]
        )

        print(
            "OOF Macro-F1:",
            f"{overall['Macro-F1']:.4f}"
        )

        print(
            "CV Macro-F1:",
            (
                f"{overall['CV Macro-F1 Mean']:.4f} "
                f"± {overall['CV Macro-F1 SD']:.4f}"
            )
        )

        print(
            "OOF Poor F1:",
            f"{overall['Poor F1']:.4f}"
        )

        print(
            "CV Poor F1:",
            (
                f"{overall['CV Poor F1 Mean']:.4f} "
                f"± {overall['CV Poor F1 SD']:.4f}"
            )
        )


    overall_results_df = pd.DataFrame(
        overall_results
    )

    expected_experiments = len(
        ABLATION_EXPERIMENTS
    )

    if len(
        overall_results_df
    ) != expected_experiments:
        raise RuntimeError(
            f"Expected {expected_experiments} ablation results, "
            f"but found {len(overall_results_df)}."
        )

    if set(
        overall_results_df["Experiment"]
    ) != set(
        ABLATION_EXPERIMENTS.keys()
    ):
        raise RuntimeError(
            "Ablation experiment names do not match "
            "the defined experiments."
        )

    return (
        overall_results_df,
        fold_results,
        predictions
    )

In [ ]:
# ============================================================
# Phase 5 - Cell 5.
# LTE Ablation Study
# ============================================================

# ------------------------------------------------------------
# 1. Create fixed LTE folds
# Same folds for every ablation experiment
# ------------------------------------------------------------

lte_ablation_cv = StratifiedKFold(
    n_splits=N_SPLITS,
    shuffle=True,
    random_state=RANDOM_STATE
)

lte_ablation_splits = list(
    lte_ablation_cv.split(
        X_lte,
        y_lte
    )
)

# ------------------------------------------------------------
# 2. Run LTE ablation experiments
# ------------------------------------------------------------

(
    lte_ablation_results,
    lte_ablation_folds,
    lte_ablation_predictions
) = run_all_ablation_experiments(
    X=X_lte,
    y=y_lte,
    le=le_lte,
    network_name="LTE",
    cv_splits=lte_ablation_splits
)

# ------------------------------------------------------------
# 3. Display results
# ------------------------------------------------------------

lte_ablation_display = lte_ablation_results[
    [
        "Experiment",
        "Number of Features",
        "Accuracy",
        "Balanced Accuracy",
        "Macro-F1",
        "CV Macro-F1 Mean",
        "CV Macro-F1 SD",
        "MCC",
        "Poor Precision",
        "Poor Recall",
        "Poor F1",
        "CV Poor F1 Mean",
        "CV Poor F1 SD"
    ]
].copy()

display(
    lte_ablation_display.round(4)
)

# ------------------------------------------------------------
# 4. Full-model reproducibility check
# ------------------------------------------------------------

lte_full_ablation = (
    lte_ablation_results[
        lte_ablation_results["Experiment"] == "Full Model"
    ]
    .iloc[0]
)

lte_phase2_reference = (
    imbalance_results[
        (imbalance_results["Network"] == "LTE")
        & (imbalance_results["Method"] == "Class Weight")
        & (imbalance_results["Model"] == "LGBM")
    ]
    .iloc[0]
)

macro_difference = (
    lte_full_ablation["CV Macro-F1 Mean"]
    - lte_phase2_reference["CV Macro-F1 Mean"]
)

print("\nLTE full-model reproducibility check")
print(
    "Ablation CV Macro-F1:",
    round(lte_full_ablation["CV Macro-F1 Mean"], 4)
)
print(
    "Phase 2 CV Macro-F1 :",
    round(lte_phase2_reference["CV Macro-F1 Mean"], 4)
)
print(
    "Difference          :",
    round(macro_difference, 8)
)

assert np.isclose(
    lte_full_ablation["CV Macro-F1 Mean"],
    lte_phase2_reference["CV Macro-F1 Mean"],
    atol=1e-10
), "LTE full model does not reproduce the Phase 2 result."

print("✓ LTE full model reproduces the Phase 2 result.")

In [ ]:
# ============================================================
# Phase 5 - Cell 7.
# 5G NR Ablation Study
# ============================================================

# ------------------------------------------------------------
# 1. Create fixed 5G NR folds
# Same folds for every ablation experiment
# ------------------------------------------------------------

nr_ablation_cv = StratifiedKFold(
    n_splits=N_SPLITS,
    shuffle=True,
    random_state=RANDOM_STATE
)

nr_ablation_splits = list(
    nr_ablation_cv.split(
        X_nr,
        y_nr
    )
)

# ------------------------------------------------------------
# 2. Run 5G NR ablation experiments
# ------------------------------------------------------------

(
    nr_ablation_results,
    nr_ablation_folds,
    nr_ablation_predictions
) = run_all_ablation_experiments(
    X=X_nr,
    y=y_nr,
    le=le_nr,
    network_name="5G NR",
    cv_splits=nr_ablation_splits
)

# ------------------------------------------------------------
# 3. Display results
# ------------------------------------------------------------

nr_ablation_display = nr_ablation_results[
    [
        "Experiment",
        "Number of Features",
        "Accuracy",
        "Balanced Accuracy",
        "Macro-F1",
        "CV Macro-F1 Mean",
        "CV Macro-F1 SD",
        "MCC",
        "Poor Precision",
        "Poor Recall",
        "Poor F1",
        "CV Poor F1 Mean",
        "CV Poor F1 SD"
    ]
].copy()

display(
    nr_ablation_display.round(4)
)

# ------------------------------------------------------------
# 4. Full-model reproducibility check
# ------------------------------------------------------------

nr_full_ablation = (
    nr_ablation_results[
        nr_ablation_results["Experiment"] == "Full Model"
    ]
    .iloc[0]
)

nr_phase2_reference = (
    imbalance_results[
        (imbalance_results["Network"] == "5G NR")
        & (imbalance_results["Method"] == "Class Weight")
        & (imbalance_results["Model"] == "LGBM")
    ]
    .iloc[0]
)

macro_difference = (
    nr_full_ablation["CV Macro-F1 Mean"]
    - nr_phase2_reference["CV Macro-F1 Mean"]
)

print("\n5G NR full-model reproducibility check")
print(
    "Ablation CV Macro-F1:",
    round(nr_full_ablation["CV Macro-F1 Mean"], 4)
)
print(
    "Phase 2 CV Macro-F1 :",
    round(nr_phase2_reference["CV Macro-F1 Mean"], 4)
)
print(
    "Difference          :",
    round(macro_difference, 8)
)

assert np.isclose(
    nr_full_ablation["CV Macro-F1 Mean"],
    nr_phase2_reference["CV Macro-F1 Mean"],
    atol=1e-10
), "5G NR full model does not reproduce the Phase 2 result."

print("✓ 5G NR full model reproduces the Phase 2 result.")

In [ ]:
# ============================================================
# Phase 5 - Cell 8.
# Performance Changes Relative to the Full Model
# ============================================================

def calculate_ablation_deltas(results_df):

    results = results_df.copy()

    # --------------------------------------------------------
    # 1. Locate full-model reference
    # --------------------------------------------------------

    full_rows = results[
        results["Experiment"] == "Full Model"
    ]

    if len(full_rows) != 1:
        raise ValueError(
            "Exactly one Full Model row must be present."
        )

    full = full_rows.iloc[0]

    # --------------------------------------------------------
    # 2. Number of removed predictors
    # --------------------------------------------------------

    full_feature_count = int(
        full["Number of Features"]
    )

    results["Removed Features"] = (
        full_feature_count
        - results["Number of Features"]
    )

    # --------------------------------------------------------
    # 3. Primary changes: mean cross-validated metrics
    # Negative value = performance decreased after removal
    # --------------------------------------------------------

    results["Delta CV Macro-F1"] = (
        results["CV Macro-F1 Mean"]
        - full["CV Macro-F1 Mean"]
    )

    results["Delta CV Poor F1"] = (
        results["CV Poor F1 Mean"]
        - full["CV Poor F1 Mean"]
    )

    # --------------------------------------------------------
    # 4. Secondary changes: pooled OOF metrics
    # --------------------------------------------------------

    results["Delta OOF Macro-F1"] = (
        results["Macro-F1"]
        - full["Macro-F1"]
    )

    results["Delta OOF Poor F1"] = (
        results["Poor F1"]
        - full["Poor F1"]
    )

    # --------------------------------------------------------
    # 5. Positive contribution score
    # Positive value = performance lost when group was removed
    # --------------------------------------------------------

    results["Macro-F1 Contribution"] = (
        -results["Delta CV Macro-F1"]
    )

    results["Poor F1 Contribution"] = (
        -results["Delta CV Poor F1"]
    )

    return results


# ------------------------------------------------------------
# Calculate LTE and 5G NR summaries
# ------------------------------------------------------------

lte_ablation_summary = calculate_ablation_deltas(
    lte_ablation_results
)

nr_ablation_summary = calculate_ablation_deltas(
    nr_ablation_results
)


# ------------------------------------------------------------
# Columns to display
# ------------------------------------------------------------

summary_cols = [
    "Experiment",
    "Removed Features",
    "Number of Features",
    "CV Macro-F1 Mean",
    "CV Macro-F1 SD",
    "Delta CV Macro-F1",
    "Macro-F1 Contribution",
    "CV Poor F1 Mean",
    "CV Poor F1 SD",
    "Delta CV Poor F1",
    "Poor F1 Contribution"
]


print("=" * 75)
print("LTE ABLATION SUMMARY")
print("=" * 75)

display(
    lte_ablation_summary[
        summary_cols
    ].round(4)
)


print("\n" + "=" * 75)
print("5G NR ABLATION SUMMARY")
print("=" * 75)

display(
    nr_ablation_summary[
        summary_cols
    ].round(4)
)

In [ ]:
# ============================================================
# Phase 5 - Cell 9.
# Paired Fold-Level Ablation Analysis
# ============================================================

def create_paired_ablation_summary(fold_results):

    if "Full Model" not in fold_results:
        raise KeyError(
            "Full Model fold results were not found."
        )

    full_df = (
        fold_results["Full Model"]
        .sort_values("Fold")
        .reset_index(drop=True)
    )

    rows = []

    for experiment, exp_df in fold_results.items():

        if experiment == "Full Model":
            continue

        exp_df = (
            exp_df
            .sort_values("Fold")
            .reset_index(drop=True)
        )

        # ----------------------------------------------------
        # Ensure folds are correctly paired
        # ----------------------------------------------------

        if not np.array_equal(
            exp_df["Fold"].to_numpy(),
            full_df["Fold"].to_numpy()
        ):
            raise ValueError(
                f"Fold mismatch detected: {experiment}"
            )

        macro_delta = (
            exp_df["Macro-F1"].to_numpy()
            - full_df["Macro-F1"].to_numpy()
        )

        poor_delta = (
            exp_df["Poor F1"].to_numpy()
            - full_df["Poor F1"].to_numpy()
        )

        rows.append({
            "Experiment":
                experiment,

            "Mean Fold Delta Macro-F1":
                macro_delta.mean(),

            "SD Fold Delta Macro-F1":
                macro_delta.std(ddof=0),

            "Macro-F1 Worse Folds":
                int((macro_delta < 0).sum()),

            "Macro-F1 Equal Folds":
                int(np.isclose(macro_delta, 0).sum()),

            "Macro-F1 Better Folds":
                int((macro_delta > 0).sum()),

            "Mean Fold Delta Poor F1":
                poor_delta.mean(),

            "SD Fold Delta Poor F1":
                poor_delta.std(ddof=0),

            "Poor F1 Worse Folds":
                int((poor_delta < 0).sum()),

            "Poor F1 Equal Folds":
                int(np.isclose(poor_delta, 0).sum()),

            "Poor F1 Better Folds":
                int((poor_delta > 0).sum())
        })

    return (
        pd.DataFrame(rows)
        .sort_values(
            "Mean Fold Delta Macro-F1"
        )
        .reset_index(drop=True)
    )


# ------------------------------------------------------------
# LTE and 5G NR
# ------------------------------------------------------------

lte_ablation_paired = create_paired_ablation_summary(
    lte_ablation_folds
)

nr_ablation_paired = create_paired_ablation_summary(
    nr_ablation_folds
)


print("=" * 75)
print("LTE PAIRED FOLD-LEVEL ABLATION")
print("=" * 75)

display(
    lte_ablation_paired.round(4)
)


print("\n" + "=" * 75)
print("5G NR PAIRED FOLD-LEVEL ABLATION")
print("=" * 75)

display(
    nr_ablation_paired.round(4)
)

## Phase 6: Hierarchical Classification

A two-stage hierarchical classifier is evaluated as an alternative to flat three-class classification.

- Stage 1: Excellent versus Good-or-Poor
- Stage 2: Good versus Poor
- Final output: Excellent, Good, or Poor

Both stages use class-weighted LightGBM. The hierarchical results are compared with an equivalent flat three-class model using the same cross-validation folds.

### ※ Related dissertation sections

- Section 2.9 (Alternative Classification Structures)
- Section 4.10 (Hierarchical Classification)
- Table 4.13 (Flat and Hierarchical Classification Performance)
- Figure 4.17 (Two-Stage Hierarchical Classification Design)
- Section 5.6 (Hierarchical Classification)

In [ ]:
# ============================================================
# Phase 6 - Cell 1.
# Hierarchical Classification Design
# ============================================================

HIERARCHICAL_STAGE_1 = {
    "Excellent": 0,
    "Good_or_Poor": 1
}

HIERARCHICAL_STAGE_2 = {
    "Good": 0,
    "Poor": 1
}

FINAL_CLASS_ORDER = [
    "Excellent",
    "Good",
    "Poor"
]

print("=" * 70)
print("HIERARCHICAL THREE-CLASS CLASSIFICATION")
print("=" * 70)

print(
    "Stage 1: Excellent (0) "
    "vs Good or Poor (1)"
)

print(
    "Stage 2: Good (0) "
    "vs Poor (1)"
)

print(
    "Final output:",
    " / ".join(FINAL_CLASS_ORDER)
)

print("\nDecision process:")
print("Stage 1 = Excellent  → Final class: Excellent")
print("Stage 1 = Good/Poor  → Apply Stage 2")
print("Stage 2 = Good       → Final class: Good")
print("Stage 2 = Poor       → Final class: Poor")

print("\nImportant:")
print(
    "Only observations predicted as Good/Poor by Stage 1 "
    "are passed to Stage 2 during validation."
)

In [ ]:
# ============================================================
# Phase 6 - Cell 2.
# Create and Validate Hierarchical Targets
# ============================================================

def create_hierarchical_targets(
    labelled_df,
    target_col,
    X
):

    # --------------------------------------------------------
    # 1. Validate number of observations
    # --------------------------------------------------------

    if len(labelled_df) != len(X):
        raise ValueError(
            f"Row mismatch: labelled_df={len(labelled_df)}, "
            f"X={len(X)}"
        )

    # Pandas index labels do not need to match.
    # All subsequent operations use positional NumPy indices.

    # --------------------------------------------------------
    # 2. Clean and validate original labels
    # --------------------------------------------------------

    y_text = (
        labelled_df[target_col]
        .astype(str)
        .str.strip()
        .to_numpy()
    )

    valid_classes = {
        "Excellent",
        "Good",
        "Poor"
    }

    detected_classes = set(
        np.unique(y_text)
    )

    if detected_classes != valid_classes:
        raise ValueError(
            f"Unexpected classes in {target_col}: "
            f"{sorted(detected_classes)}"
        )

    # --------------------------------------------------------
    # 3. Stage 1
    # Excellent = 0
    # Good or Poor = 1
    # --------------------------------------------------------

    y_stage1 = np.where(
        y_text == "Excellent",
        0,
        1
    ).astype(int)

    # --------------------------------------------------------
    # 4. Stage 2
    # Good = 0
    # Poor = 1
    # --------------------------------------------------------

    stage2_mask = np.isin(
        y_text,
        ["Good", "Poor"]
    )

    y_stage2 = np.where(
        y_text[stage2_mask] == "Poor",
        1,
        0
    ).astype(int)

    # --------------------------------------------------------
    # 5. Validation
    # --------------------------------------------------------

    if stage2_mask.sum() != len(y_stage2):
        raise ValueError(
            "Stage 2 target construction failed."
        )

    if set(np.unique(y_stage1)) != {0, 1}:
        raise ValueError(
            "Stage 1 must contain both binary classes."
        )

    if set(np.unique(y_stage2)) != {0, 1}:
        raise ValueError(
            "Stage 2 must contain both binary classes."
        )

    # --------------------------------------------------------
    # 6. Summary
    # --------------------------------------------------------

    print("=" * 70)
    print(target_col)
    print("=" * 70)

    print("\nTotal observations:", len(y_text))

    print("\nStage 1 target:")
    print(
        "Excellent (0)   :",
        int(np.sum(y_stage1 == 0))
    )
    print(
        "Good or Poor (1):",
        int(np.sum(y_stage1 == 1))
    )

    print("\nStage 2 training population:")
    print(
        "Good (0):",
        int(np.sum(y_stage2 == 0))
    )
    print(
        "Poor (1):",
        int(np.sum(y_stage2 == 1))
    )
    print(
        "Total   :",
        len(y_stage2)
    )

    print("\n✓ Hierarchical targets validated.")

    return (
        y_text,
        y_stage1,
        stage2_mask,
        y_stage2
    )

In [ ]:
# ============================================================
# Phase 6 - Cell 3.
# Generate and Check Hierarchical Targets
# ============================================================

(
    y_lte_text,
    y_lte_stage1,
    lte_stage2_mask,
    y_lte_stage2
) = create_hierarchical_targets(
    labelled_df=lte_df,
    target_col="lte_signal_class",
    X=X_lte
)

print("\n")

(
    y_nr_text,
    y_nr_stage1,
    nr_stage2_mask,
    y_nr_stage2
) = create_hierarchical_targets(
    labelled_df=nr_df,
    target_col="nr_signal_class",
    X=X_nr
)

In [ ]:
# ============================================================
# Phase 6 - Required Imports and Helper
# ============================================================

from sklearn.impute import SimpleImputer
from sklearn.utils.class_weight import compute_class_weight
from lightgbm import LGBMClassifier

def calculate_sample_weights(y_train):

    classes = np.unique(y_train)

    class_weights = compute_class_weight(
        class_weight="balanced",
        classes=classes,
        y=y_train
    )

    weight_dict = dict(zip(classes, class_weights))

    sample_weights = np.array([
        weight_dict[label]
        for label in y_train
    ])

    return weight_dict, sample_weights

In [ ]:
# ============================================================
# Phase 6 - Cell 4.
# Five-Fold OOF Hierarchical Classification
# Model: Class Weight + LightGBM
# ============================================================

def run_hierarchical_oof(
    X,
    y_text,
    network_name,
    cv_splits
):

    y_text = np.asarray(
        y_text,
        dtype=object
    )

    class_order = [
        "Excellent",
        "Good",
        "Poor"
    ]

    # --------------------------------------------------------
    # Basic validation
    # --------------------------------------------------------

    if len(X) != len(y_text):
        raise ValueError(
            "X and y_text have different row counts."
        )

    if set(np.unique(y_text)) != set(class_order):
        raise ValueError(
            f"Unexpected classes: {np.unique(y_text)}"
        )

    if len(cv_splits) != N_SPLITS:
        raise ValueError(
            f"Expected {N_SPLITS} folds, "
            f"but received {len(cv_splits)}."
        )

    # --------------------------------------------------------
    # OOF storage
    # --------------------------------------------------------

    oof_final = np.full(
        len(y_text),
        None,
        dtype=object
    )

    oof_stage1 = np.full(
        len(y_text),
        -1,
        dtype=int
    )

    validation_count = np.zeros(
        len(y_text),
        dtype=int
    )

    fold_results = []

    # --------------------------------------------------------
    # Fixed folds
    # --------------------------------------------------------

    for fold, (train_idx, val_idx) in enumerate(
        cv_splits,
        start=1
    ):

        X_train = X.iloc[
            train_idx
        ].copy()

        X_val = X.iloc[
            val_idx
        ].copy()

        y_train_text = y_text[
            train_idx
        ]

        y_val_text = y_text[
            val_idx
        ]

        validation_count[
            val_idx
        ] += 1

        # ====================================================
        # Median imputation
        # Fit on training fold only
        # ====================================================

        imputer = SimpleImputer(
            strategy="median"
        )

        X_train_imp = pd.DataFrame(
            imputer.fit_transform(X_train),
            columns=X.columns,
            index=X_train.index
        )

        X_val_imp = pd.DataFrame(
            imputer.transform(X_val),
            columns=X.columns,
            index=X_val.index
        )

        # ====================================================
        # Stage 1
        # Excellent = 0
        # Good or Poor = 1
        # ====================================================

        y_train_stage1 = np.where(
            y_train_text == "Excellent",
            0,
            1
        ).astype(int)

        if set(np.unique(y_train_stage1)) != {0, 1}:
            raise ValueError(
                f"{network_name}, Fold {fold}: "
                "Stage 1 training data does not contain both classes."
            )

        _, stage1_weights = calculate_sample_weights(
            y_train_stage1
        )

        stage1_model = LGBMClassifier(
            n_estimators=100,
            max_depth=10,
            learning_rate=0.1,
            random_state=RANDOM_STATE,
            verbose=-1,
            n_jobs=-1
        )

        stage1_model.fit(
            X_train_imp,
            y_train_stage1,
            sample_weight=stage1_weights
        )

        stage1_pred = (
            stage1_model
            .predict(X_val_imp)
            .astype(int)
        )

        oof_stage1[
            val_idx
        ] = stage1_pred

        # ====================================================
        # Stage 2 training
        # Use true Good/Poor observations from TRAINING only
        #
        # Good = 0
        # Poor = 1
        # ====================================================

        train_stage2_mask = np.isin(
            y_train_text,
            ["Good", "Poor"]
        )

        X_train_stage2 = X_train_imp.iloc[
            np.flatnonzero(train_stage2_mask)
        ].copy()

        y_train_stage2 = np.where(
            y_train_text[train_stage2_mask] == "Poor",
            1,
            0
        ).astype(int)

        if set(np.unique(y_train_stage2)) != {0, 1}:
            raise ValueError(
                f"{network_name}, Fold {fold}: "
                "Stage 2 training data does not contain both classes."
            )

        _, stage2_weights = calculate_sample_weights(
            y_train_stage2
        )

        stage2_model = LGBMClassifier(
            n_estimators=100,
            max_depth=10,
            learning_rate=0.1,
            random_state=RANDOM_STATE,
            verbose=-1,
            n_jobs=-1
        )

        stage2_model.fit(
            X_train_stage2,
            y_train_stage2,
            sample_weight=stage2_weights
        )

        # ====================================================
        # Final routing
        # Validation routing is based only on Stage 1 prediction
        # ====================================================

        final_pred = np.full(
            len(val_idx),
            "Excellent",
            dtype=object
        )

        routed_mask = (
            stage1_pred == 1
        )

        if np.any(routed_mask):

            routed_positions = np.flatnonzero(
                routed_mask
            )

            X_val_stage2 = X_val_imp.iloc[
                routed_positions
            ]

            stage2_pred = (
                stage2_model
                .predict(X_val_stage2)
                .astype(int)
            )

            final_pred[
                routed_positions
            ] = np.where(
                stage2_pred == 1,
                "Poor",
                "Good"
            )

        oof_final[
            val_idx
        ] = final_pred

        # ====================================================
        # Stage 1 routing performance
        # ====================================================

        y_val_stage1 = np.where(
            y_val_text == "Excellent",
            0,
            1
        ).astype(int)

        routing_precision = precision_score(
            y_val_stage1,
            stage1_pred,
            pos_label=1,
            zero_division=0
        )

        routing_recall = recall_score(
            y_val_stage1,
            stage1_pred,
            pos_label=1,
            zero_division=0
        )

        routing_f1 = f1_score(
            y_val_stage1,
            stage1_pred,
            pos_label=1,
            zero_division=0
        )

        # Percentage of true Poor samples passed to Stage 2
        poor_mask = (
            y_val_text == "Poor"
        )

        poor_support = int(
            poor_mask.sum()
        )

        if poor_support > 0:
            poor_routing_recall = float(
                np.mean(
                    stage1_pred[poor_mask] == 1
                )
            )
        else:
            poor_routing_recall = np.nan

        # ====================================================
        # Final three-class fold metrics
        # ====================================================

        fold_result = {
            "Network":
                network_name,

            "Fold":
                fold,

            "Training Samples":
                len(train_idx),

            "Validation Samples":
                len(val_idx),

            "Routed to Stage 2":
                int(routed_mask.sum()),

            "Stage1 Routing Precision":
                routing_precision,

            "Stage1 Routing Recall":
                routing_recall,

            "Stage1 Routing F1":
                routing_f1,

            "Stage1 Poor Routing Recall":
                poor_routing_recall,

            "Accuracy":
                accuracy_score(
                    y_val_text,
                    final_pred
                ),

            "Balanced Accuracy":
                balanced_accuracy_score(
                    y_val_text,
                    final_pred
                ),

            "Macro-F1":
                f1_score(
                    y_val_text,
                    final_pred,
                    labels=class_order,
                    average="macro",
                    zero_division=0
                ),

            "MCC":
                matthews_corrcoef(
                    y_val_text,
                    final_pred
                ),

            "Poor Precision":
                precision_score(
                    y_val_text,
                    final_pred,
                    labels=["Poor"],
                    average=None,
                    zero_division=0
                )[0],

            "Poor Recall":
                recall_score(
                    y_val_text,
                    final_pred,
                    labels=["Poor"],
                    average=None,
                    zero_division=0
                )[0],

            "Poor F1":
                f1_score(
                    y_val_text,
                    final_pred,
                    labels=["Poor"],
                    average=None,
                    zero_division=0
                )[0],

            "Poor Support":
                poor_support
        }

        fold_results.append(
            fold_result
        )

        print(
            f"{network_name} | Fold {fold}: "
            f"Macro-F1 = {fold_result['Macro-F1']:.4f}, "
            f"Poor F1 = {fold_result['Poor F1']:.4f}, "
            f"Poor routing recall = "
            f"{fold_result['Stage1 Poor Routing Recall']:.4f}"
        )

    # --------------------------------------------------------
    # OOF integrity validation
    # --------------------------------------------------------

    if not np.all(validation_count == 1):
        raise ValueError(
            "Every observation must be validated exactly once."
        )

    if np.any(oof_stage1 == -1):
        raise ValueError(
            "Missing Stage 1 OOF predictions detected."
        )

    if any(value is None for value in oof_final):
        raise ValueError(
            "Missing final OOF predictions detected."
        )

    if not set(np.unique(oof_final)).issubset(
        set(class_order)
    ):
        raise ValueError(
            "Unexpected final prediction labels detected."
        )

    # --------------------------------------------------------
    # Fold-level results
    # --------------------------------------------------------

    fold_df = pd.DataFrame(
        fold_results
    )

    # --------------------------------------------------------
    # Overall pooled OOF metrics
    # --------------------------------------------------------

    overall = {
        "Network":
            network_name,

        "Method":
            "Hierarchical",

        "Model":
            "Class Weight + LGBM",

        "N Rows":
            len(y_text),

        "N Features":
            X.shape[1],

        "Accuracy":
            accuracy_score(
                y_text,
                oof_final
            ),

        "Balanced Accuracy":
            balanced_accuracy_score(
                y_text,
                oof_final
            ),

        "Macro-F1":
            f1_score(
                y_text,
                oof_final,
                labels=class_order,
                average="macro",
                zero_division=0
            ),

        "MCC":
            matthews_corrcoef(
                y_text,
                oof_final
            ),

        "Poor Precision":
            precision_score(
                y_text,
                oof_final,
                labels=["Poor"],
                average=None,
                zero_division=0
            )[0],

        "Poor Recall":
            recall_score(
                y_text,
                oof_final,
                labels=["Poor"],
                average=None,
                zero_division=0
            )[0],

        "Poor F1":
            f1_score(
                y_text,
                oof_final,
                labels=["Poor"],
                average=None,
                zero_division=0
            )[0],

        "Poor Support":
            int(
                np.sum(y_text == "Poor")
            ),

        "Mean Stage1 Routing Precision":
            fold_df[
                "Stage1 Routing Precision"
            ].mean(),

        "Mean Stage1 Routing Recall":
            fold_df[
                "Stage1 Routing Recall"
            ].mean(),

        "Mean Stage1 Routing F1":
            fold_df[
                "Stage1 Routing F1"
            ].mean(),

        "Mean Stage1 Poor Routing Recall":
            fold_df[
                "Stage1 Poor Routing Recall"
            ].mean(),

        "CV Macro-F1 Mean":
            fold_df["Macro-F1"].mean(),

        "CV Macro-F1 SD":
            fold_df["Macro-F1"].std(
                ddof=0
            ),

        "CV Poor F1 Mean":
            fold_df["Poor F1"].mean(),

        "CV Poor F1 SD":
            fold_df["Poor F1"].std(
                ddof=0
            )
    }

    return (
        overall,
        fold_df,
        oof_final,
        oof_stage1
    )

In [ ]:
# ============================================================
# Phase 6 - Cell 5.
# LTE Hierarchical Classification
# ============================================================

(
    lte_hierarchical_overall,
    lte_hierarchical_folds,
    lte_hierarchical_pred,
    lte_stage1_oof
) = run_hierarchical_oof(
    X=X_lte,
    y_text=y_lte_text,
    network_name="LTE",
    cv_splits=lte_ablation_splits
)

print("=" * 75)
print("LTE HIERARCHICAL FOLD RESULTS")
print("=" * 75)

display(
    lte_hierarchical_folds[
        [
            "Fold",
            "Training Samples",
            "Validation Samples",
            "Routed to Stage 2",
            "Stage1 Routing Precision",
            "Stage1 Routing Recall",
            "Stage1 Routing F1",
            "Stage1 Poor Routing Recall",
            "Accuracy",
            "Balanced Accuracy",
            "Macro-F1",
            "MCC",
            "Poor Precision",
            "Poor Recall",
            "Poor F1",
            "Poor Support"
        ]
    ].round(4)
)

print("\n" + "=" * 75)
print("LTE HIERARCHICAL OVERALL RESULTS")
print("=" * 75)

display(
    pd.DataFrame(
        [lte_hierarchical_overall]
    )[
        [
            "Network",
            "Method",
            "Model",
            "N Rows",
            "N Features",
            "Accuracy",
            "Balanced Accuracy",
            "Macro-F1",
            "CV Macro-F1 Mean",
            "CV Macro-F1 SD",
            "MCC",
            "Poor Precision",
            "Poor Recall",
            "Poor F1",
            "CV Poor F1 Mean",
            "CV Poor F1 SD",
            "Mean Stage1 Routing Precision",
            "Mean Stage1 Routing Recall",
            "Mean Stage1 Routing F1",
            "Mean Stage1 Poor Routing Recall"
        ]
    ].round(4)
)

In [ ]:
# ============================================================
# Phase 6 - Cell 6.
# 5G NR Hierarchical Classification
# ============================================================

(
    nr_hierarchical_overall,
    nr_hierarchical_folds,
    nr_hierarchical_pred,
    nr_stage1_oof
) = run_hierarchical_oof(
    X=X_nr,
    y_text=y_nr_text,
    network_name="5G NR",
    cv_splits=nr_ablation_splits
)

print("=" * 75)
print("5G NR HIERARCHICAL FOLD RESULTS")
print("=" * 75)

display(
    nr_hierarchical_folds[
        [
            "Fold",
            "Training Samples",
            "Validation Samples",
            "Routed to Stage 2",
            "Stage1 Routing Precision",
            "Stage1 Routing Recall",
            "Stage1 Routing F1",
            "Stage1 Poor Routing Recall",
            "Accuracy",
            "Balanced Accuracy",
            "Macro-F1",
            "MCC",
            "Poor Precision",
            "Poor Recall",
            "Poor F1",
            "Poor Support"
        ]
    ].round(4)
)

print("\n" + "=" * 75)
print("5G NR HIERARCHICAL OVERALL RESULTS")
print("=" * 75)

display(
    pd.DataFrame(
        [nr_hierarchical_overall]
    )[
        [
            "Network",
            "Method",
            "Model",
            "N Rows",
            "N Features",
            "Accuracy",
            "Balanced Accuracy",
            "Macro-F1",
            "CV Macro-F1 Mean",
            "CV Macro-F1 SD",
            "MCC",
            "Poor Precision",
            "Poor Recall",
            "Poor F1",
            "CV Poor F1 Mean",
            "CV Poor F1 SD",
            "Mean Stage1 Routing Precision",
            "Mean Stage1 Routing Recall",
            "Mean Stage1 Routing F1",
            "Mean Stage1 Poor Routing Recall"
        ]
    ].round(4)
)

In [ ]:
# ============================================================
# Phase 6 - Cell 7.
# Flat vs Hierarchical Classification
# Same Model: Class Weight + LightGBM
# ============================================================

comparison_rows = []

for network, hierarchical in [
    ("LTE", lte_hierarchical_overall),
    ("5G NR", nr_hierarchical_overall)
]:

    flat_rows = imbalance_results[
        (imbalance_results["Network"] == network)
        & (imbalance_results["Method"] == "Class Weight")
        & (imbalance_results["Model"] == "LGBM")
    ]

    if len(flat_rows) != 1:
        raise ValueError(
            f"Expected one flat-model result for {network}, "
            f"but found {len(flat_rows)}."
        )

    flat = flat_rows.iloc[0]

    comparison_rows.append({
        "Network":
            network,

        "Flat Accuracy":
            flat["Accuracy"],

        "Hierarchical Accuracy":
            hierarchical["Accuracy"],

        "Accuracy Difference":
            hierarchical["Accuracy"]
            - flat["Accuracy"],

        "Flat Balanced Accuracy":
            flat["Balanced Accuracy"],

        "Hierarchical Balanced Accuracy":
            hierarchical["Balanced Accuracy"],

        "Balanced Accuracy Difference":
            hierarchical["Balanced Accuracy"]
            - flat["Balanced Accuracy"],

        "Flat MCC":
            flat["MCC"],

        "Hierarchical MCC":
            hierarchical["MCC"],

        "MCC Difference":
            hierarchical["MCC"]
            - flat["MCC"],

        "Flat CV Macro-F1":
            flat["CV Macro-F1 Mean"],

        "Hierarchical CV Macro-F1":
            hierarchical["CV Macro-F1 Mean"],

        "CV Macro-F1 Difference":
            hierarchical["CV Macro-F1 Mean"]
            - flat["CV Macro-F1 Mean"],

        "Flat OOF Macro-F1":
            flat["Macro-F1"],

        "Hierarchical OOF Macro-F1":
            hierarchical["Macro-F1"],

        "OOF Macro-F1 Difference":
            hierarchical["Macro-F1"]
            - flat["Macro-F1"],

        "Flat Poor Precision":
            flat["Poor Precision"],

        "Hierarchical Poor Precision":
            hierarchical["Poor Precision"],

        "Poor Precision Difference":
            hierarchical["Poor Precision"]
            - flat["Poor Precision"],

        "Flat Poor Recall":
            flat["Poor Recall"],

        "Hierarchical Poor Recall":
            hierarchical["Poor Recall"],

        "Poor Recall Difference":
            hierarchical["Poor Recall"]
            - flat["Poor Recall"],

        "Flat Poor F1":
            flat["Poor F1"],

        "Hierarchical Poor F1":
            hierarchical["Poor F1"],

        "Poor F1 Difference":
            hierarchical["Poor F1"]
            - flat["Poor F1"],

        "Stage1 Poor Routing Recall":
            hierarchical[
                "Mean Stage1 Poor Routing Recall"
            ]
    })


flat_vs_hierarchical = pd.DataFrame(
    comparison_rows
)

print("Complete comparison")
display(
    flat_vs_hierarchical.round(4)
)


# ------------------------------------------------------------
# Compact dissertation table
# ------------------------------------------------------------

hierarchical_compact = flat_vs_hierarchical[
    [
        "Network",
        "Flat CV Macro-F1",
        "Hierarchical CV Macro-F1",
        "CV Macro-F1 Difference",
        "Flat Poor Precision",
        "Hierarchical Poor Precision",
        "Flat Poor Recall",
        "Hierarchical Poor Recall",
        "Flat Poor F1",
        "Hierarchical Poor F1",
        "Poor F1 Difference",
        "Stage1 Poor Routing Recall"
    ]
]

print("\nCompact comparison")
display(
    hierarchical_compact.round(4)
)

In [ ]:
# ============================================================
# Phase 6 - Cell 8.
# Dissertation-Ready Flat vs Hierarchical Table
# ============================================================

table_rows = []

for network, hierarchical in [
    ("LTE", lte_hierarchical_overall),
    ("5G NR", nr_hierarchical_overall)
]:

    flat = (
        imbalance_results[
            (imbalance_results["Network"] == network)
            & (imbalance_results["Method"] == "Class Weight")
            & (imbalance_results["Model"] == "LGBM")
        ]
        .iloc[0]
    )

    # Flat classifier
    table_rows.append({
        "Network":
            network,

        "Approach":
            "Flat",

        "Accuracy":
            flat["Accuracy"],

        "Balanced Accuracy":
            flat["Balanced Accuracy"],

        "CV Macro-F1":
            flat["CV Macro-F1 Mean"],

        "CV Macro-F1 SD":
            flat["CV Macro-F1 SD"],

        "MCC":
            flat["MCC"],

        "Poor Precision":
            flat["Poor Precision"],

        "Poor Recall":
            flat["Poor Recall"],

        "Poor F1":
            flat["Poor F1"],

        "Poor Routing Recall":
            np.nan
    })

    # Hierarchical classifier
    table_rows.append({
        "Network":
            network,

        "Approach":
            "Hierarchical",

        "Accuracy":
            hierarchical["Accuracy"],

        "Balanced Accuracy":
            hierarchical["Balanced Accuracy"],

        "CV Macro-F1":
            hierarchical["CV Macro-F1 Mean"],

        "CV Macro-F1 SD":
            hierarchical["CV Macro-F1 SD"],

        "MCC":
            hierarchical["MCC"],

        "Poor Precision":
            hierarchical["Poor Precision"],

        "Poor Recall":
            hierarchical["Poor Recall"],

        "Poor F1":
            hierarchical["Poor F1"],

        "Poor Routing Recall":
            hierarchical[
                "Mean Stage1 Poor Routing Recall"
            ]
    })


hierarchical_dissertation_table = pd.DataFrame(
    table_rows
)

# Combine CV mean and SD into one concise column
hierarchical_dissertation_table[
    "CV Macro-F1, mean (SD)"
] = hierarchical_dissertation_table.apply(
    lambda row: (
        f"{row['CV Macro-F1']:.4f} "
        f"({row['CV Macro-F1 SD']:.4f})"
    ),
    axis=1
)

# Final dissertation layout
hierarchical_dissertation_table = (
    hierarchical_dissertation_table[
        [
            "Network",
            "Approach",
            "Accuracy",
            "Balanced Accuracy",
            "CV Macro-F1, mean (SD)",
            "MCC",
            "Poor Precision",
            "Poor Recall",
            "Poor F1",
            "Poor Routing Recall"
        ]
    ]
)

# Format numerical columns
formatted_table = (
    hierarchical_dissertation_table.copy()
)

numeric_cols = [
    "Accuracy",
    "Balanced Accuracy",
    "MCC",
    "Poor Precision",
    "Poor Recall",
    "Poor F1",
    "Poor Routing Recall"
]

formatted_table[numeric_cols] = (
    formatted_table[numeric_cols]
    .round(4)
)

formatted_table["Poor Routing Recall"] = (
    formatted_table["Poor Routing Recall"]
    .apply(
        lambda value:
        "—" if pd.isna(value)
        else f"{value:.4f}"
    )
)

print(
    "Table: Comparison of Flat and Hierarchical "
    "Three-Class Classification"
)

display(
    formatted_table
)